In [ ]:

#SAVE WRITE
print("\n Ecriture fichier")
vtk_dir = f"{testCase}{mesh_mode}{mode_ale}p{mode_p_refinement}dx{dx}h{h}Renorm{RenormON}Visco{ViscoONOFF}"
if os.path.exists(vtk_dir):
    shutil.rmtree(vtk_dir)  # Supprime le dossier et tout son contenu
os.makedirs(vtk_dir, exist_ok=True)
# --------------------------
#ModeDEBUG = True #Only for unitaire test (RENORM, MLS etc)
#testCase = 'DamBreak' #jet or car or funnel or DamBreak or TGV (selection) or TGVmesh or bodyF
#mesh_mode = 'lagrangian' #eulerian ale lagrangian
# Paramètres ALE
#mode_ale = 'wass' #wass (Wasserstein norm) or grad (PU based and centroid) or classic (shepard based)
#Modes :
#      'lloyd'   : descente F1 seul (terme dominant Lloyd)
#      'wass'    : descente F1 complet (avec correction ordre 2)  
#      'shepard' : descente F_S seul (correction Shepard)
#      'coupled' : descente F2 = F1 + alpha*F_S  ← NOUVEAU
#beta_geom=0.001 #scalar for ALE velocity
#alpha = 0.01 # Variation in 0.1 or 1 or 10
#mode_p_refinement = 1 # 0 default ordre 0 or Use MLS gradient reconstruction ordre 1
#mode_RK= 2 # Use Euler  (1) or RK2 (2)  temporal integration
# ------------------------


if mode_p_refinement != 0 and RenormON:
    raise ValueError("❌ Erreur: MLS  incompatible avec Renormalisation (p=0)")

if mode_p_refinement != 0:
    RenormON = False  # Force désactivation
# --------------------------
# CONFIGURATION UTILISATEUR
# --------------------------
#Paramètre de la simulation
#ViscoONOFF = False #Add visco physic based Morris 
#ShiftPart = False  # Add shift particule 
#randompos = True # add random pos pos to avoid artificial artefact numerical
randomlength=0.01
#ShepardCorr = False #add a sheaprd correction for gradient SPH
#RenormON = False #Matrice Renorm Consistance 1 Vila only if p=0 (incompatibilty with MLS)
#Filtre = False #Filtrage density to avoid frees surface expansion
stepfiltre = 4 #every step for filtrage
save= 50 #nombres d'itération avant une sauvegarde

# --------------------------
# Paramètres Physiques
# --------------------------
rho0 = 1000.0
mu = 1.0  # Viscosité dynamique (Pa.s)
g_acc = 9.81
H_column = 0.04  #0.0821 m à terme
#c0 = 10.0 * np.sqrt(g_acc * H_column)
c0 = C0 * np.sqrt(g_acc * 0.0621)
gamma = 7.0
B = (rho0 * c0**2) / gamma
g = np.array([0.0, -g_acc])
n_layers = 5  # Nombre de couches pour l'épaisseur du mur (min 3 pour SPH)
#dx = 0.001  #default value
#h = 1.2 * dx  # default value
CFL = 0.1     # 0.01 est très lent, 0.1 est plus standard avec Riemann
FLUID, WALL = 0, 1
R_top = 0.8
R_bottom = 0.15

global L

if testCase == 'AirFoil'  :
    # ---------------------------    
    L = 1.0
    rho0 = 1.0
    U0 = 1.0
    c0 = C0 * U0
    nu = 0.01
    mu = rho0 * nu
    gamma = 7.0
    B = (rho0 * c0**2) / gamma
    dx =    L *0.1 / dx
    h=h*dx
    #dx = 0.04
    #h = 2.0 * dx
    #CFL = 0.01
    finaltime = 8
    # Force motrice (Fe) équivalente pour atteindre U_max = 1.0 au centre :
    # U_max = (Fe * L^2) / (8 * mu)  =>  Fe = 8 * mu * U_max / L^2
    Fe_x = 8.0 * mu * U0 / (L**2)
    g = np.array([Fe_x, 0.0])  # Force motrice agissant comme une gravité selon X
    Periodic = True
    t_star =0.01
    
    t_relaxmax=0.05
    #tree = cKDTree(pos, boxsize=L)
    
    
if testCase == 'Poiseuille':
    # --------------------------------------------------------------------------
    # ÉCOULEMENT DE POISEUILLE TRANSITOIRE (1600 particules fluides)
    # https://remi-carmigniani.quarto.pub/enpc_sphyd_td3_2023/#/title-slide
    # --------------------------------------------------------------------------
    L = 1.0           # Largeur et hauteur du canal fluide [m]
    rho0 = 1.0        # Densité de référence [kg/m³]
    
    # Nombre de Reynolds Re = 10 basé sur la vitesse max théorique U_max = 1.0
    U0 = 1.0          # Vitesse max visée au centre [m/s]
    nu = 0.1          # Viscosité cinématique pour Re = 10 (Re = U0 * L / nu)
    mu = rho0 * nu
    
    # Force motrice (Fe) équivalente pour atteindre U_max = 1.0 au centre :
    # U_max = (Fe * L^2) / (8 * mu)  =>  Fe = 8 * mu * U_max / L^2
    Fe_x = 8.0 * mu * U0 / (L**2)
    g = np.array([Fe_x, 0.0])  # Force motrice agissant comme une gravité selon X
    
    # Paramètres SPH & Numériques
    c0 = C0 * U0      # Vitesse du son artificielle
    gamma = 7.0
    B = (rho0 * c0**2) / gamma
    
    # Résolution : 40 x 40 = 1600 particules fluides
    # On ajoute des couches de particules de paroi en haut et en bas
    n_fluid_per_side = 40
    dx = L / (dx*n_fluid_per_side)
    h = h * dx        # Votre lissage h basé sur le dx
    t_relaxmax=0.05
    n_layers_wall = 3 # Nombre de couches pour éviter la troncature du noyau aux parois
    
    finaltime = 15.0  # Temps nécessaire pour atteindre le régime stationnaire parabolique
    Periodic = True  # /!\ Attention : Périodique en X uniquement. 
                      # Si votre code global gère boxsize=(L, None), ajustez l'arbre cKDTree en conséquence.

if testCase == 'bodyF':
    # ==============================================================================
    # 1.  PARAMÈTRES DU CAS TEST
    # ==============================================================================

    # ---- Choix de la résolution ----
    # dx = 0.005   # Haute résolution  (~600k particules, ~24h de calcul)
    # dx = 0.01    # Résolution standard (~150k particules, ~4h de calcul)
    #dx = 0.02      # Résolution pédagogique (~12k particules, ~15min)   ← défaut
    
    # ---- Types de particules ----
    FLUID, WALL, FLAP, BODY = 0, 1, 2, 3

    # ---- Physique de base ----
    rho0    = 1000.0            # densité eau  [kg/m³]
    g_acc   = 9.81              # pesanteur    [m/s²]
    g       = np.array([0.0, -g_acc])

    # ---- Géométrie du bassin ----
    H_water = 0.40              # profondeur eau au repos [m]
    L_tank  = 8.00             # longueur du bassin      [m]
    
    x_ramp_start = 4.0   # Début de la plage inclinée [m]
    H_ramp_end   = 0.6   # Hauteur maximale de la rampe à l'extrémité droite [m]
    dx =    H_water *0.1 / dx
    # ---- Corps flottant (section 2D) ----
    L_body  = 0.10              # longueur        [m]
    H_body  = 0.05              # hauteur         [m]
    width_body = 0.29       # Largeur (pour conversion 3D->2D) [m]
    x_body_init  = 2.11  # Position du centre du corps par rapport au clapet [m]
    # Immersion (tirant d'eau) = 0.03 m d'après l'image.
    # Le bas du corps est donc à : H_water - 0.03
    # Le centre vertical (y_cm) est à : (H_water - 0.03) + H_body / 2
    y_body_init  = (H_water - 0.03) + (H_body / 2.0)
    rho_rel = 0.68              # densité relative (ρ_corps / ρ_eau)
    rho_body = rho_rel * rho0   # densité absolue [kg/m³] = 680
    mass_body_3d = 0.986    # Masse totale [kg]
    I_body_3d = 0.0014      # Inertie rotationnelle [kg.m2]
    #  En 2D, on travaille par unité de largeur (largeur réelle = 0.29 m)
    # --- Conversion pour le solveur 2D ---
    mass_body_2D = mass_body_3d / width_body   # ~3.4 kg/m
    I_2D = I_body_3d / width_body               # ~0.00483 kg.m/rad
    t_relaxmax=0.05
    
    L=1.
    # ---- Sondes de mesure ----
    x_probe1 = 1.155     # sonde 1 : avant le corps  [m]
    x_probe2 = 2.70     # sonde 2 : après le corps  [m]

    # ---- Paramètres SPH ----
    
    h=h*dx
    c0   = C0 * np.sqrt(g_acc * H_water)   # ≈ 29.7 m/s
    gamma = 7.0
    B    = rho0 * c0**2 / gamma
    CFL  = 0.05
    mu   = 1.0e-3                              # viscosité dynamique eau [Pa.s]
    ViscoONOFF  = False
    ShepardCorr = False
    n_layers    = 3
    mode_RK     = 1                            # intégration RK2
    finaltime   = 50.0                         # durée totale [s]
    save        = 100                          # fréquence VTK
    Periodic    = False

    print(f"\nSPH : dx={dx:.3f}m  h={h:.4f}m  c0={c0:.2f}m/s")
    print(f"Nombre approx. de particules fluides : {int(L_tank*H_water/dx**2)}")

    
    

if testCase == 'car':
    # longueurs horizontales
    L_plateau = 4.0       # plateau gauche
    L=L_plateau
    L_slope = 3.2         # pente descendante
    L_bottom = 7.6        # plateau fond
    L_rise = 3.2          # remontée
    L_plateau_right = 4.0 # plateau final
    finaltime = 20
    # angles
    theta_deg = 18.02       # Angle pente
    theta = np.radians(theta_deg)
    H_water =0.3    
    
    # Paramètres du quai
    L_quay = 1.0  # Longueur du quai à gauche
    H_quay = H_water  # Le quai est légèrement au-dessus de l'eau
    posXinitCar =1.2
    posYinitCar = H_quay+1.1
#     c0 = C0 * np.sqrt(g_acc * H_water)
    gamma = 7.0
    B = (rho0 * c0**2) / gamma
    g = np.array([0.0, -g_acc])
    dx =    H_water  / (2.0*dx)
    h=h*dx
    t_relaxmax=0.5    
    #dx = 0.1
    #h = 1.2 * dx
    #CFL = 0.05
    # Types de particules
    FLUID, WALL, CAR = 0, 1, 2
    car_mass = 4400
    carvitesse = 1
    Periodic = False #only for TGV false for else testcase
    R_car = dx   # rayon voiture
    R_wall =dx # rayon mur
    # 4. Paramètres physiques WALL-CAR
    dtcrtq = CFL * h / (c0 +  carvitesse)  #$dt$ doit être beaucoup plus petit que cette période (typiquement $dt < T/10$
    #9k = 0.0001*car_mass/( dtcrtq )**2   # Raideur du ressort
    #damping = 2.0 * np.sqrt(car_mass * k) * 0.7 #2*(car_mass*k)**0.5 #50.0 # Amortissement
    H_eff = H_water + np.tan(theta) * L_slope  # = 1.04 m
    c0 = C0 * np.sqrt(g_acc * H_eff)
    alpha = 1.5 # Facteur d'amortissement critique pour fluide faiblement compressible
    relax_coefficient = alpha * (c0 / H_water )
    
    print("\n Initialisation0K")

if testCase == 'TGV' or testCase == 'TGVmesh' :
    # ---------------------------
    TYPE_FLUID = 1
    TYPE_GHOST = 2
    L = 1.0
    rho0 = 1.0
    U0 = 1.0
    c0 = C0 * U0
    nu = 0.01
    mu = rho0 * nu
    gamma = 7.0
    B = (rho0 * c0**2) / gamma
    dx =    L *0.1 / dx
    h=h*dx
    #dx = 0.04
    #h = 2.0 * dx
    #CFL = 0.01
    finaltime = 8
    g = np.array([0.0, 0.0])
    Periodic = True
    t_star =0.01
    if testCase =='TGVmesh': mode_RK = 1
    t_relaxmax=0.05
    #tree = cKDTree(pos, boxsize=L)
    
if testCase == 'funnel':
    t_relaxmax=0.1
    rho0 = 1285.0
    H_column =  0.04 # 0.04 #0.0821 m à terme  //0.04 en mode degradé
    # --- Paramètres de l'entonnoir ---
    H_funnel = H_column
    space = 1
    R_top = 0.8
    R_bottom = 0.15
    mu = 1.0  # Viscosité dynamique (Pa.s)
    #L_column = 1  # 1 m à terme
    #c0 = 10.0 * np.sqrt(g_acc * H_column)
    c0 = C0 * np.sqrt(g_acc * H_column) #1 bon pour dx =0.001
    gamma = 7.0
    B = (rho0 * c0**2) / gamma
    dx =    R_top *0.1 / dx
    h=h*dx
    L=H_funnel 
    noise_amp = 0.05 * dx  #scalar for random pos [0.001; 0.15] 
    finaltime = 3
    Periodic = False #only for TGV false for else testcase
    Initcase = True
    alpha = 1.5 # Facteur d'amortissement critique pour fluide faiblement compressible
    relax_coefficient = alpha * (c0 / H_column )

if testCase == 'DamBreak':
    L_column = 0.1  # 1 m à terme
    L=L_column  
    H_column = 2.0*L_column  #2 m à terme
    c0 = C0 * np.sqrt(g_acc * H_column)
    gamma = 7.0
    mu = 0.000001  # Viscosité dynamique (Pa.s)
    B = (rho0 * c0**2) / gamma
    dx =    L_column *0.1 / dx
    h=h*dx    
    #dx = 0.004  # 0.04 Dambreak de base 
    #h = 2.0 * dx  # Un h trop grand (2*dx) augmente trop la pression locale
    noise_amp = 0.05 * dx  #scalar for random pos [0.001; 0.15] 
    finaltime = 1.0
    Periodic = False #only for TGV false for else testcase
    t_relaxmax=0.05
        
    
if testCase == 'jet' or testCase == 'Mill':
    print("\n Initialisation0")
    #----------jet test case 
    tailleJET = 0.2
    t_relaxmax=0.05
    dx =    tailleJET *0.1 / dx
    h=h*dx  
    noise_amp = 0.05 * dx  #scalar for random pos [0.001; 0.15] 
    g = np.array([0.0, 0.0])
    v_jet = 1.00
    c0 = v_jet * C0
    gamma = 7.0
    L=tailleJET
    B = (rho0 * c0**2) / gamma
    DOUBLE_JET = True
    TRIPLE_JET = True
    # Pré-remplissage fluide
    N_layers = 25            # nombre de couches de fluide après chaque buffer
    impact_gap = 2 * dx       # espace vide au centre pour retarder l'impact
    # Types
    FLUID, WALL, GHOST, BUFFER_TOP, BUFFER_BOTTOM, WHEEL,BUFFER_MIDDLE = 0, 1, 2, 3, 4, 5, 6
    # Géométrie
    jet_x_min, jet_x_max = 0.32, 0.38
    decalbx = 0.35
    decalbx1 = 0.80
    y_center, gap = 0.50, 0.3
    theta_deg1 = 40.0
    y_inlet_top = y_center + gap
    y_inlet_bottom = -0.1
    y_inlet_middle = 1.1
    wheel_centerX = 0.80
    wheel_centerY = 0.40
    finaltime = 8
    Periodic = False #only for TGV false for else testcase
    # 1. Paramètres géométriques de la roue
        # Le centre (wheel_centerX, wheel_centerY) est configuré à (0.60, 0.60)
    r_hub = 0.08                     # Rayon du moyeu central
    blade_length = 0.22              # Longueur d'une pale depuis le centre
    blade_thickness = 2 * dx         # Épaisseur d'une pale
    n_layers_wheel = 3               # Nombre de couches pour assurer le support SPH
    n_blades = 3                     # Roue à 4 pales

global search_radius
search_radius =  2*h if Kernel else 2*h
print(f"SearchRadisus: {search_radius} ")

# Note : Pour un Reynolds très faible (fluide très visqueux), 
# testez avec mu = 0.1 ou 1.0.
# --------------------------
# Fonctions SPH & VTK
# --------------------------
def update_neighborhood(new_pos, h):
    # On reconstruit l'arbre avec les positions actuelles
    new_tree = cKDTree(new_pos)
    # On cherche les paires de particules à une distance d'influence (souvent 2h ou 3h)
    new_pairs = new_tree.query_pairs(r=2.0 * h) # Adapte le rayon selon ton noyau
    return new_tree, new_pairs



def kernel_cubic(r, h):
    # Sécurité : Convertit r en tableau NumPy s'il s'agit d'un simple float/int
    r_arr = np.atleast_1d(r)
    
    # On initialise le tableau de sortie
    w = np.zeros_like(r_arr)
    
    if Kernel:  # Wendland C2 étendu à un support de 2*h
        q = r_arr / (2.0 * h)
        mask = q <= 1.0
        sigma = 7.0 / (4.0 * np.pi * h**2)
        
        w[mask] = sigma * ((1.0 - q[mask])**4) * (1.0 + 4.0 * q[mask])
        
    else:  # Spline Cubique (Support de taille 2*h)
        q = r_arr / h
        sigma = 10.0 / (7.0 * np.pi * h**2)
        
        condles = [
            q <= 1.0,
            (q > 1.0) & (q <= 2.0)
        ]
        
        fonctions = [
            lambda q: sigma * (1.0 - 1.5 * q**2 + 0.75 * q**3),
            lambda q: sigma * 0.25 * ((2.0 - q)**3)
        ]
        
        w = np.select(condles, [f(q) for f in fonctions], default=0.0)
    
    # Si l'entrée 'r' originale était un scalaire (comme 0), on renvoie un scalaire
    # Sinon, on renvoie le tableau complet.
    return w[0] if np.isscalar(r) else w

pause_duration = 0.01
pause_start = None

def carvelocityInlet(pos, t):
    global pause_start

    # Si pos est un scalaire (juste la coordonnée x), on l'utilise directement.
    # S'il s'agit d'un vecteur, on extrait l'index 0.
    if hasattr(pos, "__len__") or isinstance(pos, np.ndarray):
        x = pos[0]
    else:
        x = pos

    # Déclenchement de la pause au milieu
    if pause_start is None and x >= 10.0:
        pause_start = t

    # Pendant la pause
    if pause_start is not None and t < pause_start + pause_duration:
        vx = 0.0
    else:
        vx = 1.0

    return np.array([vx, 0.0])
    
def compute_concentration_gradient(pos, vol, h, tree, pairs, Periodic=False, 
                                  testCase=None, L=None):
    """Calcule ∇C (Sun et al. 2017)."""
    n_particles = len(pos)
    dim = pos.shape[1]
    
    # 1. Concentration C_i = ∑_j W_{ij} V_j
    C = np.zeros(n_particles)
    
    for i, j in pairs:
        if Periodic and testCase in ['Poiseuille', 'TGV', 'TGVmesh', 'AirFoil']:
            rij = periodic_rij(pos[i], pos[j], testCase, L)
        else:
            rij = pos[i] - pos[j]
        
        r = np.linalg.norm(rij)
        if r < 1e-14 or r > 2.0 * h:
            continue
        
        W = kernel_cubic(r, h)
        C[i] += W * vol[j]
        C[j] += W * vol[i]
    
    # 2. Gradient ∇C_i
    grad_C = np.zeros((n_particles, dim))
    
    for i, j in pairs:
        if Periodic and testCase in ['Poiseuille', 'TGV', 'TGVmesh','AirFoil']:
            rij = periodic_rij(pos[i], pos[j], testCase, L)
        else:
            rij = pos[i] - pos[j]
        
        r = np.linalg.norm(rij)
        if r < 1e-14 or r > 2.0 * h:
            continue
        
        # ∇W
        if r > 1e-14:
            q = r / h
            if q < 1.0:
                dW_dr = (0.75 * (-3.0 * q)) / (np.pi * h**3)
            elif q < 2.0:
                dW_dr = (-0.75 * (2.0 - q)**2) / (np.pi * h**3)
            else:
                dW_dr = 0.0
            grad_W = dW_dr * (rij / r)
        else:
            grad_W = np.zeros(dim)
        
        delta_C = C[j] - C[i]
        grad_C[i] += delta_C * grad_W * vol[j]
        grad_C[j] -= delta_C * grad_W * vol[i]
    
    return grad_C, C
 
 
def compute_shifting_displacement_delta_sph(pos, vol, grad_C, C, h, 
                                           CFL=0.05, R=0.2, 
                                           types=None, FLUID=0):
    """Calcule déplacement δ-SPH (Sun et al. 2017)."""
    
    n_particles = len(pos)
    dim = pos.shape[1]
    shift = np.zeros((n_particles, dim))
    shift_status = np.zeros(n_particles, dtype=int)
    
    if types is not None:
        mask_fluid = (types == FLUID)
    else:
        mask_fluid = np.ones(n_particles, dtype=bool)
    
    eps = 1e-14
    mask_valid = (np.abs(C) > eps) & mask_fluid
    
    if not np.any(mask_valid):
        return shift, shift_status
    
    # δx = −CFL·h·R·∇C / C
    shift[mask_valid] = -CFL * h * R * (grad_C[mask_valid] / C[mask_valid][:, None])
    
    # Sécurité: borner à 0.1*h
    shift_mag = np.linalg.norm(shift[mask_valid], axis=1)
    mask_large = shift_mag > 0.1 * h
    
    if np.any(mask_large):
        large_idx = np.where(mask_valid)[0][mask_large]
        shift[large_idx] *= (0.1 * h / shift_mag[mask_large][:, None])
    
    shift_status[mask_valid] = 1
    
    return shift, shift_status
 
 
def compute_shifting_displacement_lloyd(pos, vol, h, tree, pairs,
                                       beta_lloyd=0.5, 
                                       types=None, FLUID=0):
    """Calcule déplacement Lloyd (Vacondio et al. 2012)."""
    
    n_particles = len(pos)
    dim = pos.shape[1]
    
    centroid_num = np.zeros((n_particles, dim))
    centroid_den = np.zeros(n_particles)
    
    for i, j in pairs:
        rij = pos[i] - pos[j]
        r = np.linalg.norm(rij)
        
        if r < 1e-14 or r > 2.0 * h:
            continue
        
        W = kernel_cubic(r, h)
        weight = W * vol[j]
        
        centroid_num[i] += pos[j] * weight
        centroid_den[i] += weight
        
        weight_rev = W * vol[i]
        centroid_num[j] += pos[i] * weight_rev
        centroid_den[j] += weight_rev
    
    if types is not None:
        mask_fluid = (types == FLUID)
    else:
        mask_fluid = np.ones(n_particles, dtype=bool)
    
    mask_valid = (centroid_den > 1e-14) & mask_fluid
    
    shift = np.zeros((n_particles, dim))
    shift_status = np.zeros(n_particles, dtype=int)
    
    if np.any(mask_valid):
        centroid = centroid_num[mask_valid] / centroid_den[mask_valid][:, None]
        shift[mask_valid] = beta_lloyd * (centroid - pos[mask_valid])
        shift_status[mask_valid] = 1
    
    return shift, shift_status
 
 
def apply_shifting_correction(pos, shift, boundary_layer_width=None, h=None,
                             types=None, WALL=1):
    """Applique shifting avec amortissement."""
    
    if boundary_layer_width is None:
        boundary_layer_width = 2.0 * h if h is not None else 1.0
    
    pos_new = pos + shift
    
    if types is not None and h is not None:
        wall_mask = (types == WALL)
        
        if np.any(wall_mask):
            wall_positions = pos[wall_mask]
            
            if len(wall_positions) > 0:
                tree_wall = cKDTree(wall_positions)
                fluid_mask = ~wall_mask
                
                if np.any(fluid_mask):
                    distances, _ = tree_wall.query(pos[fluid_mask])
                    damping = np.exp(-distances / boundary_layer_width)
                    pos_new[fluid_mask] = pos[fluid_mask] + shift[fluid_mask] * damping[:, None]
    
    return pos_new
 
 

def diagnose_shifting_statistics(shift, C, grad_C, h, step=None):
    """Diagnostiques du shifting."""
    
    shift_mag = np.linalg.norm(shift, axis=1)
    grad_C_mag = np.linalg.norm(grad_C, axis=1)
    
    stats = {
        'shift_mag_mean': np.mean(shift_mag),
        'shift_mag_max': np.max(shift_mag),
        'shift_ratio_h': np.mean(shift_mag) / h,
        'C_mean': np.mean(C),
        'C_max': np.max(C),
        'C_min': np.min(C),
        'grad_C_mean': np.mean(grad_C_mag),
        'grad_C_max': np.max(grad_C_mag),
    }
    
    if ModeDEBUG:
        header = f"Shifting ({step})" if step else "Shifting"
        print(f"\n{header}: shift_mean={stats['shift_ratio_h']:.3f}h, "
              f"C_range=[{stats['C_min']:.3f}, {stats['C_max']:.3f}], "
              f"∇C_max={stats['grad_C_max']:.2e}")
    
    return stats
 


# ============================================================================
# MÉTRIQUE 1: COEFFICIENT D'ANISOTROPIE VORONOÏ
# ============================================================================

def compute_voronoi_anisotropy(pos, h, testCase=None, L=None, Periodic=False):
    """
    VORONOI ANISOTROPY: Mesure l'écart à une distribution isotrope.
    
    THÉORIE (Springel 2010, Owen et al. 1998):
      Pour chaque particule i, calculer sa cellule de Voronoï.
      Mesurer le ratio : r_inscrit / r_circonscrit
      
      • Distribution hexagonale parfaite (isotrope) → ratio ~ 0.6
      • Distribution aléatoire → ratio ~ 0.4-0.5
      • Distribution dégénérée (clusters) → ratio < 0.2
    
    Sortie: (mean_ratio, std_ratio, acceptable)
      • acceptable = True si 0.4 < mean_ratio < 0.65 (bon)
      • acceptable = False si clusters (ratio < 0.2) ou aliasing (ratio > 0.7)
    """
    
    # Cas périodique: créer copies sur les bords
    pos_extended = pos.copy()
    
    if Periodic and testCase in ['TGV', 'Poiseuille','AirFoil']:
        # Ajouter copies sur les bords pour Voronoï périodique
        pos_extended = np.vstack([
            pos + np.array([L, 0]),   # Droite
            pos + np.array([-L, 0]),  # Gauche
            pos + np.array([0, L]),   # Haut
            pos + np.array([0, -L]),  # Bas
            pos + np.array([L, L]),   # Coin
            pos + np.array([L, -L]),
            pos + np.array([-L, L]),
            pos + np.array([-L, -L]),
        ])
    
    # Calcul Voronoï
    try:
        vor = Voronoi(pos_extended)
    except Exception as e:
        print(f"⚠️ Voronoi failed: {e}")
        return np.nan, np.nan, False
    
    # Pour chaque particule dans le domaine original, calculer le ratio
    ratios = []
    
    for i in range(len(pos)):  # Seulement les particules originales
        region_idx = vor.point_region[i]
        region = vor.regions[region_idx]
        
        if len(region) < 3:  # Cellule dégénérée
            ratios.append(0.0)
            continue
        
        # Vertices de la cellule Voronoï
        vertices = vor.vertices[region]
        
        # Centre: centroïde de la cellule
        center = np.mean(vertices, axis=0)
        
        # Rayon inscrit (distance min au bord)
        r_inscrit = np.min(np.linalg.norm(vertices - center, axis=1))
        
        # Rayon circonscrit (distance max au bord)
        r_circonscrit = np.max(np.linalg.norm(vertices - center, axis=1))
        
        if r_circonscrit > 1e-14:
            ratio = r_inscrit / r_circonscrit
        else:
            ratio = 0.0
        
        ratios.append(ratio)
    
    ratios = np.array(ratios)
    
    mean_ratio = np.mean(ratios)
    std_ratio = np.std(ratios)
    
    # Critère d'acceptabilité
    acceptable = (0.35 < mean_ratio < 0.70) and (std_ratio < 0.25)
    
    return mean_ratio, std_ratio, acceptable


# ============================================================================
# MÉTRIQUE 2: DÉVIATION STANDARD DU NOMBRE DE VOISINS
# ============================================================================

def compute_neighbor_deficit(pos, vol, h, types=None, FLUID=0):
    """
    NEIGHBOR DEFICIT: Mesure l'uniformité de la distribution locale.
    
    THÉORIE (Monaghan 1992, Íñiguez et al. 2017):
      Pour un kernel de support 2h, le nombre idéal de voisins est:
      
      n_ideal = ρ * V_kernel = ρ * (4π/3 * h²)  en 2D: n_ideal ~ 60-100
      
      La déviation standard σ = sqrt(1/N * sum((n_i - n_ideal)²))
      
      • σ petit (< 5): Distribution uniforme (bon)
      • σ moyen (5-15): Acceptable
      • σ grand (> 20): Clustering ou raréfaction (mauvais)
    
    Sortie: (mean_neighbors, sigma, acceptable)
    """
    
    n_particles = len(pos)
    
    # Créer KDTree
    tree = cKDTree(pos)
    
    # Nombre de voisins dans 2h pour chaque particule
    neighbor_counts = []
    
    for i in range(n_particles):
        if types is not None and types[i] != FLUID:
            continue  # Sauter les murs
        
        neighbors = tree.query_ball_point(pos[i], 2.0 * h)
        neighbor_counts.append(len(neighbors))
    
    neighbor_counts = np.array(neighbor_counts)
    
    # Nombre idéal de voisins
    # En 2D, kernel cubic: V_kernel = 4πh²/3, donc pour densité ~1:
    # n_ideal ~ 4πh² * (1/h²) ~ 12.5 * h² / h² = 12.5 (en densité unitaire)
    # En pratique, on observe 40-80 voisins pour une distribution compacte
    n_ideal = np.mean(neighbor_counts)  # Prendre la moyenne observée comme référence
    
    mean_neighbors = np.mean(neighbor_counts)
    sigma = np.std(neighbor_counts)
    
    # Critère d'acceptabilité
    acceptable = (sigma < 0.30 * n_ideal)  # σ < 30% de la moyenne
    
    return mean_neighbors, sigma, acceptable


# ============================================================================
# MÉTRIQUE 3: DRIFT METRIC (Écart à la position grillée)
# ============================================================================

def compute_drift_metric(pos_current, pos_initial, vel_convection, dt_total, dx):
    """
    DRIFT METRIC: Mesure la dérive non-physique des particules.
    
    THÉORIE (Lind & Stansby 2016):
      Chaque particule devrait rester à une position cohérente avec:
      
      pos_expected = pos_initial + ∫ vel_convection dt
      
      L'écart de dérive (drift):
      
      ε_i = || pos_current[i] - pos_expected[i] ||_2
      
      Normalisé par la distance inter-particulaire:
      
      ε_normalized = < ε_i > / dx
      
      • ε_normalized ~ 0.01-0.05: Excellent (pas de drift)
      • ε_normalized ~ 0.1-0.2: Acceptable (drift < 20% dx)
      • ε_normalized > 0.5: Mauvais (drift > 50% dx, problématique)
    
    Sortie: (mean_drift, max_drift, acceptable)
    """
    
    n_particles = len(pos_current)
    
    # Position attendue: initiale + déplacement de convection
    # (Si vous avez l'historique de vel, calculer l'intégrale)
    # Ici, placeholder: supposer que la convection est identité (vérifier dans votre cas)
    pos_expected = pos_initial + vel_convection * dt_total
    
    # Écart de dérive pour chaque particule
    drift = np.linalg.norm(pos_current - pos_expected, axis=1)
    
    mean_drift = np.mean(drift)
    max_drift = np.max(drift)
    
    # Normaliser par la distance inter-particulaire
    drift_normalized = mean_drift / (dx + 1e-14)
    
    # Critère d'acceptabilité
    acceptable = (drift_normalized < 0.2)  # Drift < 20% de dx
    
    return mean_drift, max_drift, acceptable

def validate_shifting_quality(pos, pos_initial, vel_convection, dt_total,
                              vol, h, types, FLUID, dx,
                              testCase=None, L=None, Periodic=False,
                              step=None):
    """
    VALIDATION COMPLÈTE: Les 3 métriques en une seule fonction.
    
    Retourne: dict avec tous les diagnostiques
    """
    
    print(f"\n{'='*70}")
    if step:
        print(f"VALIDATION SHIFTING (step {step})")
    else:
        print(f"VALIDATION SHIFTING")
    print(f"{'='*70}")
    
    results = {}
    
    # MÉTRIQUE 1: Voronoï
    print("\n1️⃣  VORONOÏ ANISOTROPY (Distribution géométrique)")
    print("-" * 70)
    try:
        voronoi_ratio, voronoi_std, voronoi_ok = compute_voronoi_anisotropy(
            pos, h, testCase, L, Periodic
        )
        
        print(f"   Mean ratio: {voronoi_ratio:.4f} (idéal: ~0.55, range: 0.35-0.70)")
        print(f"   Std dev:    {voronoi_std:.4f} (idéal: < 0.25)")
        
        if voronoi_ok:
            print(f"   ✅ PASSÉ: Distribution isotrope (pas de clusters)")
        else:
            print(f"   ❌ ÉCHOUÉ: Distribution anisotrope ou dégénérée")
        
        results['voronoi_ratio'] = voronoi_ratio
        results['voronoi_std'] = voronoi_std
        results['voronoi_ok'] = voronoi_ok
    
    except Exception as e:
        print(f"   ⚠️  Voronoï calcul échoué: {e}")
        results['voronoi_ok'] = False
    
    # MÉTRIQUE 2: Déficit de voisins
    print("\n2️⃣  NEIGHBOR DEFICIT (Uniformité locale)")
    print("-" * 70)
    try:
        mean_neighbors, sigma_neighbors, neighbors_ok = compute_neighbor_deficit(
            pos, vol, h, types, FLUID
        )
        
        print(f"   Mean neighbors: {mean_neighbors:.1f}")
        print(f"   Std dev (σ):    {sigma_neighbors:.1f}")
        print(f"   σ / mean ratio: {sigma_neighbors/mean_neighbors:.3f} (idéal: < 0.30)")
        
        if neighbors_ok:
            print(f"   ✅ PASSÉ: Nombre de voisins uniforme")
        else:
            print(f"   ❌ ÉCHOUÉ: Clustering ou raréfaction détecté")
        
        results['mean_neighbors'] = mean_neighbors
        results['sigma_neighbors'] = sigma_neighbors
        results['neighbors_ok'] = neighbors_ok
    
    except Exception as e:
        print(f"   ⚠️  Neighbors calcul échoué: {e}")
        results['neighbors_ok'] = False
    
    # MÉTRIQUE 3: Drift
    print("\n3️⃣  DRIFT METRIC (Dérive non-physique)")
    print("-" * 70)
    try:
        mean_drift, max_drift, drift_ok = compute_drift_metric(
            pos, pos_initial, vel_convection, dt_total, dx
        )
        
        print(f"   Mean drift:        {mean_drift:.4f} (unités physiques)")
        print(f"   Max drift:         {max_drift:.4f}")
        print(f"   Drift / dx ratio:  {mean_drift/dx:.4f} (idéal: < 0.20)")
        
        if drift_ok:
            print(f"   ✅ PASSÉ: Pas de dérive non-physique")
        else:
            print(f"   ❌ ÉCHOUÉ: Dérive importante (> 20% dx)")
        
        results['mean_drift'] = mean_drift
        results['max_drift'] = max_drift
        results['drift_ok'] = drift_ok
    
    except Exception as e:
        print(f"   ⚠️  Drift calcul échoué: {e}")
        results['drift_ok'] = False
    
    # RÉSUMÉ FINAL
    print("\n" + "="*70)
    print("RÉSUMÉ FINAL")
    print("="*70)
    
    all_ok = results.get('voronoi_ok', False) and \
             results.get('neighbors_ok', False) and \
             results.get('drift_ok', False)
    
    if all_ok:
        print("\n✅ SHIFTING VALIDÉ: Toutes les 3 métriques sont bonnes")
        print("   → Distribution géométrique excellente")
        print("   → Pas de dérive non-physique")
        print("   → Shifting est FIABLE pour la simulation")
    else:
        print("\n⚠️  SHIFTING À RÉVISER:")
        if not results.get('voronoi_ok', False):
            print("   • Voronoï: Distribution anisotrope → réduire CFL ou R")
        if not results.get('neighbors_ok', False):
            print("   • Neighbors: Clustering → Shifting trop agressif")
        if not results.get('drift_ok', False):
            print("   • Drift: Dérive importante → vérifier vel_convection")
    
    return results

def test_shifting_with_metrics():
    """
    TEST COMPLET: Générer position initiale grillée, appliquer shifting,
    puis valider avec les 3 métriques.
    """
    
    print("\n" + "="*70)
    print("TEST: VALIDATION DU SHIFTING AVEC MÉTRIQUES GÉOMÉTRIQUES")
    print("="*70)
    
    # Configuration test: grille régulière 10x10
    nx, ny = 10, 10
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    xx, yy = np.meshgrid(x, y)
    pos_initial = np.column_stack([xx.ravel(), yy.ravel()])
    
    n_particles = len(pos_initial)
    vol = np.ones(n_particles) * (1.0 / n_particles)
    types = np.zeros(n_particles, dtype=int)
    h = 0.15
    dx = 1.0 / nx
    
    print(f"\nConfiguration initiale:")
    print(f"  • Grille {nx}x{ny} = {n_particles} particules")
    print(f"  • h = {h:.3f}, dx = {dx:.4f}")
    
    # SCENARIO 1: Pas de shifting (baseline)
    print("\n" + "-"*70)
    print("SCENARIO 1: Aucun shifting (baseline)")
    print("-"*70)
    
    vel_convection_baseline = np.zeros((n_particles, 2))  # Pas de mouvement
    dt_total = 0.0
    
    results_baseline = validate_shifting_quality(
        pos=pos_initial,
        pos_initial=pos_initial,
        vel_convection=vel_convection_baseline,
        dt_total=dt_total,
        vol=vol,
        h=h,
        types=types,
        FLUID=0,
        dx=dx,
        testCase='TGV',
        L=1.0,
        Periodic=False,
        step=0
    )
    
    # SCENARIO 2: Déplacement aléatoire (mauvais)
    print("\n" + "-"*70)
    print("SCENARIO 2: Déplacement aléatoire MAUVAIS")
    print("-"*70)
    
    np.random.seed(42)
    shift_bad = np.random.randn(n_particles, 2) * 0.15  # Shift aléatoire grand
    pos_bad = pos_initial + shift_bad
    
    # Clipper au domaine
    pos_bad = np.clip(pos_bad, 0, 1)
    
    vel_convection_bad = shift_bad / 0.1  # Vitesse équivalente
    dt_total_bad = 0.1
    
    results_bad = validate_shifting_quality(
        pos=pos_bad,
        pos_initial=pos_initial,
        vel_convection=vel_convection_bad,
        dt_total=dt_total_bad,
        vol=vol,
        h=h,
        types=types,
        FLUID=0,
        dx=dx,
        testCase='TGV',
        L=1.0,
        Periodic=False,
        step=1
    )
    
    # SCENARIO 3: Déplacement BON (δ-SPH simulé)
    print("\n" + "-"*70)
    print("SCENARIO 3: Déplacement BON (δ-SPH optimal)")
    print("-"*70)
    
    # Simpler un bon shifting: déplacement petit, isotrope
    shift_good = np.random.randn(n_particles, 2) * 0.02  # Shift petit
    pos_good = pos_initial + shift_good
    pos_good = np.clip(pos_good, 0, 1)
    
    vel_convection_good = shift_good / 0.1
    dt_total_good = 0.1
    
    results_good = validate_shifting_quality(
        pos=pos_good,
        pos_initial=pos_initial,
        vel_convection=vel_convection_good,
        dt_total=dt_total_good,
        vol=vol,
        h=h,
        types=types,
        FLUID=0,
        dx=dx,
        testCase='TGV',
        L=1.0,
        Periodic=False,
        step=2
    )
    
    # COMPARAISON
    print("\n" + "="*70)
    print("COMPARAISON RÉSULTATS")
    print("="*70)
    
    print(f"\n{'Métrique':<30} {'Baseline':<20} {'Mauvais':<20} {'Bon':<20}")
    print("-" * 90)
    
    print(f"{'Voronoï ratio':<30} {results_baseline.get('voronoi_ratio', np.nan):<20.4f} "
          f"{results_bad.get('voronoi_ratio', np.nan):<20.4f} "
          f"{results_good.get('voronoi_ratio', np.nan):<20.4f}")
    
    print(f"{'Mean neighbors':<30} {results_baseline.get('mean_neighbors', np.nan):<20.1f} "
          f"{results_bad.get('mean_neighbors', np.nan):<20.1f} "
          f"{results_good.get('mean_neighbors', np.nan):<20.1f}")
    
    print(f"{'Drift / dx':<30} {results_baseline.get('mean_drift', np.nan)/dx:<20.4f} "
          f"{results_bad.get('mean_drift', np.nan)/dx:<20.4f} "
          f"{results_good.get('mean_drift', np.nan)/dx:<20.4f}")
    
    print("\n✅ Le scénario BON passe tous les tests")
    print("❌ Le scénario MAUVAIS échoue les tests")
    
    
def reconstruct_order2(data_i, phi_i, disp):
    """
    Reconstruit la valeur d'un champ au point milieu par Taylor d'ordre 2.
    data_i : array (5,) contenant [grad_x, grad_y, hxx, hxy, hyy]
    phi_i  : valeur scalaire au point i (ex: pres[i])
    disp   : vecteur déplacement (x_target - x_i)
    """
    dx, dy = disp[0], disp[1]
    
    # 1. Terme Gradient : (grad_x * dx + grad_y * dy)
    grad_term = data_i[0] * dx + data_i[1] * dy
    
    # 2. Terme Hessienne : 0.5 * (hxx * dx^2 + 2*hxy * dx*dy + hyy * dy^2)
    hess_term = 0.5 * (data_i[2] * dx**2 + 2.0 * data_i[3] * dx * dy + data_i[4] * dy**2)
    
    return phi_i + grad_term + hess_term

def kernel_grad(rij, r, h):
    # On s'assure que r et rij ont au moins une dimension
    r_arr = np.atleast_1d(r)
    rij_arr = np.atleast_2d(rij)
    
    if Kernel:        
        # --- WENDLAND C2 ÉTENDU À 2*h ---
        # Variable d'échelle alignée sur le support de taille 2*h
        q_tilde = r_arr / (2.0 * h)

        # Normalisation 2D globale pour un support de rayon (2*h)
        sigma = 7.0 / (4.0 * np.pi * h**2)

        # Gradient SCALAIRE analytique corrigé pour l'échelle 2*h
        # dW/dr = -10 * (sigma / h) * q_tilde * (1 - q_tilde)^3
        sigma_g = sigma / h
        dW_dr = np.where(q_tilde <= 1.0, -10.0 * sigma_g * q_tilde * (1.0 - q_tilde)**3, 0.0)

        # Sécurité division par zéro
        epsilon_safe = np.where(r_arr < 1e-10 * h**2, 1e-10 * h**2, r_arr)

        # Gradient VECTORIEL = (dW/dr) * (rij / r)
        grad_W = (dW_dr[:, np.newaxis] / epsilon_safe[:, np.newaxis]) * rij_arr

        if np.isscalar(r):
            return grad_W[0], dW_dr[0]

        return grad_W, dW_dr
        
    else:   
        # --- CUBIC SPLINE STANDARD (Support 2*h) ---
        q = r_arr / h

        alpha = 10.0 / (7.0 * np.pi * h**2)
        sigma_g = alpha / h  # 1/h^3

        # Gradient SCALAIRE (Monaghan 1992)
        dW_dr = np.where(q <= 1.0, sigma_g * (-3.0 * q + 2.25 * q**2),
                         np.where(q <= 2.0, -sigma_g * 0.75 * (2.0 - q)**2, 0.0))
        
        epsilon_safe = np.where(r_arr < 1e-10 * h**2, 1e-10 * h**2, r_arr)
        
        grad_W = (dW_dr[:, np.newaxis] / epsilon_safe[:, np.newaxis]) * rij_arr
        
        if np.isscalar(r):
            return grad_W[0], dW_dr[0]

        return grad_W, dW_dr


# TEST DU CORRECTIF (Cas analytique: uniform flow)
def test_kernel_grad_uniform_flow():
    """
    Test: Pour un écoulement uniforme v=cste,
    ∂p/∂x = 0 (pas de gradient de pression)
    Donc: ∑ᵢ pᵢ ∇Wᵢⱼ doit = 0
    
    Référence: Colagrossi & Landrini (2003)
    """
    # Setup
    np.random.seed(42)
    N = 100
    h = 0.1
    
    # Particules dispersées aléatoirement
    positions = np.random.uniform(-1, 1, (N, 2))
    
    # Pression uniforme (test case)
    pressure = np.ones(N) * 1e5
    
    # Calculer ∑ pᵢ ∇W
    grad_sum = np.zeros(2)
    for i in range(N):
        for j in range(N):
            if i != j:
                rij = positions[j] - positions[i]
                r = np.linalg.norm(rij)
                grad_W, _ = kernel_grad(rij.reshape(1, 2), 
                                        np.array([r]), h)
                grad_sum += pressure[j] * grad_W[0]
    
    # Doit être ≈ 0
    error = np.linalg.norm(grad_sum)
    assert error < 1e-6, f"Erreur gradient: {error} (devrait être ~0)"
    print(f"✓ Test kernel_grad: gradient sum = {error:.2e} < 1e-6")
    
    return True

# À définir dans vos paramètres globaux (ex: égale à la pression hydrostatique max)
# P_b = rho0 * g_acc * H_water 
#P_b = 0.01 * B # Une fraction de B fonctionne souvent très bien en WCSPH
# ========== PRESSION DE FOND - RÉFÉRENCE MORRIS 1997 ==========
# Morris et al. (1997) recommande:
#   P_b ≈ ρ₀ · g · H_max
# pour assurer que la pression reste positive même avec ρ légèrement < ρ₀
# et éviter les pressions négatives (non physiques)

# Déterminer H_max du domaine:
if testCase == 'DamBreak':
    H_domain = H_column  # Hauteur colonne d'eau initiale
elif testCase == 'car':
    H_domain = 0.0#H_water + np.tan(theta) * L_slope   # Profondeur bassin
elif testCase == 'funnel':
    H_domain = 0.0      # Hauteur approx. du funnel
elif testCase == 'bodyF':
    H_domain = H_water   # Profondeur bassin
elif testCase == 'TGV' or testCase == 'TGVmesh' or testCase == 'AirFoil':
    H_domain = 1.0       # Domaine unitaire
elif testCase == 'Poiseuille' :
    H_domain = 0.0       # Domaine unitaire
elif testCase == 'Mill' :
    H_domain = 0.0       # Domaine unitaire
else:
    H_domain = 0.0       # Défaut conservateur

# Calcul physique:
P_b = rho0 * g_acc * H_domain

print(f"[INFO] P_b (Morris 1997) = ρ₀·g·H_max = {P_b:.0f} Pa")
print(f"       B (paramètre Tait) = {B:.0f} Pa")
print(f"       Ratio P_b/B = {P_b/B:.4f}")


# Paramètre global (à définir en haut):
# P_b = rho0 * g_acc * H_water  # Pression hydrostatique
# ou fallback:
#P_b = rho0 * g_acc * 0.04  # H_water typical


def get_pressure(rho, rho_bounds=(0.5, 2.0)):
    """
    Équation d'état Tait avec clipping symétrique.
    
    p = B * [(ρ/ρ₀)^γ - 1] + P_b
    
    Référence: Morris, J.P., Fox, P.J., & Zhu, Y. (1997)
               "Modeling low Reynolds number incompressible flows using SPH"
               Journal of Computational Physics 136:214-226
    
    Arguments:
        rho: densité (array)
        rho_bounds: (ratio_min, ratio_max) pour clipping symétrique
                    Default: (0.5, 2.0) = [0.5*rho₀, 2.0*rho₀]
    """
    
    rho_min = rho0 * rho_bounds[0]
    rho_max = rho0 * rho_bounds[1]
    
    # Clipping SYMÉTRIQUE
    rho_clipped = np.clip(rho, rho_min, rho_max)
    
    # Équation Tait standard (Morris 1997)
    p_tait = B * (np.power(rho_clipped / rho0, gamma) - 1.0)
    
    # Pression de fond (assure p > 0 si ρ ≈ ρ₀)
    p = p_tait + P_b
    
    # NOUVEAU: Logging des dépassements
    clipped_low = np.sum(rho < rho_min)
    clipped_high = np.sum(rho > rho_max)
    
    if clipped_low > 0 or clipped_high > 0:
        total_clipped = clipped_low + clipped_high
        pct = 100.0 * total_clipped / len(rho)
        if ModeDEBUG:
            print(f"⚠️  Clipping densité: {clipped_low} basses, "
                  f"{clipped_high} hautes ({pct:.2f}% total)")
    
    return p


def get_density(p, p_threshold=0.1):
    """
    Inversion de l'équation d'état Tait.
    
    Inversé mathématiquement de: p = B * [(ρ/ρ₀)^γ - 1] + P_b
    
    Références:
        - Morris et al. (1997)
        - Cole, R.H. (1948). "Underwater Explosions." Princeton U. Press
    
    Arguments:
        p: pression (array)
        p_threshold: fraction de B pour warning si |p_dynamic| trop grande
    """
    
    p_dynamic = p - P_b
    
    # Vérifier avant inversion
    max_p_ratio = np.max(np.abs(p_dynamic) / B) if B > 0 else 0
    if max_p_ratio > p_threshold:
        if ModeDEBUG:
            print(f"⚠️  Pression anomale: max(|p_dynamic|/B) = {max_p_ratio:.3f}")
    
    # Clip DOUX: limité à ±50% variation relative
    # Justification: Garder exponent = p_dynamic/B + 1 > 0.5
    p_dynamic_clipped = np.clip(p_dynamic, -0.5*B, 50.0*B)
    
    # Inversion: p/B + 1 = (ρ/ρ₀)^γ
    exponent = p_dynamic_clipped / B + 1.0
    
    # Sécurité: exponent DOIT être > 0
    assert np.all(exponent > 0), \
        f"Erreur: exponent ≤ 0 trouvé! min={np.min(exponent)}"
    
    # Inversion finalisée
    rho = rho0 * np.power(exponent, 1.0 / gamma)
    
    return rho

def poiseuille_analytical(y, t, L, Fe_x, mu, nu, max_terms=100):
    """Calcule la vitesse théorique u(y, t) pour Poiseuille transitoire."""
    # Terme stationnaire (la parabole asymptotique)
    u_steady = (Fe_x / (2.0 * mu)) * y * (L - y)
    
    # Somme de la série transitoire
    u_transient = np.zeros_like(y)
    for n in range(max_terms):
        k = 2 * n + 1
        term = (1.0 / k**3) * np.sin(k * np.pi * y / L) * np.exp(-(k**2) * np.pi**2 * nu * t / L**2)
        u_transient += term
        
    u_transient *= (4.0 * Fe_x * L**2) / (np.pi**3 * mu)
    return u_steady - u_transient

def plot_poiseuille_comparison(pos, vel, types, t, L, Fe_x, mu, nu, save_dir="."):
    """
    Extrait le profil SPH et le compare à la solution analytique au temps t.
    """
    mask_f = (types == FLUID)
    pos_f = pos[mask_f]
    vel_f = vel[mask_f]
    
    # 1. Sélection d'une bande verticale au centre pour éviter les effets de bord de grille
    # On prend les particules dont le x est proche du milieu (ex: L/2 +- dx)
    x_mid = L / 2.0
    band_width = dx * 1.0  
    condition_band = np.abs(pos_f[:, 0] - x_mid) < band_width
    
    y_sph = pos_f[condition_band, 1]
    u_sph = vel_f[condition_band, 0] # Vitesse longitudinale u
    
    # Trier par hauteur y pour un joli tracé de courbe
    sort_idx = np.argsort(y_sph)
    y_sph_sorted = y_sph[sort_idx]
    u_sph_sorted = u_sph[sort_idx]
    
    # 2. Calcul de la solution analytique sur un maillage fin de référence
    y_fine = np.linspace(0, L, 200)
    u_ana = poiseuille_analytical(y_fine, t, L, Fe_x, mu, nu)
    
    # 3. Tracé du graphique
    plt.figure(figsize=(6, 5))
    plt.plot(u_ana, y_fine, 'k-', lw=2, label=f'Analytique (t = {t:.2f}s)')
    plt.scatter(u_sph_sorted, y_sph_sorted, color='red', edgecolor='k', s=25, alpha=0.8, label='SPH (Bande centrale)')
    
    plt.xlim(-0.05, U0 * 1.1)
    plt.ylim(0, L)
    plt.xlabel('Vitesse longitudinale u [m/s]')
    plt.ylabel('Hauteur du canal y [m]')
    plt.title(f'Profil de Poiseuille transitoire (Re = 10) à t = {t:.2f}s')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    
    plt.savefig(f"{save_dir}/poiseuille_profile_t_{t:.2f}.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[OK] Profil extrait et sauvegardé pour t = {t:.2f}s")
# Vérification de cohérence:
def test_pressure_density_inverse():
    """Vérifie que get_density(get_pressure(rho)) ≈ rho"""
    rho0=1000
    rho_test = np.linspace(rho0 * 0.99, rho0 * 1.01, 10)
    p_test = get_pressure(rho_test)
    rho_recovered = get_density(p_test)
    error = np.max(np.abs(rho_recovered - rho_test) / rho_test)
    print(f"Erreur inverse: {error:.2e}")
    assert error < 1e-10, "get_density n'est pas l'inverse de get_pressure!"
    
#=========== TESTS UNITAIRES SPH ===========
def run_unit_tests():
    
    """Suite de tests pour valider les corrections."""
    print("\n" + "="*60)
    print("EXÉCUTION TESTS UNITAIRES SPHIRIT")
    print("="*60)
    
    # TEST 1: Inverses pressure-density
    print("\n[TEST 1] Cohérence get_pressure ↔ get_density")
    rho0=1000
    rho_test = np.linspace(rho0 * 0.99, rho0 * 1.01, 10)
    p_test = get_pressure(rho_test)
    rho_recovered = get_density(p_test)
    print(f"DEBUG: rho_test[0] = {rho_test[0]}")
    print(f"DEBUG: rho_recovered[0] = {rho_recovered[0]}")
    print(f"DEBUG: h = {h}")
    error = np.max(np.abs(rho_recovered - rho_test) / rho_test)
    print(f"  ✓ Erreur relative max: {error:.2e} (tolérance: 1e-10)")
    assert error < 1e-10, f"❌ ÉCHOUÉ: erreur {error}"
    
    # TEST 2: Limiter van Albada
    print("\n[TEST 2] Limiteur van_albada() - Cas test")
    test_cases = [
        (1.0, 1.0, "Cas basique"),
        (1.0, -1.0, "Signes opposés"),
        (0.5, 0.3, "Petites pentes"),
    ]
    for a, b, desc in test_cases:
        phi = van_albada(a, b)
        print(f"  ✓ van_albada({a}, {b}) = {phi:.4f} ({desc})")
        if a * b <= 0:
            assert abs(phi) < 1e-10, f"❌ Pentes opposées devrait donner 0"
    
    # TEST 3: CFL dynamique
    print("\n[TEST 3] Paramètres CFL")
    u_max = 5.0  # m/s
    dt_cfl_test = CFL * h / (c0 + u_max)
    print(f"  ✓ Pour u_max={u_max} m/s, c0={c0:.1f} m/s:")
    print(f"    dt_CFL = {dt_cfl_test:.6f} s")
    assert dt_cfl_test > 0, "❌ dt_CFL doit être positif"
    
    # TEST 4: Kernel normalization (optionnel - calcul lourd)
    print("\n[TEST 4] Normalisation kernel (OPTIONNEL)")
    print("  ⚠️  Test complet déactivé pour performance")
    print("  → Vérifier manuellement: ∫∫ W(r,h) r dr dθ = 1")
    
    print("\n" + "="*60)
    print("✅ TOUS LES TESTS UNITAIRES PASSENT")
    print("="*60 + "\n")


    
    
def minmod_vec(a, b):
    # Version vectorisée pour Numpy
    res = np.zeros_like(a)
    mask = a * b > 0
    res[mask] = np.sign(a[mask]) * np.minimum(np.abs(a[mask]), np.abs(b[mask]))
    return res

def minmod(a, b):
    if a*b <= 0:
        return 0
    else:
        return np.sign(a) * min(abs(a), abs(b))   
        
def van_albada(a, b):
    # Convertir en tableaux NumPy (sans copier si c'en est déjà un)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    """
    Version vectorisée du limiteur de pente Van Albada pour des tableaux NumPy.
    Prend deux tableaux de pentes 'a' et 'b' de même dimension.
    """
    eps = 2.2e-16
    # Étape 1 : On crée le masque booléen où les deux pentes sont de même signe
    mask = a * b > 0.0
    
    # Étape 2 : On initialise le tableau de sortie à 0.0 (ordre 1 par défaut)
    num_elements = a.shape if isinstance(a, np.ndarray) else 1
    lim_slope = np.zeros_like(a)
    
    # Étape 3 : On applique la formule uniquement là où c'est nécessaire
    if np.any(mask):
        a_m = a[mask]
        b_m = b[mask]
        a2 = a_m**2
        b2 = b_m**2
        
        lim_slope[mask] = (a_m * (b2 + eps) + b_m * (a2 + eps)) / (a2 + b2 + 2.0 * eps)
        
    return lim_slope
    
def riemann_solver(rhoL, uL, pL, rhoR, uR, pR, c0, P_b, is_wall=False):
    #Solveur Rusanov / Local Lax–Friedrichs linéarisé (forme acoustique)
    rho_avg = 0.5 * (rhoL + rhoR)
    if rho_avg < 1e-6: return 0.5 * (uL + uR), 0.5 * (pL + pR)
    beta = 0.5 
    # Low-Mach correction (Thornber 2008, eq.33)
    # Par défaut, le solveur acoustique exact utilise beta = 0.5
    beta_p = 0.5
    beta_u = 0.5
    
    # La correction Bas-Mach n'est appliquée que dans le fluide (pas aux parois)
#     if not is_wall:
#         # Le Mach doit dépendre de la vitesse locale de l'écoulement, pas du saut local
#         # Ici, on utilise une approximation locale via la moyenne des vitesses normales
#         u_fluid_local = 0.5 * (abs(uL) + abs(uR))
#         Ma_loc = u_fluid_local / (c0 + 1e-12)
        
#         # Thornber modifie la dissipation de la variable de pression
#         # pour éviter la sur-dissipation de l'énergie cinétique à bas Mach
#         theta = np.clip(Ma_loc, 0.01, 1.0)
#         beta_p = 0.5 * theta
#         beta_u = 0.5  # La dissipation en vitesse reste active pour éviter le découplage
    # ===============================================
    pL_dyn, pR_dyn = pL , pR 
    p_star_dyn = 0.5*(pL_dyn + pR_dyn) - beta_p * rho_avg * c0 * (uR - uL)
    u_star = 0.5*(uL + uR)         - beta_u * (pR_dyn - pL_dyn) / (rho_avg*c0 + 1e-12)    
    # On rajoute P_b pour renvoyer une pression TOTALE absolue
    p_star = p_star_dyn  
    return u_star, p_star

def get_tgv_solution(pos, t, L, U0, nu):
    k = 2 * np.pi / L
    decay = np.exp(-2 * k**2 * nu * t)
    u =  U0 * np.sin(k*pos[:,0]) * np.cos(k*pos[:,1]) * decay
    v = -U0 * np.cos(k*pos[:,0]) * np.sin(k*pos[:,1]) * decay
    p = (rho0 * U0**2 / 4.0) * (np.cos(2*k*pos[:,0]) + np.cos(2*k*pos[:,1])) * decay**2
    return np.column_stack((u, v)), p
# --------------------------
# Périodicité
# --------------------------

def periodic_rij(xi, xj, testCase, L):
    rij = -(xj - xi)
    if testCase in ['TGV', 'TGVmesh','AirFoil']:
        # Correction sur X et Y
        rij -= L * np.round(rij / L)
    elif testCase == 'Poiseuille':
        # Correction STRICTEMENT sur la première composante (X)
        rij[0] -= L * np.round(rij[0] / L)
        # Pas de correction sur Y !
    return rij


def compute_v_mesh_full(pos, vel_f, vol_f, h, dt,  mode,shepard, tree, pairs):
    n_p = len(pos)
    FS = np.zeros_like(pos)  
    F1 = np.zeros_like(pos) 
    F2 = np.zeros_like(pos)  
    # Calcul v_mesh selon le mode
    if mode == 'lagrangian':
        
        return vel_f.copy(), np.zeros(n_p), np.zeros(n_p), np.zeros((n_p,2)), np.zeros((n_p,2))
    if mode == 'eulerian':
        v_mesh = np.zeros_like(vel_f)
        return v_mesh, np.zeros(n_p), np.zeros(n_p), np.zeros((n_p,2)), np.zeros((n_p,2))
    if mode == 'ale':         
        v_mesh, F1, FS, F2, grad_W2, grad_FS, lloyd_shift = compute_v_mesh_coupled(pos, vel_f, vol_f, h, dt, shepard, tree, pairs,
                           alpha, beta_geom, mode_ale)        
        
    return v_mesh, F1, FS, grad_W2, grad_FS
    
def compute_renormalization_matrices_robust(pos, vol, h, all_indices, shepard_coeff, testCase, Periodic, L_domain):
    """
    Calcule Li avec une structure de voisinage et des conventions de signes
    strictement identiques au gradient SPH vectorisé.
    """
    num_particles = pos.shape[0]
    M = np.zeros((num_particles, 2, 2))
    
    # Accumulation de M calquée sur la structure du gradient
    for i in range(num_particles):
        idx = all_indices[i]
        if len(idx) <= 1: 
            continue
        
        # Calcul du vecteur rij (Même convention de signe que le gradient : x_j - x_i)
        if Periodic:
            # On permute pos[j] et pos[i] pour obtenir le vecteur (x_j - x_i) avec conditions périodiques
            rij = np.array([periodic_rij(pos[j], pos[i], testCase, L_domain) for j in idx])
        else:
            rij = pos[idx] - pos[i] # Forme (n_voisins, 2) -> (x_j - x_i)
            
        r = np.linalg.norm(rij, axis=1)
        
        valid = (r > 2.2e-16) & (r <= 2.0 * h)
        if not np.any(valid): 
            continue
        
        rij = rij[valid]
        r = r[valid]
        idx_valid = np.array(idx)[valid]
        
        # Calcul des gradients du noyau (Même g_w que le gradient !)
        g_w, _ = kernel_grad(rij, r, h) # Forme (n_voisins_valides, 2)
        
        # Sécurité : s'il n'y a pas assez de voisins valides après filtrage du noyau
        if g_w.shape[0] == 0:
            continue

        # Formule standard : M_i = Somme_j ( V_j * (x_j - x_i) x gradW_ij )
        # rij vaut (x_j - x_i), g_w vaut gradW_ij
        M[i] = np.einsum('k,ki,kj->ij', vol[idx_valid], rij, g_w)

    # --- Inversion, Analyse du Conditionnement & Régularisation ---
    dets = np.linalg.det(M)
    abs_dets = np.abs(dets)
    norm_M = np.linalg.norm(M, axis=(-2, -1))

    M_safe = M.copy()
    mask_singular = abs_dets <= 1e-15
    M_safe[mask_singular] = np.eye(2) 

    inv_M_safe = np.linalg.inv(M_safe)
    norm_invM = np.linalg.norm(inv_M_safe, axis=(-2, -1))
    cond_array = norm_M * norm_invM
    cond_array[mask_singular] = 1e6

    mask_regularize = (shepard_coeff < 0.2) | (abs_dets < 1e-7) | (cond_array > 100.0)
    mask_full_renorm = ~mask_regularize

    L_matrices = np.zeros_like(M)
    renorm_status = np.zeros(num_particles, dtype=int)

    if np.any(mask_regularize):
        epsilon = np.maximum(0.0, 0.5 - shepard_coeff[mask_regularize])
        M_reg = M[mask_regularize] + epsilon[:, None, None] * np.eye(2)[None, :, :]
        L_matrices[mask_regularize] = np.linalg.inv(M_reg)
        renorm_status[mask_regularize] = 1

    if np.any(mask_full_renorm):
        L_matrices[mask_full_renorm] = inv_M_safe[mask_full_renorm]
        renorm_status[mask_full_renorm] = 2
            
    return L_matrices, cond_array, renorm_status  
            
def add_periodic_ghosts(pos, types, h, L):
    """Ajoute des copies fantômes pour compléter le support aux bords"""
    idx_boundary = (pos[:, 0] < h) | (pos[:, 0] > L - h)
    
    if not np.any(idx_boundary):
        return pos, types
    
    # Copies en (-L, y) et (L, y)
    pos_left = pos[idx_boundary].copy()
    pos_left[:, 0] -= L
    
    pos_right = pos[idx_boundary].copy()
    pos_right[:, 0] += L
    
    pos_extended = np.vstack([pos, pos_left, pos_right])
    types_extended = np.concatenate([types, types[idx_boundary], types[idx_boundary]])
    
    return pos_extended, types_extended            
            
# --------------------------
# Entropie géométrique
# --------------------------
def compute_geometric_entropy(pos, vol, h,shepard):          
    S_G_local = (shepard - 1)**2   
    return S_G_local, np.mean(S_G_local)
# --------------------------
# Entropie de distribution compute_distribution_entropy(pos, vol_f, h,shepard,tree,pairs)
# --------------------------
def compute_distribution_entropy(pos, vol, h,shepard, tree,pairs):
    n_p = len(pos)    
    S_dist = -np.sum(0.0)       
    grad_S_dist = np.zeros_like(pos)   
    for i, j in pairs:
        if Periodic:
            rij = periodic_rij(pos[i], pos[j],testCase, L)
        else :
            rij = pos[i]-pos[j]
        r = np.linalg.norm(rij)
        if r < 2.2e-16: continue 
        grad_S_dist[i] = 0.0
    return grad_S_dist, S_dist

def compute_wasserstein_shift(pos, vol, h, tree, pairs):   
    n = len(pos)
    # On initialise des accumulateurs pour le centre de gravité
    num_centroid = np.zeros((n, 2))
    den_centroid = np.zeros(n)    
    # On parcourt les paires pour accumuler les poids
    for i, j in pairs:
        if Periodic:
            rij = periodic_rij(pos[i], pos[j],testCase, L)
        else :
            rij = pos[i]-pos[j]
        r = np.linalg.norm(rij)
        
        if r < 2.2e-16 or r > search_radius: continue 
        
        W = kernel_cubic(r, h)
        weight = W * vol[j]
        
        # Accumulation pour la particule i
        num_centroid[i] += pos[j] * weight
        den_centroid[i] += weight
        
        # Accumulation pour la particule j (symétrie)
        # On utilise vol[i] pour la contribution de i sur j
        weight_rev = W * vol[i]
        num_centroid[j] += pos[i] * weight_rev
        den_centroid[j] += weight_rev

    # Calcul final du shift : (Centroïde - Position Actuelle)
    shift = np.zeros_like(pos)
    # On ne calcule que là où il y a des voisins pour éviter div par 0
    mask = den_centroid > 2.2e-16
    
    # Formule : Shift = (Somme(pos_j * W_ij * V_j) / Somme(W_ij * V_j)) - pos_i
    centroid = num_centroid[mask] / den_centroid[mask][:, None]
    shift[mask] = centroid - pos[mask]
    
    return shift
   

def compute_mls_gradients_order2(pos, vol, field, h, tree, p_ref, cond_MLS, types):
    """
    Calcule le gradient et la Hessienne via MLS d'ordre 2.
    Renvoie un tableau de forme (N, 5) par composante : 
    [gx, gy, hxx, hxy, hyy]
    """
    n_p = len(pos)
    dim = 2          
    n_basis = 6      
    
    f_is_scalar = (field.ndim == 1)
    f = field[:, None] if f_is_scalar else field
    n_components = f.shape[1]
    
    # Structure de retour : (n_p, n_comp, 5)
    # [gx, gy, hxx, hxy, hyy]
    results = np.zeros((n_p, n_components, 5))
    
    all_indices = tree.query_ball_point(pos, search_radius)
    
    for i in range(n_p):
        if types[i] != FLUID: continue
            
        idx = all_indices[i]
        if len(idx) < n_basis: continue
            
        # Gestion voisinage (inclus Periodic)
        if Periodic:
            rij = -periodic_rij(pos[i], pos[idx], testCase, L)
        else:
            rij = (pos[idx] - pos[i])
            
        r = np.linalg.norm(rij, axis=1)
        valid = r > 2.2e-16
        if np.sum(valid) < n_basis: continue
            
        idx = np.array(idx)[valid]
        rij_scaled = rij[valid] / h
        x_s, y_s = rij_scaled[:, 0], rij_scaled[:, 1]
        
        P = np.column_stack([np.ones_like(x_s), x_s, y_s, x_s**2, x_s*y_s, y_s**2])  
        W = kernel_cubic(r[valid], h)
        weight = W * vol[idx]  
        
        M = P.T @ (weight[:, None] * P)  
        M += 2.2e-16 * np.eye(n_basis)
        
        df = f[idx] - f[i]  
        rhs = df.T @ (weight[:, None] * P)
           
        cond_MLS[i] = np.linalg.cond(M)                                 
        if cond_MLS[i] < 10000.0:
            coeffs = rhs @ np.linalg.pinv(M)
            
            # --- Remplissage du vecteur "complet" (n_comp, 5) ---
            results[i, :, 0] = coeffs[:, 1] / h              # grad_x
            results[i, :, 1] = coeffs[:, 2] / h              # grad_y
            results[i, :, 2] = 2.0 * coeffs[:, 3] / (h**2)   # hxx
            results[i, :, 3] = coeffs[:, 4] / (h**2)          # hxy
            results[i, :, 4] = 2.0 * coeffs[:, 5] / (h**2)   # hyy
            
            p_ref[i] = 2             
        else:  
            results[i, :, :] = 0.0
            p_ref[i] = 0
                    
    # Retourne (n_p, 5) si scalaire, sinon (n_p, n_comp, 5)
    return results[:, 0, :] if f_is_scalar else results




def compute_mls_gradients(pos, vol, field, h, neighbor_lists, p_ref, cond_MLS, types, testCase, L, Periodic):
    """
    Version OPTIMISÉE : Prend directement 'neighbor_lists' en argument pour éviter
    de recalculer les voisinages par arbre à chaque appel.
    """
    n_p = len(pos)
    dim = pos.shape[1]
    
    f_is_scalar = (field.ndim == 1)
    f = field[:, None] if f_is_scalar else field
    n_components = f.shape[1]
    
    gradients = np.zeros((n_p, n_components, dim))
    
    for i in range(n_p):   
        if types[i] != FLUID:
            continue
            
        # Extraction directe des voisins pré-calculés
        idx = neighbor_lists[i]
        
        # Sécurité ordre 1 : besoin de dim + 1 voisins
        if len(idx) < dim + 1:
            continue
            
        # Calcul du vecteur r_ij selon la convention SPH standard (j - i)
        if Periodic:
            rij = -periodic_rij(pos[i], pos[idx], testCase, L)
        else:
            rij = (pos[idx] - pos[i])
            
        r = np.linalg.norm(rij, axis=1)
        valid = r > 2.2e-16
        
        if not np.any(valid):
            continue
            
        idx = np.array(idx)[valid]
        rij = rij[valid]
        r = r[valid]
        
        W = kernel_cubic(r, h)
        weight = W * vol[idx]  
        
        # --- CORRECTION NORMALISATION MLS ---
        rij_s = rij / h  
        P = np.column_stack([np.ones(len(idx)), rij_s])  
        
        M = P.T @ (weight[:, None] * P)  
        M += 1.0e-12 * np.eye(dim + 1) # Régularisation robuste
        
        df = f[idx] - f[i]  
        rhs = df.T @ (weight[:, None] * P)
        
        cond_MLS[i] = np.linalg.cond(M)  
        
        # Seuil de conditionnement assoupli (1e6) pour éviter les gradients nuls parasites sur le TGV
        if cond_MLS[i] < 100:
            inv_M = np.linalg.inv(M)
            coeffs = rhs @ inv_M
            gradients[i] = coeffs[:, 1:dim+1] / h
            p_ref[i] = 1             
        else:  
            gradients[i] = 0.0
            p_ref[i] = 0

    return gradients[:, 0, :] if f_is_scalar else gradients


def add_position_noise(points, amplitude):
    r = amplitude * np.sqrt(np.random.rand(len(points)))
    theta = 2*np.pi*np.random.rand(len(points))
    noise = np.column_stack((r*np.cos(theta), r*np.sin(theta)))
    return points + noise

def smoothstep(a,b,y):
    t=np.clip((y-a)/(b-a),0,1)
    return t*t*(3-2*t)

def get_funnel_radius(y):
    y1 = 0.020
    y2 = 0.0321
    if y < y1:
        return 0.003
    elif y < y2:
        s = smoothstep(y1,y2,y)
        return (1-s)*0.003 + s*0.025
    else:
        return 0.025

def get_funnel_derivative(y):
    y1=0.020
    y2=0.0321
    if y<y1 or y>y2:
        return 0.0
    t=(y-y1)/(y2-y1)
    ds=6*t*(1-t)/(y2-y1)
    return (0.025-0.003)*ds
# --------------------------
def compute_energy_functionals(pos, vol, h, shepard, pairs):
    """
    Calcule F1 (Wasserstein-2), F_S (Shepard defect), F2 = F1 + alpha*F_S.
    Retourne également le gradient complet pour la descente.
    """
    n = len(pos)
    
    # --- Terme F_S : défaut de Shepard ---
    F_S = 0.5 * np.sum(vol * (shepard - 1.0)**2)
    
    # --- Terme F_1 : Wasserstein-2 centroïdal ---
    F_1 = 0.0
    num_c = np.zeros((n, 2))  # numérateur centroïde
    den_c = np.zeros(n)       # = shepard (vérification)
    variance_dist = np.zeros(n)  # variance locale des |x_ij|^2

    for i, j in pairs:
        
        if Periodic:
            rij = periodic_rij(pos[i], pos[j],testCase, L)
        else :
            rij = pos[i]-pos[j]
            
        r = np.linalg.norm(rij)
        if r < 2.2e-16 or r > search_radius: continue
        
        W = kernel_cubic(r, h)
        r2 = r * r
        
        # Contributions centroïde pour i (utilisation de j)
        num_c[i] += pos[j] * W * vol[j]
        den_c[i] += W * vol[j]
        variance_dist[i] += W * vol[j] * r2
        
        # Contributions centroïde pour j (utilisation de i)
        num_c[j] += pos[i] * W * vol[i]
        den_c[j] += W * vol[i]
        variance_dist[j] += W * vol[i] * r2

    # Centroïdes et shift de Lloyd
    mask = den_c > 2.2e-16
    centroid = np.where(mask[:, None], num_c / np.where(mask[:, None], den_c[:, None], 1), pos)
    lloyd_shift = pos - centroid  # = x_i - c_i
    
    # F_1 = (1/2) * sum_i (variance_dist[i] / sigma_i)
    F_1 = 0.5 * np.sum(
        np.where(mask, variance_dist / np.where(mask, den_c, 1), 0.0)
    )
    
    return F_1, F_S, lloyd_shift, centroid, variance_dist

def compute_grad_shepard_corrected(pos, vol, h, shepard, pairs):
    """
    Gradient de F_S = (1/2) * sum_i V_i * (sigma_i - 1)^2
    CORRIGÉ : inclut le facteur V_i manquant.
    grad_FS[i] = V_i * sum_j V_j * [(sigma_i-1) + (sigma_j-1)] * grad_W_ij
    """
    n = len(pos)
    grad_FS = np.zeros_like(pos)
    error = shepard - 1.0  # sigma_i - 1
    
    for i, j in pairs:
        if Periodic:
            rij = periodic_rij(pos[i], pos[j],testCase, L)
        else :
            rij = pos[i]-pos[j]
        r = np.linalg.norm(rij)
        if r < 2.2e-16 or r > search_radius: continue
        
        gW, _ = kernel_grad(rij, r, h)
        
        # Facteur symétrique + facteur V_i * V_j (CORRECTION vs code actuel)
        coeff_i = vol[i] * vol[j] * (error[i] + error[j])
        
        grad_FS[i] += coeff_i * gW
        grad_FS[j] -= coeff_i * gW  # antisymétrie du grad W
    
    return grad_FS



def compute_grad_wasserstein_full(pos, vol, h, shepard, pairs, lloyd_shift, variance_dist):
    """
    Gradient de F_1 incluant :
      - Terme Lloyd : x_i - c_i
      - Correction d'ordre 2 (terme en nabla W * |x_ij|^2)
    """
    n = len(pos)
    grad_W2 = lloyd_shift.copy()  # terme dominant : x_i - c_i
    lloyd_sq = np.sum(lloyd_shift**2, axis=1)  # |x_i - c_i|^2
    
    mask_sigma = shepard > 2.2e-16
    inv_sigma = np.where(mask_sigma, 1.0 / np.where(mask_sigma, shepard, 1), 0.0)
    
    for i, j in pairs:
        if Periodic:
            rij = periodic_rij(pos[i], pos[j],testCase, L)
        else :
            rij = pos[i]-pos[j]
        r = np.linalg.norm(rij)
        if r < 2.2e-16 or r > search_radius: continue
        
        gW, _ = kernel_grad(rij, r, h)
        r2 = r * r
        
        # Correction ordre 2 pour i : (1/2*sigma_i) * V_j * nabla W_ij * (|x_ij|^2 - |c_i - x_i|^2)
        corr2_i = 0.5 * inv_sigma[i] * vol[j] * (r2 - lloyd_sq[i]) * gW
        grad_W2[i] += corr2_i
        
        # Correction ordre 2 pour j
        corr2_j = 0.5 * inv_sigma[j] * vol[i] * (r2 - lloyd_sq[j]) * (-gW)
        grad_W2[j] += corr2_j
    
    return grad_W2

def compute_v_mesh_coupled(pos, vel, vol, h, dt, shepard, tree, pairs,
                           alpha, beta_geom, mode_ale):
    """
    Calcule la vitesse de grille ALE basée sur la descente de gradient de F2.
    
    Modes :
      'lloyd'   : descente F1 seul (terme dominant Lloyd)
      'wass'    : descente F1 complet (avec correction ordre 2)  
      'shepard' : descente F_S seul (correction Shepard)
      'coupled' : descente F2 = F1 + alpha*F_S  ← NOUVEAU
    """
    n = len(pos)
    v_mesh = vel.copy()
    
    # 1. Calcul des fonctionnelles et termes intermédiaires
    F1, FS, lloyd_shift, centroid, variance_dist = compute_energy_functionals(
        pos, vol, h, shepard, pairs
    )
    F2 = F1 + alpha * FS
    
    # 2. Gradient selon le mode
    if mode_ale in ('lloyd', 'wass', 'coupled'):
        grad_W2 = compute_grad_wasserstein_full(
            pos, vol, h, shepard, pairs, lloyd_shift, variance_dist
        )
    else:
        grad_W2 = np.zeros_like(pos)
    
    if mode_ale in ('shepard', 'coupled'):
        grad_FS = compute_grad_shepard_corrected(pos, vol, h, shepard, pairs)
    else:
        grad_FS = np.zeros_like(pos)
    
    # 3. Gradient total de F2
    grad_total = grad_W2 + alpha * grad_FS
    
    # 4. Normalisation locale (preserves spatial information)
    # ||x_i - c_i|| est bornée par 2h ; normaliser par h donne un shift adimensionnel
    norms = np.linalg.norm(grad_total, axis=1, keepdims=True)
    h_inv_norm = norms / h  # adimensionnel
    safe_norm = np.maximum(h_inv_norm, 1e-10)
    direction = grad_total / safe_norm  # unité : longueur
    
    # 5. Vitesse de shift proportionnelle à c0 et beta_geom
    v_shift = -beta_geom * c0 * (direction / h)  # dimension : m/s
    
    # 6. Application (seulement pour les particules bien supportées)
    mask_interior = shepard > 0.5  # éviter les zones mal définies
    v_shift[~mask_interior] *= 0.0
    v_mesh += v_shift
    
    # 7. Conservation de la quantité de mouvement globale du shift
    # (optionnel : recentrage pour éviter la dérive)
    # momentum = np.sum(vol[:, None] * v_shift, axis=0) / np.sum(vol)
    # v_shift -= momentum
    
    return v_mesh, F1, FS, F2, grad_W2, grad_FS, lloyd_shift

    



# --------------------------------------------------
# Distance point → segment (pour normales)
# --------------------------------------------------
def distance_to_segment(p, a, b):
    ap = p - a
    ab = b - a
    t = np.dot(ap, ab) / np.dot(ab, ab)
    t = np.clip(t, 0, 1)
    closest = a + t * ab
    return np.linalg.norm(p - closest), closest



# --------------------------------------------------
# POINT IN POLYGON   22
# --------------------------------------------------
def point_in_polygon(x, y, polygon):
    inside = False
    j = len(polygon) - 1

    for i in range(len(polygon)):
        xi, yi = polygon[i]
        xj, yj = polygon[j]

        if ((yi > y) != (yj > y)) and \
            (x < (xj - xi) * (y - yi) / (yj - yi + 2.2e-16) + xi):
            inside = not inside

        j = i

    return inside


# --------------------------------------------------
# AJOUT POLYGONE REMPLI
# --------------------------------------------------
def fill_polygon(polygon):

    polygon = np.array(polygon)

    xmin, ymin = polygon.min(axis=0)
    xmax, ymax = polygon.max(axis=0)

    pts = []

    xs = np.arange(xmin, xmax, dx)
    ys = np.arange(ymin, ymax, dx)

    for x in xs:
        for y in ys:
            if point_in_polygon(x, y, polygon):
                pts.append([x, y])

    return pts


# --------------------------------------------------
# AJOUT ROUE (CERCLE)
# --------------------------------------------------
def fill_circle(cx, cy, radius):

    pts = []

    xs = np.arange(cx-radius, cx+radius, dx)
    ys = np.arange(cy-radius, cy+radius, dx)

    for x in xs:
        for y in ys:
            if (x-cx)**2 + (y-cy)**2 <= radius**2 :
                pts.append([x, y])

    return pts


# --------------------------------------------------
# CREATE CAR ULTRA PRÉCIS
# --------------------------------------------------


def create_car(center_x, center_y, mass_total):
    # ==================================================
    # DIMENSIONS RÉELLES & GÉNÉRATION
    # ==================================================
    L_total = 4.020
    H_total = 1.785
    
    car_pts = []

    body_polygon = [
        (-1.80, -0.30), (-1.90,  0.40), (-1.40,  0.75), (-0.30,  0.85),
        ( 0.60,  0.85), ( 0.90,  0.65), ( 1.50,  0.45), ( 1.90,  0.20),
        ( 2.01, -0.10), ( 1.60, -0.35), (-1.80, -0.30)
    ]
    car_pts += fill_polygon(body_polygon)

    wheel_radius = 0.40 
    rear_wheel_center = (-1.10, -0.50)
    front_wheel_center = (1.20, -0.50)
    car_pts += fill_circle(*rear_wheel_center, wheel_radius)
    car_pts += fill_circle(*front_wheel_center, wheel_radius)

    # Suppression doublons
    # ==================================================
    # CORRECTION DES SUPERPOSITIONS (GRID SNAPPING)
    # ==================================================
    pts_array = np.array(car_pts)
    
    # On divise par dx, on arrondit à l'entier, et on remultiplie par dx.
    # Cela force CHAQUE particule (roue ou carrosserie) à s'aimanter 
    # sur les intersections d'une seule et même grille globale espacée de dx.
    pts_snapped = np.round(pts_array / dx) * dx
    
    # Maintenant que tout le monde est sur la même grille, 
    # les points superposés ont EXACTEMENT les mêmes coordonnées.
    # On peut donc utiliser unique() pour nettoyer :
    # (On arrondit légèrement pour éviter les erreurs flottantes de Python)
    pts_unique = np.unique(np.round(pts_snapped, 6), axis=0)
    pts = pts_unique.copy()

    # ==================================================
    # 1. CALCUL DES NORMALES (AVANT LE SCALING) ✅
    # ==================================================
    # Ici l'espacement est encore parfaitement égal à `dx`
    pts_set = set(map(tuple, np.round(pts, 6)))
    norms = np.zeros_like(pts)
    centroid_temp = np.mean(pts, axis=0)
    
    dirs = np.array([
        [ dx, 0], [-dx, 0], [0,  dx], [0, -dx],
        [ dx, dx], [ dx,-dx], [-dx, dx], [-dx,-dx],
    ])

    for i, p in enumerate(pts):
        normal = np.zeros(2)
        for d in dirs:
            neighbour = tuple(np.round(p + d, 6))
            if neighbour not in pts_set:
                normal += d

        if np.linalg.norm(normal) > 0:
            outward = p - centroid_temp
            if np.dot(normal, outward) < 0:
                normal *= -1
            norms[i] = normal / np.linalg.norm(normal)

    # ==================================================
    # 2. CONVERSION ET TRANSLATION (SCALING)
    # ==================================================
    curr_width = np.max(pts[:,0]) - np.min(pts[:,0])
    curr_height = np.max(pts[:,1]) - np.min(pts[:,1])
    
    scale_x = L_total / curr_width
    scale_y = H_total / curr_height

    # On applique le scaling aux points
    pts[:,0] *= scale_x
    pts[:,1] *= scale_y

    # ==================================================
    # 3. CORRECTION MATHÉMATIQUE DES NORMALES
    # ==================================================
    # Règle mathématique : Si on étire la géométrie d'un facteur S, 
    # le vecteur normal doit être multiplié par l'inverse de S (1/S).
    norms[:, 0] /= scale_x
    norms[:, 1] /= scale_y
    
    # On re-normalise les vecteurs normaux (pour qu'ils aient une longueur de 1)
    lengths = np.linalg.norm(norms, axis=1)
    mask = lengths > 0
    norms[mask] = norms[mask] / lengths[mask, np.newaxis]

    # Translation finale
    centroid = np.mean(pts, axis=0)
    pts[:,0] += (center_x - centroid[0])
    pts[:,1] += (center_y - centroid[1])
    centroid_final = np.mean(pts, axis=0)   # recomputer APRÈS translation
    rel = pts - centroid_final
    # ==================================================
    # MASSE + INERTIE
    # ==================================================
    m_part = mass_total / len(pts)
  
    inertia = np.sum(m_part * np.sum(rel**2, axis=1))

    return pts, m_part, inertia, norms
# --------------------------------------------------
# Création véhicule polygonal
# --------------------------------------------------

# def update_car_rigid_body(
#         pos,
#         vel,
#         accel,
#         m,
#         types,
#         dt,
#         car_vel,
#         car_omega,
#         car_mass,
#         car_I,
#         g):
#     """
#     Mise à jour du corps rigide 'voiture' à partir
#     des forces SPH accumulées dans accel.
#     """
    
#     # Particules appartenant à la voiture
#     mask_car = (types == CAR)

#     if not np.any(mask_car):
#         return car_vel, car_omega

#     # ----------------------------
#     # Centre de masse
#     # ----------------------------
#     car_center = np.mean(pos[mask_car], axis=0)

#     # ----------------------------
#     # Forces totales
#     # ----------------------------
#     forces_indiv = m[mask_car, None] * (accel[mask_car] + g)
#     total_force_car = np.sum(forces_indiv, axis=0)

#     # ----------------------------
#     # Moments
#     # ----------------------------
#     rel_pos = pos[mask_car] - car_center

#     total_torque_car = np.sum(
#         rel_pos[:, 0] * forces_indiv[:, 1]
#         - rel_pos[:, 1] * forces_indiv[:, 0]
#     )

#     # ----------------------------
#     # Accélérations rigides
#     # ----------------------------
#     acc_rigid = total_force_car / car_mass
#     alpha_rigid = total_torque_car / car_I

#     # ----------------------------
#     # Intégration vitesses rigides
#     # ----------------------------
#     # Intégration des vitesses (Résout l'erreur UnboundLocalError)
#             # Extraction de la position X max selon l'axe x de la voiture
#     # Intégration des vitesses
#     x_car = np.max(pos[types == CAR, 0])
    
#     # CORRECTION : Utiliser le vecteur vitesse global de la voiture (car_vel passé en argument)
#     # ou définir sa direction standard. Si car_vel initial est nul, on utilise un vecteur unitaire unitaire [1.0, 0.0]
#     norm = np.linalg.norm(car_vel)
#     if norm > 1e-8:
#         car_direction = car_vel / norm
#     else:
#         car_direction = np.array([1.0, 0.0]) # Direction par défaut (axe X)
        
#     # car_vel est maintenant un vecteur unique (2,)
#     car_vel = car_direction * carvelocityInlet(x_car)[0]
    
    
#     car_omega += alpha_rigid * dt

#     # ----------------------------
#     # Mise à jour du centre
#     # ----------------------------
#     car_center += car_vel * dt

#     # ----------------------------
#     # Rotation
#     # ----------------------------
#     d_theta = car_omega * dt

#     rot_mat = np.array([
#         [np.cos(d_theta), -np.sin(d_theta)],
#         [np.sin(d_theta),  np.cos(d_theta)]
#     ])

#     new_rel_pos = rel_pos @ rot_mat.T

#     # ----------------------------
#     # Update positions
#     # ----------------------------
#     pos[mask_car] = car_center + new_rel_pos

#     # ----------------------------
#     # Update vitesses particules
#     # ----------------------------
#     vel[mask_car] = (
#         car_vel
#         + np.vstack([
#             -car_omega * new_rel_pos[:, 1],
#              car_omega * new_rel_pos[:, 0]
#         ]).T
#     )

#     return car_vel, car_omega



def write_vtk(filename, pos, vel, v_mesh, pres, types, rho, vol, m, neighbors, shepard, wall_norms,
              S_G_local, S_dist, grad_S_G, grad_S_dist, cond_array, renorm_status, grad_p, grad_vel, grad_rho, p_ref, cond_MLS):
    
    n = len(pos)
    
    # Sécurité / Uniformisation des dimensions pour la vectorisation
    # (Permet d'accepter des tableaux de forme (N,) ou (N,1) de manière transparente)
    pres_2d = pres.reshape(-1, 1)
    rho_2d = rho.reshape(-1, 1)
    vol_2d = vol.reshape(-1, 1)
    types_2d = types.reshape(-1, 1)
    neighbors_2d = neighbors.reshape(-1, 1)
    shepard_2d = shepard.reshape(-1, 1)
    m_2d = m.reshape(-1, 1)
    cond_array_2d = cond_array.reshape(-1, 1)
    renorm_status_2d = renorm_status.reshape(-1, 1)
    p_ref_2d = p_ref.reshape(-1, 1)
    cond_MLS_2d = cond_MLS.reshape(-1, 1)
    
#     S_G_local et S_dist sont déjà des tableaux (N,), on les passe en (N, 1)
    S_G_to_write = S_G_local.reshape(-1, 1)
    S_dist_to_write = S_dist.reshape(-1, 1)
    
    # Matrice de zéros pour ajouter la composante Z=0 aux vecteurs 2D
    # Matrice de zéros pour ajouter la composante Z=0 aux vecteurs 2D
    zeros = np.zeros((n, 1))

    # =====================================================================
    # FIX DE SÉCURITÉ : Si wall_norms ne contient que les particules de mur
    # =====================================================================
    if len(wall_norms) != n:
        mask_wall = (types != FLUID)
        wall_norms_complete = np.zeros((len(pos), 2))
        # Find the indices where the mask is True
        wall_indices = np.where(mask_wall)[0]

        # Only map up to the maximum available data points to prevent the ValueError
        min_size = min(len(wall_indices), len(wall_norms))
        wall_norms_complete[wall_indices[:min_size]] = wall_norms[:min_size]
        wall_norms = wall_norms_complete
    # =====================================================================

    with open(filename, 'w') as f:
        # --- Entête VTK ---
        f.write("# vtk DataFile Version 3.0\n")
        f.write("SPH_RIEMANN_JET\n")
        f.write("ASCII\n")
        f.write("DATASET POLYDATA\n")

        # --- Points (Positions X, Y, Z=0) ---
        f.write(f"POINTS {n} float\n")
        np.savetxt(f, np.hstack([pos[:, :2], zeros]), fmt='%.6f')

        # --- Vertices ---
        f.write(f"\nVERTICES {n} {2*n}\n")
        vertices = np.column_stack([np.ones(n, dtype=int), np.arange(n)])
        np.savetxt(f, vertices, fmt='%d %d')

        # --- Données associées aux points ---
        f.write(f"\nPOINT_DATA {n}\n")

        # --- SCALAIRES ---
        f.write("SCALARS Pressure float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, pres_2d, fmt='%.6f')

        f.write("\nSCALARS Density float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, rho_2d, fmt='%.6f')

        f.write("\nSCALARS Volume float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, vol_2d, fmt='%.6f')

        f.write("\nSCALARS Type int 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, types_2d, fmt='%d')

        f.write("\nSCALARS Neighbors int 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, neighbors_2d, fmt='%d')

        f.write("\nSCALARS Shepard float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, shepard_2d, fmt='%.6f')
        
        f.write("\nSCALARS Masse float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, m_2d, fmt='%.6f') 

        f.write("\nSCALARS S_G float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, S_G_to_write, fmt='%.6f')
            
        f.write("\nSCALARS S_dist float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, S_dist_to_write, fmt='%.6f')

        f.write("\nSCALARS Cond_Number float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, cond_array_2d, fmt='%.6e')

        f.write("\nSCALARS Renorm_Status int 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, renorm_status_2d, fmt='%d')

        f.write("\nSCALARS refinement_MLS int 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, p_ref_2d, fmt='%d')
        
        f.write("\nSCALARS Cond_NumberMLS float 1\nLOOKUP_TABLE default\n")
        np.savetxt(f, cond_MLS_2d, fmt='%.6e')

        # --- VECTEURS (Tous complétés avec Z=0) ---
        f.write("\nVECTORS Velocity float\n")
        np.savetxt(f, np.hstack([vel[:, :2], zeros]), fmt='%.6f')
            
        f.write("\nVECTORS VelocityMeshSPH float\n")
        np.savetxt(f, np.hstack([v_mesh[:, :2], zeros]), fmt='%.6f')
            
        f.write("\nVECTORS WallNormals float\n")
        np.savetxt(f, np.hstack([wall_norms[:, :2], zeros]), fmt='%.6f') 
            
        f.write("\nVECTORS grad_S_G float\n")
        np.savetxt(f, np.hstack([grad_S_G[:, :2], zeros]), fmt='%.6f')
            
        f.write("\nVECTORS grad_S_dist float\n")
        np.savetxt(f, np.hstack([grad_S_dist[:, :2], zeros]), fmt='%.6f')

        f.write("\nVECTORS gradP float\n")
        np.savetxt(f, np.hstack([grad_p[:, :2], zeros]), fmt='%.6f')
        
        f.write("\nVECTORS gradVElx float\n")
        np.savetxt(f, np.hstack([grad_vel[:, 0, :2], zeros]), fmt='%.6f') 
        
        f.write("\nVECTORS gradVEly float\n")
        np.savetxt(f, np.hstack([grad_vel[:, 1, :2], zeros]), fmt='%.6f')  
        
        f.write("\nVECTORS gradRHO float\n")
        np.savetxt(f, np.hstack([grad_rho[:, :2], zeros]), fmt='%.6f')


            
def generate_funnel_wall(dx, n_layers):
    pos_wall = []
    
    # On génère chaque couche indépendamment en appliquant l'offset directement sur le rayon
    for layer in range(n_layers):
        offset = layer * dx
        y = 0.0
        
        while y <= H_funnel:
            r = get_funnel_radius(y)
            drdy = get_funnel_derivative(y)
            
            # L'offset élargit le rayon extérieur
            xw = r + offset
            yw = y
            
            pos_wall.append([xw, yw])
            pos_wall.append([-xw, yw])
            
            # Pas d'intégration curviligne basé sur la couche 0 pour garder la régularité y
            dy = dx / np.sqrt(1 + drdy**2)
            y += dy
            
    return np.array(pos_wall)


def compute_wall_normals(pos_wall):
    normals = np.zeros_like(pos_wall)
    for i,(x,y) in enumerate(pos_wall):
        # projection radiale vers la surface
        r_surface = get_funnel_radius(y)
        # dérivée vraie locale
        drdy = get_funnel_derivative(y)
        t = np.array([drdy,1.0])
        t /= np.linalg.norm(t)
        n = np.array([t[1],-t[0]])
        if x < 0:
            n[0] *= -1
        normals[i] = n
    return -normals
            
def apply_density_filter(pos, m, rho, vol, types, h, tree, shepard_coeff):
    """
    Réinitialisation du champ de densité (Filtre de Shepard d'ordre 0)
    pour stabiliser la pression et corriger la troncature en surface libre.
    """       
    n_p = len(pos)
    new_rho = np.copy(rho)
    
    # On ne filtre que les particules fluides
    mask_f = (types == 0) # FLUID
    
    for i in np.where(mask_f)[0]:
        idx = tree.query_ball_point(pos[i], search_radius)
        if len(idx) == 0: continue
        if Periodic:
            rij = periodic_rij(pos[i], pos[idx], testCase, L)
        else :
            rij = pos[i] - pos[idx]
        
        r = np.linalg.norm(rij, axis=1)
        W = kernel_cubic(r, h)
        
        # Formule de Shepard pour la densité (numérateur local)
        num = np.sum(rho[idx] * vol[idx] * W)
                
        # CRITIQUE : Utiliser l'indice [i] pour extraire le scalaire de la particule courante
        den_local = shepard_coeff[i]
        
        if den_local > 2.2e-16:
            rho_shepard = num / den_local
            
            # Gestion de la surface libre sans explosion ni recroquevillage
            if den_local < 0.7:
                new_rho[i] = min(rho_shepard, rho0)
            else:
                new_rho[i] = rho_shepard

    return new_rho


  # La boucle sur les paires dans compute_derivatives est entièrement Python. C'est le goulot principal. La vectorisation complète nécessite de restructurer les flux (voir Vila 1999 implémentation vectorisée). Étape intermédiaire : Numba @njit sur compute_fluid_fluid_interaction réduit le temps de 20–50×.
#Fix prioritaire : @numba.njit(parallel=True) sur la boucle de paires. Ou port vers Cython/C extension. Référence : Vacondio et al. (2021) review des implémentations hautes perf SPH.
def compute_fluid_fluid_interaction(i, j, pos, vel, v_mesh, rho, pres, vol, m, 
                                    shepard_coeff, accel, drho, 
                                    h, rho0, c0, mu, ViscoONOFF,grad_p,grad_rho,grad_vel ):
    """
    Calcule l'interaction entre deux particules de fluide (i, j) 
    selon le formalisme SPH-ALE avec solveur de Riemann.
    """
    if Periodic:
            rij = periodic_rij(pos[i], pos[j],testCase, L)
    else :
            rij = pos[i]-pos[j]# vector from j to i
        
    r = np.linalg.norm(rij)
    
    if r < 2.2e-16 or r > search_radius:
        return

    # 1. Géométrie de l'interface
    grad_w, grad_w_mag = kernel_grad(rij, r, h)  #vector from i to j              
    n_ij = rij / r
    
    
    
    # 2. Formalisme ALE : Vitesse de transport (Vitesse de l'interface)
    # On utilise la vitesse vmesh (déplacement des particules) pour définir l'interface
    v_interface = 0.5 * np.dot(v_mesh[i] + v_mesh[j], n_ij)
    
    # 3. Solveur de Riemann (États de surface)
    # n_ij pointe de j vers i, donc j est l'état Gauche (L) et i l'état Droit (R)
    uL   = np.dot(vel[j], n_ij) 
    uR   = np.dot(vel[i], n_ij)
    rhoL = rho[j]
    rhoR = rho[i]
    pL   = pres[j]
    pR   = pres[i]
    velR = vel[i].copy()  # Init sécurité
    velL = vel[j].copy()  # Init sécurité
    
    if mode_p_refinement == 1:
        dist = np.linalg.norm(rij)
        mid_dist = 0.5 * dist    
        # 1. On calcule la pente "grossière" entre les deux particules
        slope_p_ij   = (pres[i] - pres[j]) /dist  
        slope_rho_ij = (rho[i]  - rho[j])  /dist

        # 2. On récupère la projection des gradients MLS sur l'axe (i-j)
        # Ces gradients ont été calculés par ta fonction compute_mls_gradients
        g_pj   = np.dot(grad_p[j], n_ij)
        g_pi   = np.dot(grad_p[i], n_ij)
        
        g_rhoj = np.dot(grad_rho[j], n_ij)
        g_rhoi = np.dot(grad_rho[i], n_ij)

        # 3. On LIMITE les pentes avec 
        # On compare la pente au point avec la pente entre les deux points
        limited_g_pj   = van_albada(g_pj, slope_p_ij)
        limited_g_pi   = van_albada(g_pi, -slope_p_ij)
        
        limited_g_rhoj = van_albada(g_rhoj, slope_rho_ij)
        limited_g_rhoi = van_albada(g_rhoi, -slope_rho_ij)
        # 4. RECONSTRUCTION des états à l'interface (au milieu 0.5*dist)
        # On part de j pour aller vers le milieu (sens -n_ij)
        pL   = pres[j] + limited_g_pj   * mid_dist   # ← était −
        pR   = pres[i] - limited_g_pi   * mid_dist   # ← était +
        rhoL = rho[j]  + limited_g_rhoj * mid_dist   # ← était −
        rhoR = rho[i]  - limited_g_rhoi * mid_dist   # ← était +
        
        # --- RECONSTRUCTION DE LA VITESSE (ORDRE 2 LIMITE) ---
        
        # 1. Pentes grossières pour chaque composante (x, y)
        slope_vx_ij = (vel[i, 0] - vel[j, 0]) / dist   # ← était (vel[j,0]-vel[i,0])
        slope_vy_ij = (vel[i, 1] - vel[j, 1]) / dist

        # 2. Projections des gradients MLS de la vitesse sur l'axe ij
        # Rappel : grad_vel[j] est de forme (2, 2) -> [[dvx/dx, dvx/dy], [dvy/dx, dvy/dy]]
        g_vx_j = np.dot(grad_vel[j, 0, :], n_ij)
        g_vx_i = np.dot(grad_vel[i, 0, :], n_ij)
        g_vy_j = np.dot(grad_vel[j, 1, :], n_ij)
        g_vy_i = np.dot(grad_vel[i, 1, :], n_ij)

        # 3. Application du limiteur  sur les pentes de vitesse
        limited_g_vx_j = van_albada(g_vx_j, slope_vx_ij)
        limited_g_vx_i = van_albada(g_vx_i, -slope_vx_ij)
        limited_g_vy_j = van_albada(g_vy_j, slope_vy_ij)
        limited_g_vy_i = van_albada(g_vy_i, -slope_vy_ij)

        # 4. Reconstruction des vecteurs vitesse aux interfaces
        velL = np.array([
        vel[j, 0] + limited_g_vx_j * mid_dist,   # ← était −
        vel[j, 1] + limited_g_vy_j * mid_dist,
        ])
        velR = np.array([
            vel[i, 0] - limited_g_vx_i * mid_dist,   # ← était +
            vel[i, 1] - limited_g_vy_i * mid_dist,
        ])

        # 5. Projection finale pour le solveur de Riemann (scalaires uL, uR)
        uL = np.dot(velL, n_ij)
        uR = np.dot(velR, n_ij)
    if mode_p_refinement == 2:
        dist = np.linalg.norm(rij)
        mid_dist = 0.5 * dist    
        # 1. On calcule la pente "grossière" entre les deux particules
        slope_p_ij   = (pres[i] - pres[j]) /dist  
        slope_rho_ij = (rho[i]  - rho[j])  /dist

        # 1. Calcul des variations selon Taylor (incluant Hessienne)
        disp_j = -0.5 * rij  # Vecteur j vers milieu
        disp_i =  0.5 * rij  # Vecteur i vers milieu
        
        # Valeurs extrapolées au milieu (interface)
        pL = reconstruct_order2(grad_p[j], pres[j], disp_j)
        pR = reconstruct_order2(grad_p[i], pres[i], disp_i)
        
        # 2. Pentes effectives (avec courbure)
        # Au lieu du simple gradient, on utilise la pente réelle entre le point et l'interface
        delta_p_j = (pL - pres[j]) / mid_dist
        delta_p_i = (pR - pres[i]) / mid_dist
        
        # 3. Pente géométrique (pente grossière)
        slope_p_ij = (pres[i] - pres[j]) / dist
        
        # 4. Application du limiteur Van Albada
        # On limite la variation locale par rapport à la pente moyenne entre les deux points
        limited_g_pj = van_albada(delta_p_j, slope_p_ij)
        limited_g_pi = van_albada(delta_p_i, -slope_p_ij)
        
        # Idem pour rho
        rhoL = reconstruct_order2(grad_rho[j], rho[j], disp_j)
        rhoR = reconstruct_order2(grad_rho[i], rho[i], disp_i)
        
        delta_rho_j = (rhoL - rho[j]) / mid_dist
        delta_rho_i = (rhoR - rho[i]) / mid_dist
        slope_rho_ij = (rho[i] - rho[j]) / dist
        
        limited_g_rhoj = van_albada(delta_rho_j, slope_rho_ij)
        limited_g_rhoi = van_albada(delta_rho_i, -slope_rho_ij)
        
        
        # 4. RECONSTRUCTION des états à l'interface (au milieu 0.5*dist)
        # On part de j pour aller vers le milieu (sens -n_ij)
        pL   = pres[j] + limited_g_pj   * mid_dist   # ← était −
        pR   = pres[i] - limited_g_pi   * mid_dist   # ← était +
        rhoL = rho[j]  + limited_g_rhoj * mid_dist   # ← était −
        rhoR = rho[i]  - limited_g_rhoi * mid_dist   # ← était +
        
        # --- RECONSTRUCTION DE LA VITESSE (ORDRE 3 LIMITE) ---
      
      
        
        # 1. Pentes grossières
        slope_vx_ij = (vel[i, 0] - vel[j, 0]) / dist
        slope_vy_ij = (vel[i, 1] - vel[j, 1]) / dist
        
        # 2. Reconstruction à l'interface via Taylor (incluant Hessiennes)
        # On utilise tes nouveaux tableaux grad_velx et grad_vely de forme (N, 5)
        vxL = reconstruct_order2(grad_vel[j, 0, :], vel[j, 0], disp_j)
        vxR = reconstruct_order2(grad_vel[i, 0, :], vel[i, 0], disp_i)
        vyL = reconstruct_order2(grad_vel[j, 1, :], vel[j, 1], disp_j)
        vyR = reconstruct_order2(grad_vel[i, 1, :], vel[i, 1], disp_i)
        
        # 3. Calcul des pentes effectives (gradients limités par la courbure)
        delta_vx_j = (vxL - vel[j, 0]) / mid_dist
        delta_vx_i = (vxR - vel[i, 0]) / mid_dist
        delta_vy_j = (vyL - vel[j, 1]) / mid_dist
        delta_vy_i = (vyR - vel[i, 1]) / mid_dist
        
        # 4. Application du limiteur Van Albada
        limited_g_vx_j = van_albada(delta_vx_j, slope_vx_ij)
        limited_g_vx_i = van_albada(delta_vx_i, -slope_vx_ij)
        limited_g_vy_j = van_albada(delta_vy_j, slope_vy_ij)
        limited_g_vy_i = van_albada(delta_vy_i, -slope_vy_ij)
        
        # 5. Reconstruction finale des vecteurs vitesse
        velL = np.array([
            vel[j, 0] + limited_g_vx_j * mid_dist,
            vel[j, 1] + limited_g_vy_j * mid_dist,
        ])
        velR = np.array([
            vel[i, 0] - limited_g_vx_i * mid_dist,
            vel[i, 1] - limited_g_vy_i * mid_dist,
        ])

        # 5. Projection finale pour le solveur de Riemann (scalaires uL, uR)
        uL = np.dot(velL, n_ij)
        uR = np.dot(velR, n_ij)
        
        
        
        #if(mesh_mode == 'lagrangian') : v_interface = 0.5 * (uL+uR)
    
    # Appel au solveur de Riemann (attention à bien passer j en premier, puis i)
    u_star, p_star = riemann_solver(rhoL, uL, pL, rhoR, uR, pR, c0, P_b, False)
    
    # Vitesse relative (Vitesse matérielle Riemann - Vitesse de transport ALE)
    u_relative = u_star - v_interface 
    
    # ===== RECONSTRUCTION VEL_STAR (COMPOSANTE TANGENTIELLE) =====
    # Vitesse complète = vitesse normale du Riemann + composante tangentielle de R
    vel_star =  0.5 * (velL + velR) + (u_star - 0.5 * (uL + uR)) * n_ij
       
    
    if shepard_coeff[i] < 0.75 or shepard_coeff[j] < 0.75:
        p_star = np.clip(p_star, 0.0 , 2*rho0*c0**2)
    
    rho_star = get_density(p_star)        
    #rho_star = 0.5*(rho[j]+rho[i])
    # 4. Calcul du Vecteur Surface (A_vec) corrigé par Vila
    if RenormON:
        # Moyenne arithmétique des matrices de renormalisation
        L_ij = 0.5 * (L_matrices[i] + L_matrices[j])        
        # On applique la matrice au gradient original
        grad_w_corr = np.dot(L_ij, grad_w)
        A_vec = vol[i] * vol[j] * grad_w_corr
    elif ShepardCorr:
        ci = max(shepard_coeff[i], 0.4)
        cj = max(shepard_coeff[j], 0.4)
        A_vec = (vol[i]/ci) * (vol[j]/cj) * grad_w
    else:
        A_vec = vol[i] * vol[j] * grad_w

    A_mag = abs(np.dot(A_vec, n_ij))
                             
    # 5. Flux de Masse (Équation de continuité)
    # mass_flux = rho * (u_matériel - v_transport) * Surface
    mass_flux = rho_star * u_relative * A_mag           
    
    drho[i] += 2.0 * mass_flux / vol[i]
    drho[j] -= 2.0 * mass_flux / vol[j]

    # 6. Flux de Quantité de Mouvement (Équation de Momentum)
    # F_mom = (Flux de masse * Vitesse de transport) + Force de pression
    F_mom = (rho_star * vel_star * u_relative + (p_star)* n_ij) * A_mag  #from j to i
    
    accel[i] += 2.0 * F_mom / m[i]  
    accel[j] -= 2.0 * F_mom / m[j]          

    # 7. Viscosité Physique (Morris et al.)
    # ----------------------------------------------------------------
    # Viscosité Morris et al. (1997) — CORRECT
    # f_ij = x_ij · ∇_i W_ij / (r² + η²)  doit être < 0
    # x_ij = pos[i]-pos[j] = rij  ;  ∇_i W = grad_w
    # dot(rij, grad_w) = (dW/dr)·r < 0  ✓
    # ----------------------------------------------------------------
    if ViscoONOFF:
        vij = vel[i] - vel[j]  # v_ij = v_i - v_j
        # --- CORRECTION DE CONSISTANCE ---
        # On utilise le même gradient que celui qui a servi à calculer A_vec
        # Si RenormON, on utilise grad_w_corr, sinon grad_w
        if RenormON:
            # On récupère le gradient renormalisé calculé plus haut
            target_grad_w = grad_w_corr 
        elif ShepardCorr:
            # Si Shepard, le gradient effectif est multiplié par les facteurs 1/C
            ci = max(shepard_coeff[i], 0.4)
            cj = max(shepard_coeff[j], 0.4)
            target_grad_w = grad_w / (ci * cj)
        else:
            target_grad_w = grad_w

        # Le produit scalaire doit utiliser le gradient "vu" par les autres flux
        rij_dot_gradW = np.dot(rij, target_grad_w)

        # η² évite la singularité (0.01 * h²)
        dist_sq_eps = r**2 + 0.01 * h**2

        # Terme de Morris : (mu_i + mu_j) / (rho_i * rho_j)
        # Note : mu ici est la viscosité dynamique (Pa.s)
        visco_term = (2.0* mu) / (rho[i] * rho[j])

        # Calcul de l'accélération spécifique (force par unité de masse)
        # On multiplie par m[j] pour l'accélération de i
        f_visc_base = visco_term * (rij_dot_gradW / dist_sq_eps) * vij

        accel[i] += m[j] * f_visc_base
        accel[j] -= m[i] * f_visc_base  # antisymétrie
    
    # Paramètres de Chamberlain pour la phase de relaxation
    alpha_relax = 2.0
    beta_relax = 4.0
    eta_sq = 0.01 * (h ** 2)
    if Relax :
        # Vecteurs géométriques
        x_ij = pos[i] - pos[j]
        r_sq = np.dot(x_ij, x_ij)
        r_ij = np.sqrt(r_sq)        
        # Vitesse relative
        v_ij = vel[i] - vel[j]
        v_dot_x = np.dot(v_ij, x_ij)

        # La viscosité de Monaghan ne s'active que si les particules se rapprochent
        if v_dot_x < 0:
            # Gradient du noyau (supposé pré-calculé ou accessible, ex: dw_ij)
            # dw_ij = kernel_gradient(r_ij, h) * (x_ij / r_ij)

            # Calcul de mu_ij
            mu_ij = (h * v_dot_x) / (r_sq + eta_sq)

            # Densité moyenne
            rho_bar = 0.5 * (rho[i] + rho[j])

            # Terme Pi_ij de Monaghan dopé
            pi_ij = (-alpha_relax * c0 * mu_ij + beta_relax * (mu_ij ** 2)) / rho_bar

            # Force visqueuse conservative (symétrique)
            # Formule SPH standard pour la contribution visqueuse à l'accélération
            # dw_ij est le vecteur gradient du noyau de i par rapport à j
            force_visc = - m[j] * pi_ij * grad_w  

            # Distribution sur les accélérations (Principe d'action-réaction)
            if types[i] == FLUID:
                accel[i] += force_visc
            if types[j] == FLUID:
                accel[j] -= force_visc  
        
        
        

def compute_fluid_wall_interaction(i, j, pos, vel, types, rho, pres, vol, m, 
                                   shepard_coeff, wall_normals_full, accel, drho,
                                   h, mu, gamma, ViscoONOFF, FLUID):
    """
    Calcule l'interaction entre une particule de fluide et une particule de mur
    en utilisant un solveur de Riemann (Miroir No-Slip) et le formalisme SPH-ALE.
    """  
    # 1. Identification Fluide / Mur
    idx_f, idx_w = (i, j) if types[i] == FLUID else (j, i)   
    rij = pos[idx_w] - pos[idx_f]
    r = np.linalg.norm(rij) 
    if r < 2.2e-16 or r > search_radius:
        return
    # 2. Géométrie et Normale du mur
    n_fw = wall_normals_full[idx_w] #from w to f
    # Sécurité orientation : la normale doit pointer vers le fluide
    if np.dot(n_fw, pos[idx_f] - pos[idx_w]) < 0:
        n_fw = -n_fw       
    grad_w, grad_w_mag_fw = kernel_grad(rij, r, h)   
    # 0. Correction de surface (Shepard)
    # On utilise le coefficient de la particule fluide pour compenser la troncature
    # Vecteur surface efficace
    # 0. Correction de surface
    if RenormON:
        L_fw = 0.5 * (L_matrices[idx_f] + L_matrices[idx_w])
        grad_w_corr = np.dot(L_fw, grad_w)
        A_vec = vol[idx_f] * vol[idx_w] * grad_w_corr
    elif ShepardCorr:
        cf = max(shepard_coeff[idx_f], 0.4)
        cw = max(shepard_coeff[idx_w], 0.4)
        A_vec = (vol[idx_f]/cf) * (vol[idx_w]/cw) * grad_w
    else:
        A_vec = vol[idx_f] * vol[idx_w] * grad_w

    A_mag = abs(np.dot(A_vec, n_fw))  
                
                
                
    # 1. États de Riemann (Miroir No-Slip)   
    u_wall_n = np.dot(vel[idx_w], n_fw)
    uR       = np.dot(vel[idx_f], n_fw)         # Right = Fluide
       
    
    rhoR = rho[idx_f]
    rhoL = rho[idx_w]
    pR   = pres[idx_f]
    pL   = pres[idx_w]
    
    velR = vel[idx_f].copy()  # Init sécurité
    
    if mode_p_refinement == 1:
        # Distance et vecteur normal unitaire de l'interface (du mur vers le fluide)
        dist = r 
        # n_fw est déjà calculée et pointe du mur vers le fluide
        
        # 1. Calcul des pentes "grossières" (Finite Difference entre fluide et mur)
        # Note : On considère le mur (idx_w) comme l'état Gauche (L) 
        # et le fluide (idx_f) comme l'état Droit (R)
        slope_p_fw   = (pres[idx_f] - pres[idx_w]) / dist
        slope_rho_fw = (rho[idx_f] - rho[idx_w]) / dist

        # 2. Récupération des gradients (Attention : souvent nuls pour les murs, 
        # on se repose alors sur le limiteur vers la pente grossière)
        g_pf   = np.dot(grad_p[idx_f], n_fw)
        g_pw   = np.dot(grad_p[idx_w], n_fw)
        g_rhof = np.dot(grad_rho[idx_f], n_fw)
        g_rhow = np.dot(grad_rho[idx_w], n_fw)

        # 3. Limiteur  (stabilise la reconstruction à l'ordre 2)
        # On reconstruit l'état au milieu de la distance (0.5 * dist)
        limited_g_pf   = van_albada(g_pf, slope_p_fw)
        limited_g_pw   = van_albada(g_pw, slope_p_fw)
        limited_g_rhof = van_albada(g_rhof, slope_rho_fw)
        limited_g_rhow = van_albada(g_rhow, slope_rho_fw)

        # 4. RECONSTRUCTION à l'interface
        # État Mur (L) : on avance de 0.5*dist vers le fluide
        pL   = pres[idx_w] + limited_g_pw * (0.5 * dist)
        rhoL = rho[idx_w] + limited_g_rhow * (0.5 * dist)
        
        # État Fluide (R) : on recule de 0.5*dist vers le mur
        pR   = pres[idx_f] - limited_g_pf * (0.5 * dist)
        rhoR = rho[idx_f] - limited_g_rhof * (0.5 * dist)
        
        # --- RECONSTRUCTION DE LA VITESSE (Condition No-Slip Miroir) ---
        
        # Pour le fluide (R)
        slope_vx_fw = (vel[idx_f, 0] - vel[idx_w, 0]) / dist
        slope_vy_fw = (vel[idx_f, 1] - vel[idx_w, 1]) / dist
        
        g_vx_f = np.dot(grad_vel[idx_f, 0, :], n_fw)
        g_vy_f = np.dot(grad_vel[idx_f, 1, :], n_fw)
        
        # On limite le gradient du fluide
        lim_g_vx_f = van_albada(g_vx_f, slope_vx_fw)
        lim_g_vy_f = van_albada(g_vy_f, slope_vy_fw)
        
        # Vitesse reconstruite côté fluide (R)
        velR = np.zeros(2)
        velR[0] = vel[idx_f, 0] - lim_g_vx_f * (0.5 * dist)
        velR[1] = vel[idx_f, 1] - lim_g_vy_f * (0.5 * dist)
        uR = np.dot(velR, n_fw)
        
    if mode_p_refinement == 2:
        dist = r
        mid_dist = 0.5 * dist    
        slope_p_fw   = (pres[idx_f] - pres[idx_w]) / dist
        slope_rho_fw = (rho[idx_f] - rho[idx_w]) / dist

        # rij pointe du fluide (idx_f) vers le mur (idx_w)
        disp_f =  0.5 * rij  # Vecteur de déplacement Fluide -> Milieu
        disp_w = -0.5 * rij  # Vecteur de déplacement Mur -> Milieu
        
        # 1. Extrapolation de Taylor au milieu de l'interface (incluant les Hessiennes)
        pL_taylor = reconstruct_order2(grad_p[idx_w], pres[idx_w], disp_w)
        pR_taylor = reconstruct_order2(grad_p[idx_f], pres[idx_f], disp_f)
        
        rhoL_taylor = reconstruct_order2(grad_rho[idx_w], rho[idx_w], disp_w)
        rhoR_taylor = reconstruct_order2(grad_rho[idx_f], rho[idx_f], disp_f)

        # 2. Calcul des pentes effectives projetées sur la normale n_fw (Mur -> Fluide)
        delta_p_w = (pL_taylor - pres[idx_w]) / mid_dist
        delta_rho_w = (rhoL_taylor - rho[idx_w]) / mid_dist
        
        # Pour le fluide, la pente par rapport à n_fw utilise la convention (Fluide - Interface)
        delta_p_f = (pres[idx_f] - pR_taylor) / mid_dist
        delta_rho_f = (rho[idx_f] - rhoR_taylor) / mid_dist

        # 3. Application du limiteur de pente Van Albada
        limited_g_pw   = van_albada(delta_p_w, slope_p_fw)
        limited_g_pf   = van_albada(delta_p_f, slope_p_fw)
        limited_g_rhow = van_albada(delta_rho_w, slope_rho_fw)
        limited_g_rhof = van_albada(delta_rho_f, slope_rho_fw)

        # 4. Reconstruction finale des variables thermodynamiques à l'interface
        pL   = pres[idx_w] + limited_g_pw * mid_dist
        pR   = pres[idx_f] - limited_g_pf * mid_dist
        rhoL = rho[idx_w] + limited_g_rhow * mid_dist
        rhoR = rho[idx_f] - limited_g_rhof * mid_dist
        
        # 5. Reconstruction d'ordre supérieur de la vitesse côté Fluide (R)
        slope_vx_fw = (vel[idx_f, 0] - vel[idx_w, 0]) / dist
        slope_vy_fw = (vel[idx_f, 1] - vel[idx_w, 1]) / dist

        vxR_taylor = reconstruct_order2(grad_vel[idx_f, 0, :], vel[idx_f, 0], disp_f)
        vyR_taylor = reconstruct_order2(grad_vel[idx_f, 1, :], vel[idx_f, 1], disp_f)

        delta_vx_f = (vel[idx_f, 0] - vxR_taylor) / mid_dist
        delta_vy_f = (vel[idx_f, 1] - vyR_taylor) / mid_dist

        lim_g_vx_f = van_albada(delta_vx_f, slope_vx_fw)
        lim_g_vy_f = van_albada(delta_vy_f, slope_vy_fw)

        velR = np.zeros(2)
        velR[0] = vel[idx_f, 0] - lim_g_vx_f * mid_dist
        velR[1] = vel[idx_f, 1] - lim_g_vy_f * mid_dist
        uR = np.dot(velR, n_fw)   

    uL = 2*u_wall_n - uR                  # Left = Mur (Miroir)    
        
    
    
    
    # 2.Appel au solveur de Riemann (attention à bien passer j en premier, puis i)
    u_star, p_star = riemann_solver(rhoL, uL, pL, rhoR, uR, pR, c0, P_b, True)    
    rho_star = get_density(p_star) 
   
    v_interface = 0.5* np.dot(vel[idx_w]+vel[idx_f], n_fw)    
    
    u_rel_w = u_star - v_interface     
    vel_star =  velR + (u_star - uR) * n_fw  # Vecteur complet 
    
    # 3. Flux de Quantité de Mouvement
    # Momentum = (Pression + Terme advectif ALE) * Surface
    # Note: u_star * u_rel_w est nul pour un mur fixe, seul p_star reste.
    momentum_flux_w = ((p_star* n_fw) + rho_star * vel_star * u_rel_w) * A_mag 
    # 6. Application des Accélérations    
    accel[idx_f] += 2.0 * momentum_flux_w / m[idx_f] 
    # Flux de masse à la paroi (Continuité)
    # rho_star * (u_star - u_wall_n) est la vitesse relative nette à travers l'interface
    mass_flux_w =  rho_star * (u_rel_w) * A_mag
    # Mise à jour de la variation de densité
    drho[idx_f] += 2.0 * mass_flux_w / vol[idx_f]
    
    
    # 7. Viscosité au mur (Condition No-Slip physique)
    # ----------------------------------------------------------------
    # Viscosité Morris — paroi  CORRECTION DU SIGNE
    #
    # Morris : f_fw = x_fw · ∇_f W_fw / (r² + η²)
    #   x_fw  = pos[f] - pos[w] = -rij
    #   ∇_f W = (dW/dr/r)·(pos[f]-pos[w]) = -grad_w
    #   ⟹ x_fw · ∇_f W = (−rij)·(−grad_w) = dot(rij, grad_w) = (dW/dr)·r < 0  ✓
    #
    if ViscoONOFF:
        # Rappel : idx_f = fluide, idx_w = mur
        vfw = vel[idx_f] - vel[idx_w]
        rij_dot_gradW = np.dot(rij, grad_w)
        dist_sq_eps = r**2 + 0.01 * h**2
        # Terme de Morris
        # On utilise souvent la viscosité du fluide pour le mur
        visco_term = ( 2.0*mu) / (rho[idx_f] * rho[idx_w])

        # Accélération du fluide due au mur
        # Facteur 2.0 souvent ajouté en SPH pour les parois pour compenser la troncature
        # (Colagrossi & Landrini 2003) — choix de modélisation documenté
        f_visc_wall =   visco_term * (rij_dot_gradW  / dist_sq_eps) * vfw
        accel[idx_f] += m[idx_w] *f_visc_wall       
        
        
    if (testCase == 'car' and types[idx_w] == CAR) \
    or (testCase == 'bodyF' and types[idx_w] == BODY) \
    or (testCase == 'Mill' and types[idx_w] == WHEEL):  
        accel[idx_w] -= 2.0 * momentum_flux_w / m[idx_w]
          
        if ViscoONOFF :
            accel[idx_w] -= m[idx_f] * f_visc_wall     
        
def compute_shepard_coefficients(pairs, pos, vol, h,
                                 shepard_coeff,
                                 num_neighbors,
                                 kernel_cubic):
    """
    Calcule les coefficients de normalisation de Shepard
    et le nombre de voisins SPH.
    """

    h2 = search_radius

    for i, j in pairs:
        if Periodic:
            rij = periodic_rij(pos[i], pos[j],testCase, L)
        else :
            rij = pos[i]-pos[j]# vector from j to i
        
        r = np.linalg.norm(rij)
        
        # Skip interactions hors support
        if r < 2.2e-16 or r > h2:
            continue

        Wij = kernel_cubic(r, h)

        # --- contribution symétrique ---
        if types[i] == FLUID :
            shepard_coeff[i] += Wij * vol[j]
            
        if types[j] == FLUID:            
            shepard_coeff[j] += Wij * vol[i]

        num_neighbors[i] += 1
        num_neighbors[j] += 1
        
def compute_pressure_wall(pairs, pos, vol, h, kernel_cubic, pres, types, g, rho0, testCase):
    h2 = search_radius
    pres_fluid = pres.copy()
    
    shepard_fluid_wall = np.zeros(len(pos))   # Σ W*V fluide vu du mur
    solid_types = [WALL]
    if testCase == 'Mill':
        solid_types.append(WHEEL)
    if testCase == 'Car' :
        solid_types.append(CAR)
    if testCase =='BodyF' :
        solid_types.append(BODY)
        solid_types.append(FLAP)
    mask_wall = np.isin(types, solid_types)        
    pres[mask_wall] = 0.0
    for i, j in pairs:
        rij = pos[i] - pos[j] if not Periodic else periodic_rij(pos[i], pos[j],testCase, L)
        r = np.linalg.norm(rij)
        if r < 2.2e-16 or r > h2:
            continue
        Wij = kernel_cubic(r, h)
        if mask_wall[i] and types[j] == FLUID :
            # Correction hydrostatique : ramener la pression au niveau de la paroi
            #dy = pos[j, 1] - pos[i, 1]          # positif si fluide au-dessus
            p_extrap = pres_fluid[j] #+ rho0 * g[1] * (-dy)  # g[1] = -g_acc
            pres[i] += Wij * vol[j] * p_extrap
            shepard_fluid_wall[i] += Wij * vol[j]
            
        elif mask_wall[j] and types[i] == FLUID :
            dy = pos[i, 1] - pos[j, 1]
            p_extrap = pres_fluid[i] #+ rho0 * g[1] * (-dy)
            pres[j] += Wij * vol[i] * p_extrap
            shepard_fluid_wall[j] += Wij * vol[i]   
    
    
    pres[mask_wall] /= (shepard_fluid_wall[mask_wall] + 2.2e-16)

        

def compute_derivatives(        
        pos, vel, rho, pres,
        vol, m,
        accel, drho,
        types,
        h, dt,
        shepard_coeff,
        wall_normals,
        v_mesh, tree,pairs,L_matrices, grad_p, grad_vel, grad_rho ):
    
        
    #Compute interactions (except for mesh analysis : testCase TGVmesh)
    if testCase != 'TGVmesh':
        for i, j in pairs:
            if types[i] == FLUID and types[j] == FLUID:

                compute_fluid_fluid_interaction(
                        i, j,
                        pos, vel, v_mesh,
                        rho, pres, vol, m,
                        shepard_coeff,
                        accel, drho,
                        h, rho0, c0, mu, ViscoONOFF,grad_p,grad_rho,grad_vel 
                    )

            elif (types[i] == FLUID and types[j] == WALL) or \
                     (types[i] == WALL and types[j] == FLUID):

                compute_fluid_wall_interaction(
                        i, j,
                        pos, vel, types,
                        rho, pres, vol, m,
                        shepard_coeff,
                        wall_normals,
                        accel,drho,
                        h, mu, gamma, ViscoONOFF,
                        FLUID
                    )
            if testCase =='bodyF':
                if (types[i] == FLUID and types[j] == FLAP) or \
                    (types[j] == FLUID and types[i] == FLAP) or \
                    (types[i] == FLUID and types[j] == BODY) or \
                    (types[j] == FLUID and types[i] == BODY): 
                    compute_fluid_wall_interaction(
                            i, j,
                            pos, vel, types,
                            rho, pres, vol, m,
                            shepard_coeff,
                            wall_normals,
                            accel,drho,
                            h, mu, gamma, ViscoONOFF,
                            FLUID
                            )
                        
            if testCase =='Mill'   :
                if (types[i] == FLUID and types[j] == WHEEL) or (types[j] == FLUID and types[i] == WHEEL) :
                        compute_fluid_wall_interaction(
                            i, j,
                            pos, vel, types,
                            rho, pres, vol, m,
                            shepard_coeff,
                            wall_normals,
                            accel,drho,
                            h, mu, gamma, ViscoONOFF,
                            FLUID
                            )
            if testCase =='car'   :
                if (types[i] == FLUID and types[j] == CAR) or (types[j] == FLUID and types[i] == CAR) :
                        compute_fluid_wall_interaction(
                            i, j,
                            pos, vel, types,
                            rho, pres, vol, m,
                            shepard_coeff,
                            wall_normals,
                            accel,drho,
                            h, mu, gamma, ViscoONOFF,
                            FLUID
                            )

                    # Interaction CAR - WALL (Collision simple pour que la voiture roule sur le quai)
                    # Vérification des types
                is_car_wall = (types[i] == CAR and types[j] == WALL)
                is_wall_car = (types[j] == CAR and types[i] == WALL)

                if is_car_wall or is_wall_car:
                        # Identification de qui est la voiture
                    idx_c, idx_w = (i, j) if types[i] == CAR else (j, i)
            
                        # Appel de la fonction de collision                    
                    
                    apply_spring_collision(idx_c, idx_w, pos, vel, accel, m, R_car, R_wall, k_local, damping_local, wall_normals)
    
    return drho, accel



def update_rigid_body_physics(
    testCase, pos, vel, accel, m, types, g, dt, t,
    # Paramètres spécifiques au cas 'bodyF'
    body_vel=None, body_omega=None, body_theta=None, body_cm=None,
    rel_pos0=None, nrm_body0_global=None, mass_body_2D=None, I_2D=None,
    idx_body_start=None, n_body=None,
    # Paramètres spécifiques aux cas mobiles 'car' et 'Mill'
    CAR=None, car_mass=None, car_I=None, car_vel=None, car_omega=None, car_theta=None,
    wheel_centerX=0.8, wheel_centerY=0.4,
    # Tableau des normales à modifier in-place pour tous les cas
    wall_normals=None 
):
    """
    Met à jour la cinématique et la dynamique des objets rigides (bodyF, car ou Mill).
    Les modifications sur 'pos', 'vel' et 'wall_normals' sont faites in-place.
    Retourne un dictionnaire contenant les variables cinématiques actualisées.
    """
    
    updates = {}

    # ------------------------------------------------------------------
    # Cas 1 : BODYF (Corps flottant 3 DDL via RK2)
    # ------------------------------------------------------------------
    if testCase == 'bodyF':
        accel_body_only = accel[idx_body_start:idx_body_start + n_body].copy()
        body_vel, body_omega, body_theta, body_cm = update_floating_body_rk2(
            pos, vel, accel_body_only, m, types,
            body_vel, body_omega, body_theta, body_cm,
            rel_pos0, nrm_body0_global, mass_body_2D, I_2D, g,
            wall_normals, idx_body_start, n_body, dt
        )
        updates['body_vel'] = body_vel
        updates['body_omega'] = body_omega
        updates['body_theta'] = body_theta
        updates['body_cm'] = body_cm

    # ------------------------------------------------------------------
    # Cas 2 : CAR (Mouvement rigide de la Voiture)
    # ------------------------------------------------------------------
    elif testCase == 'car':
        mask_car = (types == CAR)
        if np.any(mask_car):
            car_center = np.mean(pos[mask_car], axis=0)
            rel_pos = pos[mask_car] - car_center

            vx = carvelocityInlet(car_center[0], t)[0]
            car_direction = np.array([np.cos(car_theta), np.sin(car_theta)])
            car_vel = car_direction * vx

            forces_indiv = m[mask_car, None] * (accel[mask_car] + g)
            total_torque_car = np.sum(rel_pos[:, 0] * forces_indiv[:, 1] - rel_pos[:, 1] * forces_indiv[:, 0])

            alpha_rigid = total_torque_car / car_I
            car_omega = car_omega + alpha_rigid * dt
            car_theta = car_theta + car_omega * dt

            d_theta = car_omega * dt
            cos_t, sin_t = np.cos(d_theta), np.sin(d_theta)
            rot_mat = np.array([[cos_t, -sin_t], 
                                [sin_t,  cos_t]])
            
            new_rel_pos = np.dot(rel_pos, rot_mat.T)

            pos[mask_car] = car_center + car_vel * dt + new_rel_pos
            vel[mask_car] = car_vel + np.vstack([-car_omega * new_rel_pos[:, 1], 
                                                  car_omega * new_rel_pos[:, 0]]).T
            
            # Rotation in-place des normales de surface de la voiture
            # Rotation des normales de surface de la voiture (avec capture du retour)
            if wall_normals is not None:
                wall_normals = rotate_wall_normals(mask_car, wall_normals, d_theta)
                
            updates['car_vel'] = car_vel
            updates['car_omega'] = car_omega
            updates['car_theta'] = car_theta

    # ------------------------------------------------------------------
    # Cas 3 : MILL (Rotation libre autour d'un axe fixe)
    # ------------------------------------------------------------------
    # ------------------------------------------------------------------
    # Cas 3 : MILL (Rotation libre autour d'un axe fixe)
    # ------------------------------------------------------------------
    elif testCase == 'Mill':
        mask_wheel = (types == WHEEL) 
        
        if np.any(mask_wheel):
            wheel_center = np.array([wheel_centerX, wheel_centerY])
            rel_pos = pos[mask_wheel] - wheel_center

            forces_indiv = m[mask_wheel, None] * (accel[mask_wheel] + g)
            total_torque_wheel = np.sum(rel_pos[:, 0] * forces_indiv[:, 1] - rel_pos[:, 1] * forces_indiv[:, 0])

            wheel_alpha = total_torque_wheel / car_I 
            car_omega = car_omega + wheel_alpha * dt
            car_theta = car_theta + car_omega * dt

            # AFFICHAGE DE DÉBOGAGE (Fréquence contrôlable si besoin, ici à chaque pas)
            #print(f"[DEBUG Mill] t={t:.4f}s | Torque={total_torque_wheel:+.4e} | Omega={car_omega:+.4f} rad/s | Theta={car_theta:+.4f} rad ({np.rad2deg(car_theta):+.2f}°)")

            d_theta = car_omega * dt
            cos_t, sin_t = np.cos(d_theta), np.sin(d_theta)
            rot_mat = np.array([[cos_t, -sin_t], 
                                [sin_t,  cos_t]])
            
            new_rel_pos = np.dot(rel_pos, rot_mat.T)

            pos[mask_wheel] = wheel_center + new_rel_pos
            vel[mask_wheel] = np.vstack([-car_omega * new_rel_pos[:, 1], 
                                          car_omega * new_rel_pos[:, 0]]).T
            
            # Rotation in-place des normales de surface de la roue
            # Rotation des normales de surface de la roue (avec capture du retour)
            if wall_normals is not None:
                wall_normals = rotate_wall_normals(mask_wheel, wall_normals, d_theta)
                
            updates['wheel_alpha'] = wheel_alpha
            updates['wheel_omega'] = car_omega
            updates['wheel_theta'] = car_theta

    return updates, wall_normals

def rotate_wall_normals(mask, wall_normals, d_theta):
    """
    Rotation des normales de surface avec synchronisation stricte de la dimension.
    """
    if d_theta == 0.0 or wall_normals is None:
        return wall_normals
        
    n_required = len(mask)
    n_current = wall_normals.shape[0]
    
    # Synchronisation de la taille du tableau des normales avec celle du masque
    if n_current != n_required:
        extended_normals = np.zeros((n_required, 2))
        limit = min(n_current, n_required)
        extended_normals[:limit, :] = wall_normals[:limit, :]
        wall_normals = extended_normals
    
    # Construction de la matrice de rotation
    cos_t, sin_t = np.cos(d_theta), np.sin(d_theta)
    rot_mat = np.array([[cos_t, -sin_t], 
                        [sin_t,  cos_t]])
    
    # Application de la rotation sur le masque
    wall_normals[mask] = np.dot(wall_normals[mask], rot_mat.T)
    
    return wall_normals
    
    
def get_car_theta(x):
    """
    Retourne l'angle analytique de la route (en radians) 
    en fonction de la position X globale de la voiture.
    """
    # Segment 1 : Plateau gauche horizontal
    if x <= L_plateau:
        return 0.0
    
    # Segment 2 : Pente descendante (angle négatif car incliné vers le bas)
    elif x <= L_plateau + L_slope:
        return -theta
    
    # Segment 3 : Plateau fond du bac horizontal
    elif x <= L_plateau + L_slope + L_bottom:
        return 0.0
    
    # Segment 4 : Remontée symétrique (angle positif)
    elif x <= L_plateau + L_slope + L_bottom + L_rise:
        return theta
    
    # Segment 5 : Plateau final horizontal
    else:
        return 0.0
    
    
def update_jet_buffers(
        testCase,
        pos, vel, types,
        rho, pres, vol, m,
        num_neighbors, shepard_coeff,
        bx, dx,
        y_inlet_top, y_inlet_bottom, y_inlet_middle,
        v_jet, rho0,
        BUFFER_TOP, BUFFER_BOTTOM,BUFFER_MIDDLE, FLUID,
        DOUBLE_JET=True,
        TRIPLE_JET= True,
        randompos=False,
        add_position_noise=None,
        theta_deg1=15.0,
        decalbx=0.0,   # Ajouté pour cohérence géométrique
        decalbx1=0.0   # Ajouté pour cohérence géométrique
    ):
    """
    Gestion des buffers d'injection pour le test 'jet'.
    - conversion buffer -> fluide
    - régénération automatique des couches buffer avec inclinaison géométrique 
      et cinématique d'angle theta_deg.
    """

    if testCase not in ['jet', 'Mill']:
        return pos, vel, types, rho, pres, vol, m, num_neighbors, shepard_coeff

    # Calcul des composantes de vitesse et géométriques inclinées
    theta_rad = np.radians(theta_deg1)
    
    # TOP JET : descend (-Y) et s'incline vers l'intérieur (+X)
    v_top_incline = [-v_jet * np.sin(theta_rad), -v_jet * np.cos(theta_rad)]
    
    # BOTTOM JET : monte (+Y) et s'incline vers l'intérieur (+X)
    v_bot_incline = [-v_jet * np.sin(theta_rad), v_jet * np.cos(theta_rad)]
    
    # MIDDLE JET : monte (+Y) et s'incline vers l'intérieur (+X)
    v_mid_incline = [v_jet * np.sin(theta_rad), -v_jet * np.cos(theta_rad)]

    # =====================================================
    # 1. Conversion buffer -> fluide
    # =====================================================
    mask_top = (types == BUFFER_TOP)
    mask_bot = (types == BUFFER_BOTTOM)
    mask_mid = (types == BUFFER_MIDDLE)

    to_fluid_top = mask_top & (pos[:, 1] < y_inlet_top)
    to_fluid_bot = mask_bot & (pos[:, 1] > y_inlet_bottom)
    to_fluid_mid = mask_mid & (pos[:, 1] < y_inlet_middle)

    types[to_fluid_top | to_fluid_bot | to_fluid_mid] = FLUID

    # =====================================================
    # 2. Régénération buffers
    # =====================================================
    to_add_pos = []
    to_add_vel = []
    to_add_types = []

    tol = 1e-7

    # ---------- TOP JET ----------
    if np.any(types == BUFFER_TOP):

        y_max_top = np.max(pos[types == BUFFER_TOP, 1])

        if y_max_top < (y_inlet_top + 3 * dx + tol):

            y_new = y_max_top + dx
            
            # Décalage en X pour suivre la pente du jet descendant
            dx_pente = (y_new - y_inlet_top) * np.tan(theta_rad)
            bx_incline = bx + dx_pente
            
            new_layer = np.vstack([bx_incline, np.full_like(bx_incline, y_new)]).T

            if randompos and add_position_noise is not None:
                new_layer = add_position_noise(new_layer, 0.15 * dx)

            to_add_pos.append(new_layer)
            to_add_vel.append(np.tile(v_top_incline, (len(bx), 1)))
            to_add_types.append(np.full(len(bx), BUFFER_TOP))

    # ---------- BOTTOM JET ----------
    if DOUBLE_JET and np.any(types == BUFFER_BOTTOM):

        y_min_bot = np.min(pos[types == BUFFER_BOTTOM, 1])

        if y_min_bot > (y_inlet_bottom - 3 * dx - tol):

            y_new = y_min_bot - dx
            
            # Décalage en X pour suivre la pente du jet montant
            dx_pente = (y_inlet_bottom - y_new) * np.tan(theta_rad)
            bx_incline = bx + decalbx + dx_pente
            
            new_layer = np.vstack([bx_incline, np.full_like(bx_incline, y_new)]).T

            if randompos and add_position_noise is not None:
                new_layer = add_position_noise(new_layer, 0.15 * dx)

            to_add_pos.append(new_layer)
            to_add_vel.append(np.tile(v_bot_incline, (len(bx), 1)))
            to_add_types.append(np.full(len(bx), BUFFER_BOTTOM))
            
    # ---------- MIDDLE JET (CORRIGÉ) ----------
    if TRIPLE_JET and np.any(types == BUFFER_MIDDLE):
        # On surveille le haut du buffer car le jet descend
        y_max_mid = np.max(pos[types == BUFFER_MIDDLE, 1])

        if y_max_mid < (y_inlet_middle + 3 * dx + tol):
            y_new = y_max_mid + dx  # S'étend vers le haut à l'extérieur pour remplir le vide
            
            # Application de la pente géométrique cohérente avec l'initialisation descendante
            dx_pente = (y_new - y_inlet_middle) * np.tan(theta_rad)
            bx_incline = bx + decalbx1 - dx_pente
            
            new_layer = np.vstack([bx_incline, np.full_like(bx_incline, y_new)]).T
            if randompos and add_position_noise is not None:
                new_layer = add_position_noise(new_layer, 0.15 * dx)
                
            to_add_pos.append(new_layer)
            to_add_vel.append(np.tile(v_mid_incline, (len(bx), 1)))
            to_add_types.append(np.full(len(bx), BUFFER_MIDDLE))

    # =====================================================
    # 3. Ajout des nouvelles particules
    # =====================================================
    if to_add_pos:

        new_p = np.vstack(to_add_pos)
        new_v = np.vstack(to_add_vel)
        new_t = np.concatenate(to_add_types)

        n_add = len(new_p)

        pos = np.vstack([pos, new_p])
        vel = np.vstack([vel, new_v])
        types = np.concatenate([types, new_t])

        rho = np.concatenate([rho, np.full(n_add, rho0)])
        pres = np.concatenate([pres, np.zeros(n_add)])
        vol = np.concatenate([vol, np.full(n_add, dx**2)])
        m = np.concatenate([m, np.full(n_add, rho0 * dx**2)])

        num_neighbors = np.concatenate(
            [num_neighbors, np.zeros(n_add, dtype=int)]
        )

        shepard_coeff = np.concatenate(
            [shepard_coeff, np.zeros(n_add)]
        )

    return (
        pos, vel, types,
        rho, pres, vol, m,
        num_neighbors, shepard_coeff
    )   

def apply_spring_collision(idx_c, idx_w, pos, vel, accel, m, R_car, R_wall, k_local, damping_local, wall_normals):
    """
    Version universelle : s'adapte dynamiquement au quai plat et aux pentes
    sans générer de pichenette aux ruptures de géométrie.
    """
    rij = pos[idx_c] - pos[idx_w]
    r = np.linalg.norm(rij)

    if r < 1e-12: 
        return

    # 1. On récupère la normale théorique du mur à cet endroit
    nij_wall = wall_normals[idx_w]

    # 2. Gestion adaptative de la direction de la force
    if np.all(nij_wall == 0) or abs(nij_wall[1]) > 0.99:
        # Cas du SOL PLAT : La force est strictement verticale (évite l'effet boîte d'œufs)
        nij = np.array([0.0, 1.0])
        penetration = (R_car + R_wall) - rij[1]  # Pénétration purement verticale (y)
    else:
        # Cas de la PENTE : On utilise la vraie normale de la pente
        nij = nij_wall
        # La pénétration est la projection réelle sur cette pente
        penetration = (R_car + R_wall) - np.dot(rij, nij)

    # 3. Application de la force si interaction
    if penetration > 0:
            vrel = vel[idx_c] - vel[idx_w]
            vn = np.dot(vrel, nij)        
            
            f_spring = k_local * penetration
            f_damper = -damping_local * vn       
            
            # --- JOUER SUR LA FORCE (FILTRE) ---
            # Si la force totale de poussée dépasse une valeur critique capable de satelliser la particule
            # (par exemple, 5 fois le poids de la voiture entière ramené à la particule), on la sature.
            force_scalaire = f_spring + f_damper
            
            # Sécurité : la force du sol ne doit JAMAIS être négative (effet ventouse)
            if force_scalaire < 0: 
                force_scalaire = 0.0
                
            # Saturation optionnelle pour lisser les arêtes (évite les pics de force discrets)
            # On limite l'accélération maximale subie par une seule particule à 3 * |g| par exemple
            g_norm = 9.81
            max_accel = 5.0 * g_norm
            max_force = max_accel * m[idx_c]
            
            if force_scalaire > max_force:
                force_scalaire = max_force
                
            total_force = force_scalaire * nij       
            accel[idx_c] += total_force / m[idx_c]




#--------------------------------------------------------------------------------------------------------------------- 
#                       Initialisation des maillages
#---------------------------------------------------------------------------------------------------------------------
if testCase == 'bodyF':
    # ==============================================================================
    # 2.  GÉNÉRATEUR DE HOULE — CLAPET ARTICULÉ (FLAP WAVEMAKER)
    # ==============================================================================
    #
    #  Le clapet tourne autour de son axe inférieur (0, 0).
    #  Son mouvement est un paquet de N_waves composantes spectrales focalisées
    #  au point (x_focus, t_focus), simulant l'expérience de Hadzic et al. (2005).
    #
    #  Si le fichier Flap.dat (fourni par SPHERIC) est disponible, il est chargé
    #  directement via load_flap_dat(). Sinon, la génération analytique est utilisée.

    N_waves   = 20
    omega_min = 1.0       # [rad/s]  (T_max ≈ 6.3 s)
    omega_max = 4.0       # [rad/s]  (T_min ≈ 1.6 s)
    t_focus   = 32.0      # instant de focalisation [s]
    x_focus   = x_body_init   # position de focalisation = centre du corps
    H_focus   = H_body    # hauteur de vague au focus = hauteur du corps = 0.05 m

    omega_n   = np.linspace(omega_min, omega_max, N_waves)

    def solve_dispersion(omega, H, tol=1e-10, max_iter=100):
        """Résout ω² = g k tanh(kH) par Newton-Raphson."""
        k = omega**2 / g_acc   # estimation eau profonde
        for _ in range(max_iter):
            f  =  g_acc * k * np.tanh(k * H) - omega**2
            df =  g_acc * (np.tanh(k * H) + k * H / np.cosh(k * H)**2)
            dk = -f / (df + 1e-20)
            k += dk
            if abs(f) < tol:
                break
        return max(k, 1e-8)

    k_n = np.array([solve_dispersion(w, H_water) for w in omega_n])
    T_n = 2.0 * np.pi / omega_n     # périodes
    L_n = 2.0 * np.pi / k_n         # longueurs d'onde

    def flap_transfer_function(k, H):
        """
        Fonction de transfert du clapet articulé au fond.
        Donne le rapport amplitude_surface / (stroke_clapet / 2).
        Source : Dean & Dalrymple (1991), Eq. 7.5
        """
        kH = k * H
        num = 4.0 * np.sinh(kH)**2
        den = 2.0 * kH + np.sinh(2.0 * kH)
        return np.sqrt(num / den + 1e-20)

    Kf_n = flap_transfer_function(k_n, H_water)

    # Amplitude de chaque composante spectrale (spectre plat)
    a_n = np.full(N_waves, H_focus / (2.0 * N_waves))

    # Phase : focalisation constructive en (x_focus, t_focus)
    phi_n = k_n * x_focus - omega_n * t_focus

    # Amplitude angulaire du clapet pour chaque composante [rad]
    # a = Kf * (amplitude_stroke) => amplitude_stroke = a / Kf
    # Pour le clapet : stroke = θ_max * H_water / 2 => θ_max = 2*stroke / H_water
    theta_amp_n = 2.0 * (a_n / Kf_n) / H_water

    print(f"\nParquet de vagues : {N_waves} composantes, f ∈ [{omega_min/(2*np.pi):.2f}, {omega_max/(2*np.pi):.2f}] Hz")
    print(f"Focalisation : x={x_focus:.2f}m, t={t_focus:.1f}s, H_focus={H_focus*100:.1f}cm")
    print(f"Amplitude angulaire max du clapet : {np.max(theta_amp_n)*180/np.pi:.2f}°")

    def flap_angle(t, ramp_dur=5.0):
        """Angle du clapet θ(t) [rad] avec rampe initiale pour démarrage doux."""
        ramp = min(t / ramp_dur, 1.0)
        return ramp * float(np.sum(theta_amp_n * np.sin(omega_n * t + phi_n)))

    def flap_angular_velocity(t, ramp_dur=5.0):
        if t < ramp_dur:
            ramp  = t / ramp_dur
            ramp_prime = 1.0 / ramp_dur
        else:
            ramp, ramp_prime = 1.0, 0.0
        f      = float(np.sum(theta_amp_n * np.sin(omega_n * t + phi_n)))
        f_prime = float(np.sum(theta_amp_n * omega_n * np.cos(omega_n * t + phi_n)))
        return ramp_prime * f + ramp * f_prime

    def load_flap_dat(filepath):
        """
        Charge le fichier Flap.dat officiel SPHERIC (téléchargeable sur spheric-sph.org).
        Colonnes : Temps [s], Angle [degrés]
        Gère automatiquement les lignes d'en-tête (Time, #, %, lettres...).
        Retourne des fonctions interpolées θ(t) et θ̇(t).
        """
        # --- Détection automatique du nombre de lignes d'en-tête ---
        skiprows = 0
        with open(filepath, 'r') as fh:
            for line in fh:
                stripped = line.strip()
                # Ligne vide ou commençant par un caractère non numérique/signe
                if not stripped:
                    skiprows += 1
                    continue
                first_char = stripped[0]
                if first_char in ('#', '%', '!', '"', "'") or first_char.isalpha():
                    skiprows += 1
                else:
                    break   # première ligne numérique trouvée

        data   = np.loadtxt(filepath, skiprows=skiprows)
        t_dat  = data[:, 0]
        th_raw = data[:, 1]

        
        th_rad = np.deg2rad(th_raw)
        th_rad = th_rad - th_rad[0] # Sécurité pour partir de 0.0
        unit   = "degrés → radians"
        

        thdot = np.gradient(th_rad, t_dat)

        f_theta    = interp1d(t_dat, th_rad, kind='cubic',
                              fill_value=(th_rad[0], th_rad[-1]), bounds_error=False)
        f_thetadot = interp1d(t_dat, thdot,  kind='cubic',
                              fill_value=0.0, bounds_error=False)

        print(f"[OK] Flap.dat chargé : {len(t_dat)} points, "
              f"t ∈ [{t_dat[0]:.2f}, {t_dat[-1]:.2f}]s, "
              f"angle {unit}, "
              f"|θ|_max = {np.max(np.abs(th_rad))*180/np.pi:.2f}°, "
              f"lignes en-tête sautées : {skiprows}")
        return f_theta, f_thetadot

    # Utilisation de Flap.dat si disponible
    _flap_dat_path = os.path.join('.', 'Flap.dat')
    if os.path.exists(_flap_dat_path):
        _f_theta, _f_thetadot = load_flap_dat(_flap_dat_path)
        flap_angle            = lambda t, **kw: float(_f_theta(t))
        flap_angular_velocity = lambda t, **kw: float(_f_thetadot(t))
        print("[INFO] Utilisation de Flap.dat (données SPHERIC officielles)")
    else:
        print("[INFO] Flap.dat non trouvé → paquet de vagues analytique")

    # ==============================================================================
    # 3.  FONCTIONS DU CORPS FLOTTANT (RIGID BODY — 3 DDL)
    # ==============================================================================

    def build_floating_body(x_cm, y_cm, L, H, dx_body):
        """
        Crée une grille régulière de particules pour le corps flottant.
        Retourne : positions, normales de surface, positions relatives au CM.
        """
        nx = max(2, int(L / dx_body))
        ny = max(2, int(H / dx_body))
        dx_eff = L / nx
        dy_eff = H / ny

        xs = np.arange(dx_eff/2, L, dx_eff) - L/2
        ys = np.arange(dy_eff/2, H, dy_eff) - H/2
        Xb, Yb = np.meshgrid(xs, ys)
        rel_pos0 = np.column_stack([Xb.ravel(), Yb.ravel()])  # relative au CM
        pos_b    = rel_pos0 + np.array([x_cm, y_cm])

        # Calcul des normales (détection des particules de surface)
        n_parts = len(pos_b)
        normals = np.zeros((n_parts, 2))
        pts_set = set(map(lambda p: (round(p[0], 8), round(p[1], 8)), rel_pos0))

        for idx in range(n_parts):
            p   = rel_pos0[idx]
            nrm = np.zeros(2)
            for d in np.array([[dx_eff,0],[-dx_eff,0],[0,dy_eff],[0,-dy_eff]]):
                nb  = (round(p[0]+d[0], 8), round(p[1]+d[1], 8))
                if nb not in pts_set:
                    nrm += d
            if np.linalg.norm(nrm) > 2.2e-16:
                # Vérifier que la normale pointe vers l'extérieur
                if np.dot(nrm, p) < 0:
                    nrm = -nrm
                normals[idx] = nrm / np.linalg.norm(nrm)

        print(f"Corps flottant : {n_parts} particules ({nx} × {ny})")
        return pos_b, normals, rel_pos0

    def update_floating_body_rk2(pos, vel, accel_body, m, types,
                                   body_vel, body_omega, body_theta, body_cm,
                                   rel_pos0, normals0, mass_2D, I_2D, g,
                                   wall_normals, idx_body_start, n_body, dt,
                                   step='full'):
        """
        Intègre le mouvement du corps rigide (sway, heave, roll).

        Équations du mouvement :
          m × ẍ_cm = ΣF_fluid_x
          m × ÿ_cm = ΣF_fluid_y + m × g_y        (gravité sur le corps)
          I × θ̈   = Σ(r × F_fluid)              (moment par rapport au CM)

        La pression de flottaison et les forces de rappel ne sont PAS ajoutées
        explicitement : elles découlent naturellement de la pression SPH.

        Paramètres
        ----------
        step : 'full' (Euler), 'predictor' ou 'corrector' (RK2 externe)
        """
        mask_body = (types == BODY)

        # --- Forces SPH exercées sur le corps ---
        # accel[mask_body] contient les accélérations dues au fluide
        F_sph = np.sum(m[mask_body, None] * accel_body, axis=0)   # [N/m]

        # Gravité
        F_gravity = np.array([0.0, -mass_2D * g_acc])             # [N/m]

        F_total = F_sph + F_gravity

        # --- Moment angulaire par rapport au CM courant ---
        rel_cur = pos[mask_body] - body_cm         # positions relatives actuelles
        f_indiv = m[mask_body, None] * accel_body  # forces individuelles SPH
        # Produit vectoriel 2D : M = r_x * F_y - r_y * F_x
        torque = np.sum(rel_cur[:, 0] * f_indiv[:, 1] - rel_cur[:, 1] * f_indiv[:, 0])

        # --- Accélérations linéaire et angulaire ---
        a_lin   = F_total / mass_2D                # [m/s²]
        alpha   = torque   / I_2D                  # [rad/s²]

        # --- Intégration temporelle ---
        new_vel   = body_vel   + a_lin  * dt
        new_omega = body_omega + alpha  * dt
        new_theta = body_theta + new_omega * dt
        new_cm    = body_cm    + new_vel   * dt

        # --- Mise à jour des positions et vitesses des particules du corps ---
        ct, st = np.cos(new_theta), np.sin(new_theta)
        R = np.array([[ct, -st], [st, ct]])

        # Positions : pos_i = CM_new + R(θ_new) @ rel_pos0_i
        new_rel = rel_pos0 @ R.T                         # (n_body, 2)
        pos[mask_body]    = new_cm + new_rel

        # Vitesses : v_i = v_CM + ω × r_rel_i
        vel[mask_body, 0] = new_vel[0] - new_omega * new_rel[:, 1]
        vel[mask_body, 1] = new_vel[1] + new_omega * new_rel[:, 0]

        # Normales : rotation des normales initiales
        new_norms = normals0 @ R.T
        wall_normals[idx_body_start:idx_body_start + n_body] = new_norms

        return new_vel, new_omega, new_theta, new_cm


    # ==============================================================================
    # 4.  FONCTIONS DU CLAPET (FLAP WAVEMAKER)
    # ==============================================================================

    def build_flap_particles(H_water, dx, n_layers=3):
        """
        Crée les particules du clapet wavemaker.
        Le clapet est initialement vertical, axe en (0, 0).
        n_layers couches de particules dans l'épaisseur du clapet.
        Retourne : positions, normales initiales, hauteurs y0 de chaque particule.
        """
        y_vals = np.arange(dx/2, H_ramp_end + dx/2, dx)
        n_pts  = len(y_vals)

        pos_f, nrm_f, y0_f = [], [], []
        for l in range(n_layers):
            offset = -(l + 0.5) * dx      # particules à gauche de x=0
            for y0 in y_vals:
                pos_f.append([offset, y0])
                nrm_f.append([1.0, 0.0])  # normale initiale vers +x (vers le fluide)
                y0_f.append(y0)           # distance à l'axe de rotation

        return (np.array(pos_f),
                np.array(nrm_f),
                np.array(y0_f))


    def update_flap_particles(pos_all, vel_all, wall_normals,
                              t, idx_flap_start, n_flap, n_layers,
                              y0_arr, dx):
        """
        Met à jour les positions, vitesses et normales des particules du clapet.

        Pour le clapet en rotation autour de (0, 0) avec angle θ et vitesse θ̇ :
          Particule à distance y0 de l'axe, couche l (offset = -(l+0.5)*dx) :

            x(t) =  y0 * sin(θ) - offset * cos(θ)
            y(t) =  y0 * cos(θ) + offset * sin(θ)

            ẋ(t) = (y0 * cos(θ) + offset * sin(θ)) * θ̇
            ẏ(t) = (-y0 * sin(θ) + offset * cos(θ)) * θ̇

          Normale (pointant vers +x dans la configuration initiale) :
            n = [cos(θ), -sin(θ)]
        """
        theta    = flap_angle(t)
        thetadot = flap_angular_velocity(t)
        ct, st   = np.cos(theta), np.sin(theta)

        n_per_layer = n_flap // n_layers

        for l in range(n_layers):
            offset = -(l + 0.5) * dx   # même logique que build_flap_particles
            for j in range(n_per_layer):
                idx = idx_flap_start + l * n_per_layer + j
                y0  = y0_arr[l * n_per_layer + j]

                # --- FORMULES CORRIGÉES ---
                # Si theta=0 : x = offset (négatif), y = y0 (vertical)
                pos_all[idx, 0] = offset * ct + y0 * st
                pos_all[idx, 1] = -offset * st + y0 * ct

                # Vitesse : v = omega x r
                vel_all[idx, 0] = (pos_all[idx, 1]) * thetadot
                vel_all[idx, 1] = (-pos_all[idx, 0]) * thetadot

        # Normale : perpendiculaire au clapet, pointant vers le fluide
        wall_normals[idx_flap_start:idx_flap_start + n_flap, 0] =  ct
        wall_normals[idx_flap_start:idx_flap_start + n_flap, 1] = -st


    # ==============================================================================
    # 5.  MESURE DES SONDES ET COMPARAISON EXPÉRIMENTALE
    # ==============================================================================

    def measure_surface_elevation(pos, types, rho, rho0, x_probe, h_sph):
        """
        Élévation de surface à la sonde x_probe.
        Méthode : interpolation Shepard de ρ sur la colonne, puis détection
        du niveau ρ = ρ0/2 (interface eau/air SPH).
        Fallback : position de la plus haute particule fluide dans |x-x_p| < h.
        """
        mask_f = (types == FLUID)
        band   = mask_f & (np.abs(pos[:, 0] - x_probe) < h_sph)

        if not np.any(band):
            return 0.0

        # Méthode rapide : plus haute particule dans la bande
        y_top = np.max(pos[band, 1])
        return y_top - H_water


    def load_experimental(data_dir='.'):
        """
        Charge les fichiers expérimentaux du Test 12 SPHERIC
        (télécharger SPHERIC_TestCase12.zip sur spheric-sph.org).
        """
        exp = {}
        files = {
            'Elevation.dat': ['t1', 'eta1', 't2', 'eta2'],
            'Motions.dat'  : ['t_h', 'heave', 't_s', 'sway'],
            'Rotation.dat' : ['t_r', 'roll'],
        }
        for fname, keys in files.items():
            fpath = os.path.join(data_dir, fname)
            if os.path.exists(fpath):
                data = np.genfromtxt(fpath, comments='#', skip_header=1, invalid_raise=False)
                for i, k in enumerate(keys):
                    exp[k] = data[:, i]
                print(f"[OK] {fname} chargé ({len(data)} points)")
            else:
                print(f"[INFO] {fname} non trouvé — pas de comparaison pour ce jeu")
        return exp


    def write_probe_ascii(vtk_dir, t_arr, eta1_arr, eta2_arr,
                          heave_arr, sway_arr, roll_arr, cm_arr):
        """Écrit les fichiers de sortie au format SPHERIC."""
        # Élévation de surface
        out = np.column_stack([t_arr, eta1_arr, t_arr, eta2_arr])
        np.savetxt(os.path.join(vtk_dir, 'SPH_Elevation.dat'), out,
                   header='Time(s)  Probe1_eta(m)  Time(s)  Probe2_eta(m)',
                   fmt='%.6f')
        # Mouvements du corps
        out = np.column_stack([t_arr, heave_arr, t_arr, sway_arr])
        np.savetxt(os.path.join(vtk_dir, 'SPH_Motions.dat'), out,
                   header='Time(s)  Heave(m)  Time(s)  Sway(m)',
                   fmt='%.6f')
        # Rotation
        out = np.column_stack([t_arr, roll_arr])
        np.savetxt(os.path.join(vtk_dir, 'SPH_Rotation.dat'), out,
                   header='Time(s)  Roll(degrees)',
                   fmt='%.6f')
        # Trajectoire du centre de masse
        out = np.column_stack([t_arr, cm_arr])
        np.savetxt(os.path.join(vtk_dir, 'SPH_BodyCM.dat'), out,
                   header='Time(s)  CM_x(m)  CM_y(m)',
                   fmt='%.6f')
        print(f"[OK] Fichiers ASCII écrits dans {vtk_dir}/")


    def plot_comparison(vtk_dir, t_arr, eta1, eta2, heave, sway, roll, exp):
        """
        Génère les figures de comparaison SPH vs expérience.
        Les figures sont enregistrées en PNG dans vtk_dir.
        """
        fig, axes = plt.subplots(3, 2, figsize=(14, 10))
        fig.suptitle('SPHERIC Test 12 — SPH-ALE vs Expérience', fontsize=14, fontweight='bold')

        # Sonde 1 — avant le corps
        ax = axes[0, 0]
        ax.plot(t_arr, eta1 * 100, 'b-', lw=1.5, label='SPH-ALE')
        if 't1' in exp:
            ax.plot(exp['t1'], exp['eta1'] * 100, 'k--', lw=1, alpha=0.8, label='Expérience')
        ax.set_xlabel('Temps [s]')
        ax.set_ylabel('η [cm]')
        ax.set_title(f'Élévation surface — Sonde 1 (x={x_probe1:.2f}m)')
        ax.legend(); ax.grid(True, alpha=0.3)

        # Sonde 2 — après le corps
        ax = axes[0, 1]
        ax.plot(t_arr, eta2 * 100, 'r-', lw=1.5, label='SPH-ALE')
        if 't2' in exp:
            ax.plot(exp['t2'], exp['eta2'] * 100, 'k--', lw=1, alpha=0.8, label='Expérience')
        ax.set_xlabel('Temps [s]')
        ax.set_ylabel('η [cm]')
        ax.set_title(f'Élévation surface — Sonde 2 (x={x_probe2:.2f}m)')
        ax.legend(); ax.grid(True, alpha=0.3)

        # Heave
        ax = axes[1, 0]
        ax.plot(t_arr, heave * 100, 'b-', lw=1.5, label='SPH-ALE')
        if 't_h' in exp:
            ax.plot(exp['t_h'], exp['heave'] * 100, 'k--', lw=1, alpha=0.8, label='Expérience')
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_xlabel('Temps [s]')
        ax.set_ylabel('Heave [cm]')
        ax.set_title('Mouvement vertical du corps (heave)')
        ax.legend(); ax.grid(True, alpha=0.3)

        # Sway
        ax = axes[1, 1]
        ax.plot(t_arr, sway * 100, 'g-', lw=1.5, label='SPH-ALE')
        if 't_s' in exp:
            ax.plot(exp['t_s'], exp['sway'] * 100, 'k--', lw=1, alpha=0.8, label='Expérience')
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_xlabel('Temps [s]')
        ax.set_ylabel('Sway [cm]')
        ax.set_title('Mouvement horizontal du corps (sway)')
        ax.legend(); ax.grid(True, alpha=0.3)

        # Roll
        ax = axes[2, 0]
        ax.plot(t_arr, roll, 'purple', lw=1.5, label='SPH-ALE')
        if 't_r' in exp:
            ax.plot(exp['t_r'], exp['roll'], 'k--', lw=1, alpha=0.8, label='Expérience')
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_xlabel('Temps [s]')
        ax.set_ylabel('Roll [°]')
        ax.set_title('Rotation du corps (roll)')
        ax.legend(); ax.grid(True, alpha=0.3)

        # Erreur RMS si données dispo
        ax = axes[2, 1]
        if 't_h' in exp and len(t_arr) > 10:
            f_heave_exp = interp1d(exp['t_h'], exp['heave'], bounds_error=False, fill_value=0)
            f_sway_exp  = interp1d(exp['t_s'], exp['sway'],  bounds_error=False, fill_value=0)
            err_h = heave - f_heave_exp(t_arr)
            err_s = sway  - f_sway_exp(t_arr)
            ax.plot(t_arr, np.abs(err_h)*100, 'b-',  lw=1, label='|Erreur heave|')
            ax.plot(t_arr, np.abs(err_s)*100, 'g--', lw=1, label='|Erreur sway|')
            ax.set_xlabel('Temps [s]')
            ax.set_ylabel('|Erreur| [cm]')
            ax.set_title('Erreur absolue SPH vs Expérience')
            ax.legend(); ax.grid(True, alpha=0.3)
            rms_h = np.sqrt(np.mean(err_h**2)) * 100
            rms_s = np.sqrt(np.mean(err_s**2)) * 100
            ax.set_title(f'Erreur abs.  RMS heave={rms_h:.2f}cm  sway={rms_s:.2f}cm')
        else:
            ax.text(0.5, 0.5, "Données expérimentales\nnon disponibles\n\n"
                    "Télécharger SPHERIC_TestCase12.zip\nsur spheric-sph.org",
                    ha='center', va='center', transform=ax.transAxes, fontsize=11,
                    color='gray')
            ax.set_title('Comparaison RMS')

        plt.tight_layout()
        out_fig = os.path.join(vtk_dir, 'SPHERIC_Test12_comparison.png')
        plt.savefig(out_fig, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"[OK] Figure comparaison → {out_fig}")


    # ==============================================================================
    # 6.  VTK ÉTENDU (élévation de surface + champs corps)
    # ==============================================================================

    def write_vtk_wf(filename, pos, vel, v_mesh, pres, types, rho, vol, m,
                     num_neighbors, shepard, wall_normals,
                     body_cm, body_theta, body_vel, body_omega):
        """VTK enrichi avec les champs spécifiques WaveFloat."""
        n = len(pos)
        with open(filename, 'w') as f:
            f.write("# vtk DataFile Version 3.0\nSPH_WAVEFLAT\nASCII\nDATASET POLYDATA\n")
            f.write(f"POINTS {n} float\n")
            for p in pos:
                f.write(f"{p[0]:.6f} {p[1]:.6f} 0.0\n")
            f.write(f"\nVERTICES {n} {2*n}\n")
            for i in range(n):
                f.write(f"1 {i}\n")
            f.write(f"\nPOINT_DATA {n}\n")
            # Champs scalaires
            for name, arr in [('Pressure', pres), ('Density', rho),
                               ('Volume', vol), ('Shepard', shepard), ('Mass', m)]:
                f.write(f"SCALARS {name} float 1\nLOOKUP_TABLE default\n")
                for v in arr:
                    f.write(f"{float(v):.6f}\n")
            f.write("SCALARS Type int 1\nLOOKUP_TABLE default\n")
            for tp in types:
                f.write(f"{int(tp)}\n")
            f.write("SCALARS Neighbors int 1\nLOOKUP_TABLE default\n")
            for nb in num_neighbors:
                f.write(f"{int(nb)}\n")
            # Champs vectoriels
            for name, arr in [('Velocity', vel), ('VelocityMesh', v_mesh),
                               ('WallNormals', wall_normals)]:
                f.write(f"VECTORS {name} float\n")
                for v in arr:
                    f.write(f"{v[0]:.6f} {v[1]:.6f} 0.0\n")


    # ==============================================================================
    # 7.  INITIALISATION DU CAS WaveFloat
    # ==============================================================================

    print("\n" + "="*60)
    print("  INITIALISATION — SPHERIC Test 12 WaveFloat")
    print("="*60)

    # ---- 7a. Clapet wavemaker ----
    pos_flap, nrm_flap, y0_arr = build_flap_particles(H_water, dx, n_layers)
    n_flap = len(pos_flap)
    print(f"Clapet wavemaker : {n_flap} particules ({n_layers} couches)")

    # ---- 7b. Corps flottant ----
    pos_body_pts, nrm_body0, rel_pos0 = build_floating_body(
        x_body_init, y_body_init, L_body, H_body, dx
    )
    n_body = len(pos_body_pts)

    # ---- 7c. Murs du bassin (fond + mur droit) ----
    # ---- 7c. Murs du bassin (fond + plage inclinée + mur droit) ----
    pos_wall_list, nrm_wall_list = [], []

    # 1. Fond plat (de x = -n_layers*dx à x = x_ramp_start)
    # On génère n_layers sous le niveau y = 0
    xs_flat = np.arange(-n_layers * dx, x_ramp_start, dx)
    for l in range(n_layers):
        for xi in xs_flat:
            pos_wall_list.append([xi, -(l + 0.5) * dx])
            nrm_wall_list.append([0.0, 1.0])

    # 2. Plage inclinée (de x = x_ramp_start à x = L_tank)
    # Pente de la rampe
    slope = H_ramp_end / (L_tank - x_ramp_start)  # Ici (0.6 - 0.0) / (8.0 - 4.0) = 0.15
    angle_ramp = np.arctan(slope)
    # Normale unitaire pointant vers l'intérieur du bassin (vers le haut et la gauche)
    nrm_r = np.array([-np.sin(angle_ramp), np.cos(angle_ramp)])

    xs_ramp = np.arange(x_ramp_start, L_tank + dx/2, dx)
    for xi in xs_ramp:
        y_surface_ramp = slope * (xi - x_ramp_start)
        for l in range(n_layers):
            # On décale les couches perpendiculairement à la surface de la rampe
            p_layer = np.array([xi, y_surface_ramp]) - (l + 0.5) * dx * nrm_r
            pos_wall_list.append(p_layer)
            nrm_wall_list.append(nrm_r)

    # 3. Mur vertical de fermeture tout à droite (à x = L_tank, de y = H_ramp_end à y_max)
    # Utile pour éviter que les vagues ne sortent si elles dépassent la rampe
    y_max_bassin = 0.7 # Hauteur de garde
    ys_rgt = np.arange(H_ramp_end, y_max_bassin, dx)
    for xi_mur in range(n_layers):
        x_pos = L_tank + (xi_mur + 0.5) * dx
        for yi in ys_rgt:
            pos_wall_list.append([x_pos, yi])
            nrm_wall_list.append([-1.0, 0.0])

    pos_wall = np.array(pos_wall_list)
    nrm_wall = np.array(nrm_wall_list)
    n_wall   = len(pos_wall)
    print(f"Murs du bassin (avec rampe) : {n_wall} particules")

    # ---- 7d. Fluide (grille régulière, exclusion zone corps) ----
    # ---- 7d. Fluide (Prise en compte de la rampe et exclusion du corps) ----
    pos_fluid_list = []
    
    for xi in np.arange(dx/2, L_tank, dx):
        # Calcul de la limite inférieure du fluide (0 si fond plat, hauteur de rampe sinon)
        if xi < x_ramp_start:
            y_bottom = 0.0
        else:
            y_bottom = slope * (xi - x_ramp_start)
            
        for yi in np.arange(y_bottom + dx/2, H_water, dx):
            # Vérification si la particule est à l'intérieur du corps flottant
            in_body = (abs(xi - x_body_init) <= L_body/2 + dx/2 and
                       abs(yi - y_body_init) <= H_body/2 + dx/2)
            if not in_body:
                pos_fluid_list.append([xi, yi])

    pos_fluid = np.array(pos_fluid_list)
    n_fluid   = len(pos_fluid)
    print(f"Fluide                      : {n_fluid} particules")

    # ---- 7e. Assemblage global ----
    #  Ordre : WALL | FLAP | BODY | FLUID
    idx_wall_start  = 0
    idx_flap_start  = n_wall
    idx_body_start  = n_wall + n_flap
    idx_fluid_start = n_wall + n_flap + n_body

    pos   = np.vstack([pos_wall, pos_flap, pos_body_pts, pos_fluid])
    types = np.concatenate([
        np.full(n_wall,  WALL),
        np.full(n_flap,  FLAP),
        np.full(n_body,  BODY),
        np.full(n_fluid, FLUID),
    ])
    n_p   = len(pos)
    print(f"TOTAL            : {n_p} particules")

    # ---- 7f. Normales globales ----
    wall_normals = np.zeros((n_p, 2))
    wall_normals[idx_wall_start  : idx_flap_start]  = nrm_wall
    wall_normals[idx_flap_start  : idx_body_start]  = nrm_flap
    wall_normals[idx_body_start  : idx_fluid_start] = nrm_body0

    # ---- 7g. Conditions initiales (physique) ----
    vel  = np.zeros((n_p, 2))
    rho  = np.full(n_p, rho0)
    pres = np.zeros(n_p)
    vol  = np.full(n_p, dx**2)
    m    = np.zeros(n_p)

    # Fluide : pression hydrostatique initiale
    mask_f    = (types == FLUID)
    y_fl      = pos[mask_f, 1]
    pres[mask_f] = rho0 * g_acc * np.maximum(0.0, H_water - y_fl) 
    rho[mask_f]  = get_density(pres[mask_f])
    m[mask_f]    = rho[mask_f] * vol[mask_f]

    # Murs et clapet
    mask_wf        = (types == WALL) | (types == FLAP)
    rho[mask_wf]   = rho0
    m[mask_wf]     = rho0 * dx**2

    # Corps flottant : masse répartie uniformément
    m_part_body    = mass_body_2D * dx**2 / (L_body * H_body)
    mask_body_g    = (types == BODY)
    m[mask_body_g] = m_part_body
    rho[mask_body_g] = rho_body
    pres[mask_body_g] = 0.0

    # ---- 7h. État du corps rigide ----
    body_vel   = np.array([0.0, 0.0])   # [vx, vy] du CM
    body_omega = 0.0                     # vitesse angulaire [rad/s]
    body_theta = 0.0                     # angle cumulé [rad]
    body_cm    = np.array([x_body_init, y_body_init])

    # Normales initiales du corps (pour la rotation)
    nrm_body0_global = nrm_body0.copy()

    # ---- 7i. Données de suivi (sondes + corps) ----
    probe_t    = []
    probe_eta1 = []
    probe_eta2 = []
    body_heave = []   # déplacement vertical CM (= body_cm[1] - y_body_init)
    body_sway  = []   # déplacement horizontal CM (= body_cm[0] - x_body_init)
    body_roll  = []   # angle θ en degrés
    body_cm_hist = [] # historique CM [x, y]

    # ---- 7j. Initialisation de la boucle ----
    num_neighbors = np.zeros(n_p, dtype=int)
    shepard_coeff = np.zeros(n_p)
    drho          = np.zeros(n_p)
    accel         = np.zeros((n_p, 2))
    v_mesh        = np.zeros((n_p, 2))
    S_G_local     = np.zeros(n_p)
    S_dist        = np.zeros(n_p)
    grad_S_G      = np.zeros((n_p, 2))
    grad_S_dist   = np.zeros((n_p, 2))

    t, step, dt = 0.0, 0, CFL * h / c0
    print(f"\nΔt initial : {dt:.6f} s")

    # Chargement des données expérimentales (si disponibles)
    exp_data = load_experimental('.')

    print("\nDémarrage de la boucle temporelle...")
    print("-" * 60)
 

#--------------------------------------------------------------------------------------------------------------------- 
if testCase == 'car':
    # --------------------------
    # Initialisation Scène
    # --------------------------
    # 1. Murs du bac
    # --------------------------
    # Initialisation Scène (Quai + Bassin)
    # --------------------------
    print("\nDémarrage de la création de la géométrie...")
    # 1. Murs du bac et du quai
    layers = 5
    # -------------------------------
    # 1. Création des murs du bac
    # -------------------------------
#     x_wall = []
#     y_wall = []

#     # Plateau gauche horizontal
#     x_plateau = np.arange(0, L_plateau + dx, dx)
#     y_plateau = np.zeros_like(x_plateau) + H_water
#     for l in range(layers):
#         x_wall.extend(x_plateau)
#         y_wall.extend(y_plateau - l*dx)

#     # Pente descendante
#     x_slope = np.arange(L_plateau, L_plateau + L_slope , dx)
#     y_slope = H_water - np.tan(theta) * (x_slope - L_plateau)
#     for l in range(layers):
#         x_wall.extend(x_slope)
#         y_wall.extend(y_slope - l*dx)

#     # Plateau fond du bac
#     x_bottom = np.arange(L_plateau + L_slope, L_plateau + L_slope + L_bottom , dx)
#     y_bottom = np.full_like(x_bottom, y_slope[-1])
#     for l in range(layers):
#         x_wall.extend(x_bottom)
#         y_wall.extend(y_bottom - l*dx)

#     # Remontée symétrique
#     x_rise = np.arange(L_plateau + L_slope + L_bottom, L_plateau + L_slope + L_bottom + L_rise + dx, dx)
#     y_rise = y_bottom[-1] + np.tan(theta) * (x_rise - x_rise[0])
#     for l in range(layers):
#         x_wall.extend(x_rise)
#         y_wall.extend(y_rise - l*dx)

#     # Plateau final horizontal
#     x_plateau2 = np.arange(x_rise[-1], x_rise[-1] + L_plateau_right + dx, dx)
#     y_plateau2 = np.full_like(x_plateau2, y_rise[-1])
#     for l in range(layers):
#         x_wall.extend(x_plateau2)
#         y_wall.extend(y_plateau2 - l*dx)

#     pos_wall = np.vstack([x_wall, y_wall]).T
# -------------------------------
    # 1. Création des murs du bac (Corrigée)
    # -------------------------------
    x_wall = []
    y_wall = []

    # Facteurs de correction géométrique pour l'orthogonalité des couches
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    for l in range(layers):
        offset = l * dx
        
        # Plateau gauche horizontal : décalage vers le bas (y)
        x_plateau = np.arange(0, L_plateau, dx)
        y_plateau = np.zeros_like(x_plateau) + H_water - offset
        x_wall.extend(x_plateau)
        y_wall.extend(y_plateau)

        # Pente descendante : décalage normal à la pente (-sin*offset, -cos*offset)
        # On commence à l'extrémité du plateau gauche mis à jour pour éviter le trou/chevauchement
        x_start_slope = L_plateau - offset * sin_t
        x_end_slope = L_plateau + L_slope - offset * sin_t
        x_slope = np.arange(x_start_slope, x_end_slope, dx)
        y_slope = H_water - np.tan(theta) * (x_slope - L_plateau) - offset / cos_t
        x_wall.extend(x_slope)
        y_wall.extend(y_slope)

        # Plateau fond du bac : décalage vers le bas (y)
        y_bottom_val = H_water - np.tan(theta) * L_slope - offset
        x_start_bottom = L_plateau + L_slope - offset * sin_t
        x_end_bottom = L_plateau + L_slope + L_bottom + offset * sin_t
        x_bottom = np.arange(x_start_bottom, x_end_bottom, dx)
        y_bottom = np.full_like(x_bottom, y_bottom_val)
        x_wall.extend(x_bottom)
        y_wall.extend(y_bottom)

        # Remontée : décalage normal à la pente (+sin*offset, -cos*offset)
        x_start_rise = L_plateau + L_slope + L_bottom + offset * sin_t
        x_end_rise = L_plateau + L_slope + L_bottom + L_rise + offset * sin_t
        x_rise = np.arange(x_start_rise, x_end_rise, dx)
        y_rise = (H_water - np.tan(theta) * L_slope) + np.tan(theta) * (x_rise - (L_plateau + L_slope + L_bottom)) - offset / cos_t
        x_wall.extend(x_rise)
        y_wall.extend(y_rise)

        # Plateau final horizontal : décalage vers le bas (y)
        x_start_right = L_plateau + L_slope + L_bottom + L_rise + offset * sin_t
        x_plateau2 = np.arange(x_start_right, x_start_right + L_plateau_right + dx, dx)
        y_plateau2 = np.full_like(x_plateau2, H_water - offset)
        x_wall.extend(x_plateau2)
        y_wall.extend(y_plateau2)

    pos_wall = np.vstack([x_wall, y_wall]).T
    print("\n Mur ok...")
    # -------------------------------
    # 2. Création du fluide
    # -------------------------------
#     x_fluid = np.arange(0, L_plateau + L_slope + L_bottom + L_rise + L_plateau_right, dx)
#     H_fluid = H_water  # profondeur d'eau
#     target_surface_level = H_water
#     # -------------------------------
#     # 2. Création du fluide (uniquement dans le bassin)
#     # -------------------------------
#     x_f, y_f = [], []

#     # On définit les limites du bassin pour ne pas remplir les plateaux
#     x_start_basin = L_plateau
#     x_end_basin = L_plateau + L_slope + L_bottom + L_rise

#     for x in x_fluid:
#         # Check if x is within the basin zone
#         if x_start_basin <= x <= x_end_basin:
            
#             # (Calculation of y_bac is identical as before)
#             if x <= L_plateau + L_slope:
#                 y_bac = H_water - np.tan(theta) * (x - L_plateau)
#             elif x <= L_plateau + L_slope + L_bottom:
#                 y_bac = H_water - np.tan(theta) * L_slope
#             else:
#                 y_bac = (H_water - np.tan(theta) * L_slope) + np.tan(theta) * (x - (L_plateau + L_slope + L_bottom))

#             # --- KEY CHANGE FOR BULK FILLING ---
            
#             # Calculate the total vertical space to fill
#             total_height_to_fill = target_surface_level - y_bac
            
#             # Determine how many particles fit in this space, 
#             # starting one dx above the floor and stopping just below the surface.
#             n_layers_to_fill = int((total_height_to_fill - dx) / dx)

#             # Bulk fill loop
#             for i in range(1, n_layers_to_fill + 1): 
#                 y_particle = y_bac + i * dx
#                 if y_particle < target_surface_level - 0.1 * dx:
#                     x_f.append(x)
#                     y_f.append(y_particle)                
                
#     print("\n Fluid ok...")
    
#     pos_fluid = np.vstack([x_f, y_f]).T
# -------------------------------
    # 2. Création du fluide (Approche par masque de distance)
    # -------------------------------
    y_floor_min = H_water - np.tan(theta) * L_slope
    target_surface_level= H_water
    x_f_grid = np.arange(L_plateau, L_plateau + L_slope + L_bottom + L_rise, dx)
    y_f_grid = np.arange(y_floor_min - dx, H_water + dx, dx)
    
    x_f, y_f = [], []
    
    for x in x_f_grid:
        for y in y_f_grid:
            if y >= target_surface_level - 0.1 * dx:
                continue
                
            # Calcul analytique exact de la hauteur du fond du bac à la position x (couche l=0)
            if x <= L_plateau + L_slope:
                y_bac = H_water - np.tan(theta) * (x - L_plateau)
            elif x <= L_plateau + L_slope + L_bottom:
                y_bac = H_water - np.tan(theta) * L_slope
            else:
                y_bac = (H_water - np.tan(theta) * L_slope) + np.tan(theta) * (x - (L_plateau + L_slope + L_bottom))
            
            # Condition de sécurité stricte : la particule de fluide doit être 
            # à au moins 0.8 * dx au-dessus du fond pour ne pas être injectée dans le mur
            if y >= y_bac + 0.8 * dx:
                x_f.append(x)
                y_f.append(y)

    pos_fluid = np.vstack([x_f, y_f]).T
    if randompos:
        pos_fluid += (np.random.rand(*pos_fluid.shape) - 0.5) * randomlength  * dx
    # 3. Voiture (Positionnée sur le quai à gauche)
    
    pos_car, m_car_part, car_I, car_normals_init = create_car(posXinitCar, posYinitCar, car_mass)
    # --- CORRECTION DE L'OVERLAP ICI (Avant assemblage global) ---
    # Le sol sous la voiture est le plateau gauche horizontal, son y maximal est H_water
    y_wall_max = H_water 
    y_car_min = np.min(pos_car[:, 1])
    
    distance_securite = R_car + R_wall + 1e-4
    overlap = (y_wall_max + distance_securite) - y_car_min
    
    if overlap > 0:
        pos_car[:, 1] += overlap
        print(f"--> [Correction géométrique] pos_car remontée de {overlap:.4f}m pour éviter le jump au démarrage.")
    
    print("CAR OK...")
    
    # Assemblage
    pos = np.vstack([pos_wall, pos_fluid, pos_car])
    types = np.concatenate([np.full(len(pos_wall), WALL), np.full(len(pos_fluid), FLUID), np.full(len(pos_car), CAR)])
    n_p=len(pos) 
    
     # On la place à x = -1.0, juste au dessus du quai (H_quay + rayon de sécurité)
    # 1. Initialisation du vecteur des normales pour toutes les particules
    # -------------------------------
    # 5. Initialisation des normales
    # -------------------------------
    wall_normals = np.zeros((n_p,2))

    # Normales murs/pentes
    mask_wall = (types == WALL)
    wall_normals[mask_wall] = [0,1]  # initial vertical

    # Plateau gauche
    mask_plateau = (pos[:,0] <= L_plateau) & mask_wall
    wall_normals[mask_plateau] = [0,1]

    # Pente descendante
    mask_slope = (pos[:,0] > L_plateau) & (pos[:,0] <= L_plateau + L_slope) & mask_wall
    wall_normals[mask_slope] = [np.sin(theta), np.cos(theta)]

        # Plateau fond
    mask_bottom = (pos[:,0] > L_plateau + L_slope) & (pos[:,0] <= L_plateau + L_slope + L_bottom) & mask_wall
    wall_normals[mask_bottom] = [0,1]

    # Remontée
    mask_rise = (pos[:,0] > L_plateau + L_slope + L_bottom) & (pos[:,0] <= L_plateau + L_slope + L_bottom + L_rise) & mask_wall
    wall_normals[mask_rise] = [-np.sin(theta), np.cos(theta)]

    # Plateau final
    mask_plateau2 = (pos[:,0] > L_plateau + L_slope + L_bottom + L_rise) & mask_wall
    wall_normals[mask_plateau2] = [0,1]

    # 3. Assignation pour la voiture (mobiles)
    # On récupère les normales calculées par create_car
    
    wall_normals[types == CAR] = car_normals_init

    # 4. Nettoyage : s'assurer que les fluides ont une normale nulle
    wall_normals[types == FLUID] = [0, 0]
    print("\n NORMALES ok...")   
    car_vel = np.array([carvitesse, 0.0]) # Vitesse horizontale vers le bassin (2.5 m/s)
    car_omega = 0.0   
    car_theta = get_car_theta(posXinitCar)
    rho, pres = np.zeros(n_p), np.zeros(n_p)
    m = np.zeros(n_p)
    # États
    vel = np.zeros((n_p, 2))
    vel[types == CAR] = car_vel
    # Fluid
    mask_f = (types == FLUID)
    y_fluid = pos[mask_f, 1]
    pres[mask_f] = rho0 * g_acc * np.maximum(0.0, (1.0 - (0.7+y_fluid))) 
    rho[mask_f] = get_density(pres[mask_f])
    m [mask_f] =rho[mask_f]*dx**2
    
    # Wall
    mask_wall = (types == WALL)
    pres_wall_init = rho0 * g_acc * np.maximum(0.0, H_water - pos[mask_wall, 1])
    rho[mask_wall] = get_density(pres_wall_init)
    m [mask_wall] =rho[mask_wall]*dx**2
    # Car
    m[types == CAR] = m_car_part
    rho[types == CAR] = m_car_part / (dx**2)
    pres[types == CAR] = 0
    # 1. Détermine la masse d'une particule type (fluide ou mur)
    m_p =  car_mass / len(pos_car) # (ou une valeur représentative)

    # 2. Pas de temps critique SPH (mis à jour avec ton c0 = 10 * Umax)
    dtcrtq = CFL * h / (c0 + carvitesse)

    # 3. Raideur LOCALE : calibrée pour stopper UNE particule lancée à Umax
    # Le préfacteur (ici 2.0) peut être augmenté si la voiture tente encore de traverser
    k_local = 2.0 * m_p / (dtcrtq)**2
    
    # 4. Amortissement LOCAL : basé sur le système critique d'une particule
    damping_local = 2.0 * np.sqrt(m_p * k_local) * 0.95
    

    print(f"--- Calibration Contact Particule ---")
    print(f"k_local: {k_local:.4e} | damping_local: {damping_local:.4e}")
    vol = m / rho
    num_neighbors = np.zeros(n_p, dtype=int) 
    shepard_coeff = np.zeros(n_p)
    v_mesh=np.zeros_like(vel)
    t, step, dt = 0.0, 0, 0.01 * h / (c0+ carvitesse)
    
#---------------------------------------------------------------------------------------------------------------------    
if testCase == 'TGV' or testCase =='TGVmesh' :
    print("\n Initialisation")
    x=np.arange(dx/2,L,dx)
    X,Y=np.meshgrid(x,x)
    pos=np.column_stack((X.ravel(),Y.ravel()))
    if randompos:
        np.random.seed(42)
        pos += np.random.uniform(-0.1*dx, 0.1*dx, pos.shape)
    
    n=len(pos)
    vel,p_init=get_tgv_solution(pos, 0, L, U0, nu)
    rho=rho0*(1+p_init/(rho0*c0*c0)) #get_density(p_init)#rho0*(1+p_init/(rho0*c0*c0))
    vol=np.ones(n)*dx*dx
    m=rho*vol
    v_mesh=np.zeros_like(vel)
    #=types = np.array([TYPE_FLUID]*n )
    t, step, dt = 0.0, 0, 0.01 * h / c0
    num_neighbors = np.zeros(n, dtype=int)
    shepard_coeff = np.zeros(n)  
           
    wall_normals  = np.zeros((n, 2))
    pres = get_pressure(rho)                
    types =  np.full(len(pos), FLUID)
    mask_f = (types == FLUID)
    
    
    
if testCase == 'AirFoil':
    print("\n Initialisation")
    # Definition geometrique du cylindre central
    center_x, center_y = L / 2.0, L / 2.0
    radius = 0.15  # Rayon du cylindre central
    
    # CORRECTION : On passe à 5 ou 6 couches pour garantir l'épaisseur de support du noyau (couramment >= 3 * dx)
    # Vu que le pas radial est divisé par deux, il faut doubler le nombre de couches.
    dx_wall = dx / 2.0
    n_layers_wall = 6  
    
    # 2. Generation des parois concentriques denses (WALL)
    pos_wall_list = []
    wall_normals_list = []        
    vol_wall_list = [] # Stockage individuel des volumes pour gérer la géométrie courbe
    
    for layer in range(n_layers_wall):
        # Rayon de la couche courante avec un pas radial reduit a dx/2
        r_layer = radius - (layer * dx_wall)
            
        # Estimation du perimetre avec le pas d'espace reduit de moitie
        perimeter = 2.0 * np.pi * r_layer
        n_points_layer = int(np.round(perimeter / dx_wall))
            
        # Arc de cercle reel par particule sur cette couche
        ds = perimeter / n_points_layer if n_points_layer > 0 else dx_wall
            
        for i in range(n_points_layer):
            theta = (2.0 * np.pi * i) / n_points_layer
                
            # Coordonnees de la particule de paroi dense
            xw = center_x + r_layer * np.cos(theta)
            yw = center_y + r_layer * np.sin(theta)
            pos_wall_list.append([xw, yw])
                
            # La normale pointe vers le fluide (exterieur du cercle)
            wall_normals_list.append([np.cos(theta), np.sin(theta)])
            
            # Volume elementaire associe a la surface de revolution discrete : ds * dr
            vol_wall_list.append(ds * dx_wall)
                
    pos_wall = np.array(pos_wall_list)
    wall_normals_wall = np.array(wall_normals_list)
    vol_wall_arr = np.array(vol_wall_list)
    n_w = len(pos_wall)

    # 3. Generation de la grille de fluide cartesienne de base (Resolution dx standard)
    xf = np.arange(dx/2, L, dx)
    yf = np.arange(dx/2, L, dx)
    Xf, Yf = np.meshgrid(xf, yf)
    pos_fluid_raw = np.column_stack((Xf.ravel(), Yf.ravel()))
        
    # 4. Nettoyage : Distance d'exclusion basee sur le rayon physique et le dx du fluide
    dist_to_center = np.sqrt((pos_fluid_raw[:, 0] - center_x)**2 + (pos_fluid_raw[:, 1] - center_y)**2)
    fluid_mask_clean = dist_to_center > (radius + 2.0 * dx)        
    pos_fluid = pos_fluid_raw[fluid_mask_clean]
    n_f = len(pos_fluid)
    
    # 5. Assemblage final : WALL en premier, puis FLUID
    pos = np.vstack([pos_wall, pos_fluid])
    n = len(pos)        
    
    # Attribution stricte des types de particules
    types = np.zeros(n, dtype=int)
    types[:n_w] = WALL
    types[n_w:] = FLUID
        
    # 6. Perturbation aleatoire du fluide si demandee
    if randompos:
        np.random.seed(42)
        pos[types == FLUID] += np.random.uniform(-0.05 * dx, 0.05 * dx, (n_f, 2))
            
    # 7. Initialisation des matrices de normales globales
    wall_normals = np.zeros((n, 2))
    wall_normals[:n_w, :] = wall_normals_wall  
    
    # 8. Allocation des etats physiques initiaux
    vel = np.zeros((n, 2))
    vel[types == FLUID, 0] = U0  
    
    v_mesh = np.zeros_like(vel)        
    p_init = np.zeros(n)
    rho = np.full(n, rho0)      
    
    # CORRECTION : Attribution rigoureuse des volumes distincts
    vol = np.zeros(n)
    vol[types == WALL] = vol_wall_arr  # Utilise le volume d'integration curviligne ajuste (dx/2 * ds)
    vol[types == FLUID] = dx**2        # Conserve le volume standard pour le fluide sain
    
    m = rho * vol
    
    drho = np.zeros(n)
    accel = np.zeros((n, 2))
    
    # Variables de suivi de la boucle SPH
    t, step, dt = 0.0, 0, 0.01 * h / c0
    num_neighbors = np.zeros(n, dtype=int)    
    shepard_coeff = np.zeros(n)        
    pres = get_pressure(rho)
    mask_f = (types == FLUID)
#---------------------------------------------------------------------------------------------------------------------    
    
if testCase == 'Poiseuille':
    print("\n Initialisation Écoulement de Poiseuille (1600 fluides)")
    
    # 1. Génération des particules fluides (Y allant de dx/2 à L - dx/2)
    xf = np.arange(dx/2, L, dx)
    yf = np.arange(dx/2, L, dx)
    Xf, Yf = np.meshgrid(xf, yf)
    pos_fluid = np.column_stack((Xf.ravel(), Yf.ravel()))
    n_f = len(pos_fluid) # Égal à 1600
    
    # 2. Génération des parois (Plaque inférieure et supérieure)
    pos_wall_list = []
    # Parois basses (y < 0)
    for l in range(n_layers_wall):
        y_val = -(l + 0.5) * dx
        for x_val in xf:
            pos_wall_list.append([x_val, y_val])
            
    # Parois hautes (y > L)
    for l in range(n_layers_wall):
        y_val = L + (l + 0.5) * dx
        for x_val in xf:
            pos_wall_list.append([x_val, y_val])
            
    pos_wall = np.array(pos_wall_list)
    n_w = len(pos_wall)
    
    # Assemblage des positions : WALL en premier, puis FLUID
    pos = np.vstack([pos_wall, pos_fluid])
    n = len(pos)
    
    # Attribution des types
    types = np.zeros(n, dtype=int)
    types[:n_w] = WALL   # Mettre votre constante associée aux parois fixes (ex: TYPE_GHOST)
    types[n_w:] = FLUID  # Votre constante FLUID
    
    # Perturbation aléatoire optionnelle (uniquement sur le fluide pour tester la stabilité)
    if randompos:
        np.random.seed(42)
        pos[types == FLUID] += np.random.uniform(-0.05*dx, 0.05*dx, (n_f, 2))
        
    # États initiaux (Au repos complet)
    vel = np.zeros((n, 2))
    v_mesh = np.zeros_like(vel)
    
    # Pression initiale hydrostatique nulle (le fluide est mis en mouvement par g)
    p_init = np.zeros(n)
    rho= rho0 * (1.0 + p_init / (rho0 * c0**2))
    vol = np.ones(n) * dx * dx
    m = rho * vol
    
    # Paramètres de boucle
    t, step, dt = 0.0, 0, 0.01 * h / c0
    num_neighbors = np.zeros(n, dtype=int)
    shepard_coeff = np.zeros(n)         
    wall_normals = np.zeros((n, 2))
    
    # Les normales pointent vers l'intérieur du canal fluide
    wall_normals[:len(xf)*n_layers_wall, 1] = 1.0  # Paroi basse pointe vers +y
    wall_normals[len(xf)*n_layers_wall:n_w, 1] = -1.0 # Paroi haute pointe vers -y
    
    pres = get_pressure(rho)
    mask_f = (types == FLUID)    
#--------------------------------------------------------------------------------------------------------------------- 
if testCase == 'DamBreak':
    box_w, box_h, layers = L_column*4.0, L_column*4.0, 3  #4 par 4 à terme
    # On s'assure que les murs finissent juste avant 0
    x_wall = np.arange(-layers*dx, box_w + layers*dx, dx)
    y_wall = np.arange(-layers*dx, box_h, dx)
    x_grid, y_grid = np.meshgrid(x_wall, y_wall)
    p_grid = np.vstack([x_grid.ravel(), y_grid.ravel()]).T

    # Masque pour ne garder que le "U" de la boite
    mask_wall = (p_grid[:,1] < 0) | (p_grid[:,0] < 0) | (p_grid[:,0] > box_w - dx)
    pos_wall = p_grid[mask_wall]
    types_wall = np.full(len(pos_wall), WALL)

    # Calcul des normales géométriques (Unitaires)
    wall_normals = np.zeros((len(pos_wall), 2))
    for i, p in enumerate(pos_wall):
    # Mur Gauche (x < 0) -> Normale vers la droite (1, 0)
        if p[0] < 0:
            wall_normals[i] = [1.0, 0.0]
        # Mur Droit (x > box_w - dx) -> Normale vers la gauche (-1, 0)
        elif p[0] > box_w - 1.5*dx:
            wall_normals[i] = [-1.0, 0.0]
        # Fond (y < 0) -> Normale vers le haut (0, 1)
        if p[1] < 0:
            # Gestion des coins : on additionne et normalise
            wall_normals[i] += [0.0, 1.0]
    
        # Normalisation pour avoir un vecteur unitaire
        norm = np.linalg.norm(wall_normals[i])
        if norm > 1e-9:
            wall_normals[i] /= norm


    # --- Grille Fluide ---
    # On commence à 0.0 pour être contre le mur, 
    # mais on s'assure que le mur n'occupe pas l'espace >= 0 à l'intérieur
    xf, yf = np.meshgrid(np.arange(0, L_column, dx), 
                     np.arange(0, H_column, dx))
    pos_fluid = np.vstack([xf.ravel(), yf.ravel()]).T
    if randompos:
        pos_fluid += (np.random.rand(*pos_fluid.shape) - 0.5) * noise_amp  * dx
    
    # Extension du tableau des normales à toutes les particules (0 pour le fluide)
    normals = np.zeros((len(pos_wall) + len(pos_fluid), 2))
    normals[:len(pos_wall)] = wall_normals
    pos = np.vstack([pos_wall, pos_fluid])
    types = np.concatenate([types_wall, np.full(len(pos_fluid), FLUID)])
    n_p = len(pos)

    vel, vol = np.zeros((n_p, 2)), np.full(n_p, dx**2)
    rho, pres = np.zeros(n_p), np.zeros(n_p)

    mask_f = (types == FLUID)
    mask_wall = (types == WALL)
    y_fluid = pos[mask_f, 1]
    
    pres[mask_f] = rho0 * g_acc * np.maximum(0.0, (H_column - y_fluid)) 
    rho[mask_f] = get_density(pres[mask_f])
    
    # --------------------------
    # Boucle Temporelle
    # --------------------------
    t, step, dt = 0.0, 0, 0.01 * h / c0


    vel[types == FLUID] = 0
    vel[types == WALL]  = 0
    rho[types == WALL]  = rho0
    pres[types==WALL] = 0.0
    vol[mask_wall] = dx**2
    m=rho*vol
    #d_m = np.zeros(n_p)
    drho = np.zeros(n_p)

    accel = np.zeros((n_p, 2))
    num_neighbors = np.zeros(n_p, dtype=int)
    shepard_coeff = np.zeros(n_p)
    E_history = []    
    time_history = []
    
    
#--------------------------------------------------------------------------------------------------------------------- 
if testCase == 'funnel':
# --- Génération des Murs (Lisses) ---
    pos_wall = generate_funnel_wall(dx, n_layers)
    
    # --- Génération du Fluide (Correction par masquage de grille) ---
    pos_fluid_list = []
    
    # Établir les limites globales de la boîte englobante du fluide
    y_min, y_max = dx, H_column - 0.5 * dx
    r_max_global = 0.025 # Rayon max du funnel (y2)
    
    # Générer une grille cartésienne propre, centrée sur l'axe x=0
    y_vals = np.arange(y_min, y_max, dx)
    
    for y in y_vals:
        # Calcul du rayon max autorisé à cette altitude y, 
        # en tenant compte de la pente locale pour avoir 0.5*dx orthogonal
        drdy = get_funnel_derivative(y)
        r_wall = get_funnel_radius(y)
        
        # Distance de sécurité radiale ajustée à la pente locale : dx * sqrt(1 + (dr/dy)^2) / 2
        safety_margin = 0.5 * dx * np.sqrt(1.0 + drdy**2)
        r_fluid_limit = r_wall - safety_margin
        
        if r_fluid_limit < 0.5 * dx:
            continue
            
        # Détermination des bornes x sur la grille régulière centrée sur 0
        # On s'assure que le point le plus à droite ne dépasse pas r_fluid_limit
        max_nx = int(np.floor((r_fluid_limit - 0.0) / dx))
        
        # Génération symétrique stricte de -max_nx*dx à +max_nx*dx
        for nx in range(-max_nx, max_nx + 1):
            x = nx * dx
            pos_fluid_list.append([x, y])
            
    pos_fluid = np.array(pos_fluid_list).reshape(-1, 2)
    
    if randompos:    
        pos_fluid = add_position_noise(pos_fluid, noise_amp)
    
    # --- Après avoir défini pos_wall et pos_fluid ---
    pos = np.vstack([pos_wall, pos_fluid])
    types = np.concatenate([np.full(len(pos_wall), WALL), np.full(len(pos_fluid), FLUID)])
    n_p = len(pos)
    # --- Juste après la génération de pos_wall ---
    wall_normals = np.zeros((len(pos),2))
    wall_normals[types==WALL] = compute_wall_normals(pos_wall)

    # Initialisation des tableaux à la bonne taille
    vel = np.zeros((n_p, 2))
    rho = np.full(n_p, rho0)
    pres = np.zeros(n_p)
    vol = np.full(n_p, dx**2)
    m = np.zeros(n_p)

    m = rho * vol
    # --- Définition des masques initiaux ---
    mask_f = (types == FLUID)
    mask_w = (types == WALL)

    
    sum_w = np.zeros(n_p)
    # Utilisation d'une requête globale plus rapide
    vol[:] = dx**2
    # 2. Pression hydrostatique théorique
    # On utilise la hauteur maximale du fluide pour le calcul (H_column)
    y_fluid = pos[mask_f, 1]
    p_hydro = 0.0#rho0 * g_acc * np.maximum(0.0, (H_column - y_fluid))
    if Initcase : 
        p_hydro = rho0 * g_acc * np.maximum(0.0, (H_column - y_fluid))
    else:
        p_hydro = 0.0
    
    pres[mask_f] = p_hydro 
    rho[mask_f] = get_density(pres[mask_f])    
    m[mask_f] = rho[mask_f] * vol[mask_f]

    # Pour les murs : densité de base et masse calculée sur le volume SPH
    rho[mask_w] = rho0
    pres[mask_w] = 0.0
    m[mask_w] = rho0 * vol[mask_w]
#e_int[mask_f] = pres[mask_f] / ((gamma - 1.0) * rho[mask_f])
#E[mask_f] = e_int[mask_f] + 0.5 * np.sum(vel[mask_f]**2, axis=1)
#e = np.zeros(n_p)
#e[mask_f] = pres[mask_f] / ((gamma - 1) * rho[mask_f])



# --------------------------
# Boucle Temporelle
# --------------------------
    t, step, dt = 0.0, 0, 0.01 * h / c0


    vel[types == FLUID] = 0
    vel[types == WALL]  = 0
    rho[types == WALL]  = rho0
    #d_m = np.zeros(n_p)
    drho = np.zeros(n_p)

    accel = np.zeros((n_p, 2))
    num_neighbors = np.zeros(n_p, dtype=int)
    shepard_coeff = np.zeros(n_p)
    v_mesh = np.copy(vel)
    
    
    def funnel_height(times, pos_history, types_history,
                  FLUID,
                  z_exit,
                  percentile=98):
        """
        Compute normalized height H~(t)

        Parameters
        ----------
        times : (Nt,)
        pos_history : list of arrays (N,3)
        types_history : list
        z_exit : outlet elevation 0.0
        """

        H = []

        for pos, types in zip(pos_history, types_history):

            fluid_z = pos[types == FLUID, 1]

            z_surface = np.percentile(fluid_z, percentile)

            H.append(z_surface - 0.0)

        H = np.array(H)

        H0 = H[0]

        H_tilde = H / H0

        return H, H_tilde
    def funnel_discharge_coefficient(
        pos,
        vel,
        vol,
        types,
        FLUID,
        exit_center,
        exit_radius,
        normal,
        g,
        H):

        fluid_mask = (types == FLUID)

        pos_f = pos[fluid_mask]
        vel_f = vel[fluid_mask]
        vol_f = vol[fluid_mask]

        # distance à la sortie
        r = np.linalg.norm(pos_f - exit_center, axis=1)

        outlet_mask = r < exit_radius

        if np.sum(outlet_mask) == 0:
            return 0.0, 0.0

        vel_out = vel_f[outlet_mask]
        vol_out = vol_f[outlet_mask]

        # projection normale
        vn = vel_out @ normal
        # 1. Calculer la hauteur d'eau actuelle (H)
        # Si pos_fluide est l'ensemble de tes particules fluides
        if len(pos) > 0:
            current_z_max = np.max(pos[:, 1])
            z_exit = 0.020 # À ajuster selon ta géométrie (y1 dans ton code)
            H = max(0.0, current_z_max - z_exit)
        else:
            H = 0.0

        # 2. Calcul du débit théorique sécurisé
        g = 9.81
        exit_radius = 0.003 # Rayon du bas de l'entonnoir (R_bottom)
        Q_theory = np.pi * exit_radius**2 * np.sqrt(2 * g * H)
        Q = np.sum(vol_out * vn)
        Cd = Q / Q_theory
        
        
      
        return Q, Cd
    def funnel_scaling_law(times, H):
        """
        Fit alpha exponent:
            -dH/dt = K H^alpha
        """

        dt = np.gradient(times)
        dHdt = np.gradient(H, times)

        mask = (H > 1e-6) & (dHdt < 0)

        Hfit = H[mask]
        Y = -dHdt[mask]

        logH = np.log(Hfit)
        logY = np.log(Y)

        alpha, logK = np.polyfit(logH, logY, 1)

        K = np.exp(logK)

        return alpha, K
    import numpy as np

    import numpy as np

    def funnel_height(pos,
                      types,
                      FLUID,
                      vertical_axis=1,
                      z_exit=0.0,
                      percentile=98,
                      last_H=None):
        """
        Robust funnel height estimator.
        Works in 2D or 3D.

        vertical_axis :
            0 -> x vertical
            1 -> y vertical (MOST 2D SPH)
            2 -> z vertical (3D SPH)
        """

        fluid_mask = (types == FLUID)

        # ---- no fluid left ----
        if np.sum(fluid_mask) == 0:
            return 0.0 if last_H is None else last_H

        fluid_coord = pos[fluid_mask, vertical_axis]

        if fluid_coord.size == 0:
            return 0.0 if last_H is None else last_H

        surface = np.percentile(fluid_coord, percentile)

        H = max(surface - 0.0, 0.0)

        return H
    H_history = []
    Cd_history = []
    time_history = []
    Q_history = []
#---------------------------------------------------------------------------------------------------------------------     
if testCase == 'jet' or testCase == 'Mill':
    # Initialisation GÉOMÉTRIE (Version Inclinaison Unifiée)
    # ---------------------------------------------------
    pos = np.empty((0, 2)); vel = np.empty((0, 2)); types = np.empty(0, dtype=int)

    # 1. Définition précise des lignes de jet de base
    bx = np.arange(jet_x_min, jet_x_max + dx/2, dx) 

    y_inlet_top = y_center + gap
    y_inlet_bottom = y_inlet_bottom
    y_inlet_middle = y_inlet_middle
    # Suppression des anciens blocs redondants de buffers verticaux (p_bt, p_bb) 
    # pour tout centraliser proprement dans la section E.

    if  testCase != 'Mill':
        # Création d'un MUR horizontal en bas pour recevoir le jet
        wall_width = (jet_x_max - jet_x_min) * 8.0
        margin = 10.0 * h  
        x_wall = np.arange(jet_x_min - margin, jet_x_max + margin + dx/2, dx)   
        p_wall = np.array([[x, y_inlet_bottom - l*dx] for l in range(n_layers) for x in x_wall])    
        pos = np.vstack([pos, p_wall])
        types = np.concatenate([types, np.full(len(p_wall), WALL)])
        vel = np.vstack([vel, np.zeros((len(p_wall), 2))])        
        wall_normals = np.zeros((len(pos), 2))
        mask_wall = (types == WALL)
        wall_normals[mask_wall] = [0.0, 1.0]

    elif  testCase == 'Mill':
        print("\n Initialisation de la roue à aubes (Cas Mill - Courbure & Continuité Corrigées)")  

        pos_wheel_list = []
        normals_wheel_list = []

        # --- A. Configuration des Aubes Incurvées ---
        blade_angles = [np.deg2rad(360.0 / n_blades * b) for b in range(n_blades)]
        r_in = r_hub
        r_out = r_hub + blade_length
        delta_theta_blade = np.deg2rad(-35.0) 

        approx_length = np.sqrt((r_out - r_in)**2 + (r_in * delta_theta_blade)**2)
        n_s = int(np.ceil(approx_length / dx))
        s_values = np.linspace(0.0, 1.0, n_s)
        t_values = np.arange(-blade_thickness/2, blade_thickness/2 + dx/2, dx)

        for beta in blade_angles:
            for s in s_values:
                r_local = r_in + s * (r_out - r_in)
                theta_local = beta + s * delta_theta_blade

                x_spine = wheel_centerX + r_local * np.cos(theta_local)
                y_spine = wheel_centerY + r_local * np.sin(theta_local)

                dr_ds = r_out - r_in
                dtheta_ds = delta_theta_blade
                tx = dr_ds * np.cos(theta_local) - r_local * np.sin(theta_local) * dtheta_ds
                ty = dr_ds * np.sin(theta_local) + r_local * np.cos(theta_local) * dtheta_ds

                t_norm = np.sqrt(tx**2 + ty**2)
                tx /= t_norm
                ty /= t_norm

                nx_blade, ny_blade = -ty, tx

                for t in t_values:
                    xw = x_spine + t * nx_blade
                    yw = y_spine + t * ny_blade
                    pos_wheel_list.append([xw, yw])

                    if t == 0:
                        normals_wheel_list.append([nx_blade, ny_blade])
                    else:
                        normals_wheel_list.append([nx_blade * np.sign(t), ny_blade * np.sign(t)])

        p_wheel = np.array(pos_wheel_list)
        wheel_normals_init = np.array(normals_wheel_list)

        pos = np.vstack([pos, p_wheel])
        types = np.concatenate([types, np.full(len(p_wheel), WHEEL)])
        vel = np.vstack([vel, np.zeros((len(p_wheel), 2))])        

        # --- B. Enceinte Circulaire Extérieure (Carter) ---
        pos_wall_list = []
        normals_wall_list = []

        r_wheel_max = r_out
        r_wall_start = r_wheel_max + 3.0*dx
        n_layers_wall = 5
        x_left_channel = jet_x_min - dx
        x_right_channel = jet_x_max + dx

        for layer in range(n_layers_wall):
            r_layer = r_wall_start + (layer * dx)
            perimeter = 2.0 * np.pi * r_layer
            n_points_layer = int(np.round(perimeter / dx))
            d_theta = (2.0 * np.pi) / n_points_layer 

            for i in range(n_points_layer):
                theta = i * d_theta
                # Conversion en degrés pour la condition de coupe (normalisé entre 0 et 360)
                theta_deg = np.degrees(theta) % 360.0                # Coupe radiale entre 100 et 110 degrés
                if 130.0 <= theta_deg <= 160.0 or 250.0 <= theta_deg <= 290.0  or 60.0 <= theta_deg <= 95.0:
                    continue
                    
                xw = wheel_centerX + r_layer * np.cos(theta)
                yw = wheel_centerY + r_layer * np.sin(theta)

#                 if yw > wheel_centerY and (x_left_channel + 2.5* dx < xw < x_right_channel + 4.0*dx):
#                     continue

                pos_wall_list.append([xw, yw])
                normals_wall_list.append([-np.cos(theta), -np.sin(theta)])

        # --- D. Fusion des structures fixes (WALL) ---
        all_wall_pos = []
        all_wall_normals = []
        if pos_wall_list:
            all_wall_pos.append(np.array(pos_wall_list))
            all_wall_normals.append(np.array(normals_wall_list))

        if all_wall_pos:
            p_wall_ext = np.vstack(all_wall_pos)
            wall_ext_normals = np.vstack(all_wall_normals)

            pos = np.vstack([pos, p_wall_ext])
            types = np.concatenate([types, np.full(len(p_wall_ext), WALL)])
            vel = np.vstack([vel, np.zeros((len(p_wall_ext), 2))])

    # --- E. Couches de fluide et de buffers initiales Inclinées ---
    fluid_pos = []
    fluid_vel = []
    fluid_types = []

    y_limit_top = y_center + impact_gap/2

    theta_degRAD = - theta_deg1 
    theta_rad = np.radians(theta_degRAD)

    # Calcul des vitesses directrices inclinées
    v_top_incline = [-v_jet * np.sin(theta_rad), -v_jet * np.cos(theta_rad)]
    v_bot_incline = [-v_jet * np.sin(theta_rad), v_jet * np.cos(theta_rad)]
    v_mid_incline = [v_jet * np.sin(theta_rad), -v_jet * np.cos(theta_rad)]

    N_layers_buffer = 4 

    # 1. INITIALISATION DU TOP JET (Fluide + Buffer)
    for k in range(-N_layers_buffer + 1, N_layers + 1):
        y_off = k * dx
        y_val_top = y_inlet_top - y_off
        
        if y_val_top >= y_inlet_top - 1e-9:
            current_type = BUFFER_TOP
        elif y_val_top > y_limit_top + 1e-9:
            current_type = FLUID
        else:
            continue

        # Application obligatoire de la pente géométrique sur les positions initiales
        dx_pente = (y_val_top - y_inlet_top) * np.tan(theta_rad)
        bx_incline = bx + dx_pente

        layer = np.array([[x, y_val_top] for x in bx_incline])
        if randompos and current_type == FLUID:
            layer = add_position_noise(layer, randomlength * dx)
            
        fluid_pos.append(layer)
        fluid_vel.append(np.tile(v_top_incline, (len(bx_incline), 1)))
        fluid_types.append(np.full(len(bx_incline), current_type))

    # 2. INITIALISATION DU BOTTOM JET (Fluide + Buffer)
    if DOUBLE_JET:
        y_limit_bottom = y_center - impact_gap/2
        N_layers_half = max(1, int(N_layers / 2))
        for k in range(-N_layers_buffer + 1, N_layers_half+ 1):
            y_off = k * dx
            y_val_bot = y_inlet_bottom + y_off 
            
            if y_val_bot <= y_inlet_bottom + 1e-9:
                current_type = BUFFER_BOTTOM
            elif y_val_bot < y_limit_bottom - 1e-9:
                current_type = FLUID
            else:
                continue

            # Application de la pente géométrique inversée (le flux monte)
            dx_pente = (y_inlet_bottom - y_val_bot) * np.tan(theta_rad)
            bx_incline = bx + decalbx + dx_pente

            layer = np.array([[x, y_val_bot] for x in bx_incline])
            if randompos and current_type == FLUID:
                layer = add_position_noise(layer, randomlength * dx)
                
            fluid_pos.append(layer)
            fluid_vel.append(np.tile(v_bot_incline, (len(bx_incline), 1)))
            fluid_types.append(np.full(len(bx_incline), current_type))
    # 2. INITIALISATION DU MID JET (Fluide + Buffer)
    if TRIPLE_JET:
        y_limit_middle = y_center - impact_gap/2
        
        # Réduction stricte de la longueur du jet de moitié
        N_layers_half = max(1, int(N_layers / 2))
        
        # La boucle commence au buffer (k négatif) et descend vers le fluide (k positif)
        # On part de y_inlet_middle pour éviter tout décalage spatial externe
        for k in range(-N_layers_buffer + 1, N_layers_half + 1):
            y_off = k * dx
            y_val_mid = y_inlet_middle - y_off  # Correction : basé sur y_inlet_middle
            
            # Séparation stricte et continue (inclusion du cas k=0 dans le FLUID)
            if y_val_mid > y_inlet_middle + 1e-9:
                current_type = BUFFER_MIDDLE
            elif y_inlet_middle + 1e-9 >= y_val_mid > y_limit_middle + 1e-9: # Changement de signe ici (+1e-9)
                current_type = FLUID
            else:
                continue # Stoppe la création si on dépasse la limite d'impact

            # Application de la pente géométrique inversée (le flux monte)
            dx_pente = (y_inlet_middle - y_val_mid) * np.tan(theta_rad)
            bx_incline = bx + decalbx1 + dx_pente

            layer = np.array([[x, y_val_mid] for x in bx_incline])
            if randompos and current_type == FLUID:
                layer = add_position_noise(layer, randomlength * dx)
                
            fluid_pos.append(layer)
            fluid_vel.append(np.tile(v_mid_incline, (len(bx_incline), 1)))
            fluid_types.append(np.full(len(bx_incline), current_type))

    # Assemblage final
    if fluid_pos:
        pos = np.vstack([pos, np.vstack(fluid_pos)])
        vel = np.vstack([vel, np.vstack(fluid_vel)])
        types = np.concatenate([types, np.concatenate(fluid_types)])

    # --- F. Assignation des normales globales ---
    wall_normals = np.zeros((len(pos), 2))

    mask_wheel = (types == WHEEL)
    if 'wheel_normals_init' in locals():
        wall_normals[mask_wheel] = wheel_normals_init

    mask_wall = (types == WALL)
    if all_wall_normals:
        wall_normals[mask_wall] = np.vstack(all_wall_normals)       

    t, step, dt = 0.0, 0, 0.01 * h / c0

    n_init = len(pos)
    rho = np.full(n_init, rho0)
    pres = np.zeros(n_init)
    vol = np.full(n_init, dx**2)
    m = rho * vol
    shifting_displacement = np.zeros((n_init, 2))
    num_neighbors = np.zeros(n_init, dtype=int)
    shepard_coeff = np.zeros(n_init)
    v_mesh = np.copy(vel)
    print("\n OK Géométrie et étanchéité du carter validées (Cas Mill)")
#--------------------------------------------------------------------------------------------------------------------- 
#--------------------------------------------------------------------------------------------------------------------- 
#--------------------------------------------------------------------------------------------------------------------- 
# temporal iterations
#--------------------------------------------------------------------------------------------------------------------- 
#--------------------------------------------------------------------------------------------------------------------- 
#--------------------------------------------------------------------------------------------------------------------- 
if ModeDEBUG: 
    
    print("Begin mode debug:")
    # =====================================
    # Génération nuage particulaire
    # =====================================
    def generate_particles(N):
        dx_grid = 1.0 / N
        v = np.linspace(dx_grid / 2, 1.0 - dx_grid / 2, N)
        X, Y = np.meshgrid(v, v)

        # Ajout d'un bruit (jitter) pour casser la symétrie parfaite
        np.random.seed(42) # Pour garder le test reproductible
        noise = (np.random.rand(*X.shape) - 0.5) * 0.2 * dx_grid
        X += noise
        Y += noise

        pos = np.column_stack((X.ravel(), Y.ravel()))
        vol = np.ones(len(pos)) * (dx_grid ** 2)
        h = 2.0 *  dx_grid 
        tree = cKDTree(pos)

        return pos, vol, h, tree

    # =====================================
    # Champs tests
    # =====================================
    def field_constant(pos):
        return np.ones(len(pos))

    def field_linear(pos):
        return 2.0 + 3.0*pos[:,0] - 4.0*pos[:,1]

    def grad_linear_exact(pos):
        g = np.zeros((len(pos),2))
        g[:,0]=3.0
        g[:,1]=-4.0
        return g

    def field_quadratic(pos):
        return 100.0*(pos[:,0]**2 + pos[:,1]**2)

    def grad_quadratic_exact(pos):
        g=np.zeros((len(pos),2))
        g[:,0]=200.0*pos[:,0]
        g[:,1]=200.0*pos[:,1]
        return g
        
    def hess_quadratic_exact(pos):
        n_p = len(pos)
        # On veut (n_p, 3) pour correspondre à [hxx, hxy, hyy]
        H_flat = np.zeros((n_p, 3))
        
        # Pour f = 100 * (x^2 + y^2)
        # d2f/dx2 = 200, d2f/dxdy = 0, d2f/dy2 = 200
        H_flat[:, 0] = 200.0 # hxx
        H_flat[:, 1] = 0.0   # hxy
        H_flat[:, 2] = 200.0 # hyy
        return H_flat
       
    def field_sinus(pos):
        """
        f(x,y) = 100 * sin(pi*x) * cos(pi*y)
        """
        return 100.0 * np.sin( np.pi * pos[:, 0]) * np.cos( np.pi * pos[:, 1])

    def grad_sinus_exact(pos):
        """
        df/dx =  200*pi * cos(2*pi*x) * cos(2*pi*y)
        df/dy = -200*pi * sin(2*pi*x) * sin(2*pi*y)
        """
        g = np.zeros((len(pos), 2))
        
        # Gradient en x
        g[:, 0] = 100.0 * np.pi * np.cos( np.pi * pos[:, 0]) * np.cos( np.pi * pos[:, 1])
        
        # Gradient en y
        g[:, 1] = -100.0 * np.pi * np.sin(np.pi * pos[:, 0]) * np.sin( np.pi * pos[:, 1])
        
        return g

    # =====================================
    # Erreur L2
    # =====================================
    # =====================================
    # Erreur L2 (Compatible Scalaires et Vecteurs)
    # =====================================
    def L2_error_interior(num, exact, pos, vol):
        """
        Calcule l'erreur L2 normalisée.
        Gère automatiquement :
        - Scalaires (N)
        - Vecteurs (N, dim)
        - Tenseurs/Hessiennes (N, dim, dim)
        """
        interior = (pos[:, 0] > 0.2) & (pos[:, 0] < 0.8) & \
                   (pos[:, 1] > 0.2) & (pos[:, 1] < 0.8)
        
        if not np.any(interior):
            interior = np.ones(len(pos), dtype=bool)
            
        diff = num[interior] - exact[interior]
        
        # On calcule le carré de la différence
        diff_sq = diff**2
        
        # On somme sur toutes les dimensions, peu importe le rang du tenseur
        # si diff est (N, ...), on somme sur les axes après l'axe 0
        if diff.ndim > 1:
            # axes = (1, 2, ...) selon la dimension
            axes = tuple(range(1, diff.ndim))
            sq_diff = np.sum(diff_sq, axis=axes)
        else:
            sq_diff = diff_sq
        
        v_int = vol[interior]
        # Calcul de la norme L2 : sqrt( somme( (err^2) * Vol ) / somme(Vol) )
        l2_error = np.sqrt(np.sum(sq_diff * v_int) / np.sum(v_int))
        
        return l2_error
        
    def compute_mls_interpolation_order2(pos, vol, field, h, tree, p_ref, cond_MLS, types):
        """
        Calcule la valeur reconstruite (interpolation) d'un champ via Moving Least Squares d'ordre 2.
        Permet de vérifier l'ordre 3 de la convergence sur la fonction elle-même.
        """
        n_p = len(pos)
        dim = 2          
        n_basis = 6      
        
        f_is_scalar = (field.ndim == 1)
        f = field[:, None] if f_is_scalar else field
        n_components = f.shape[1]
        
        # Tableau pour stocker les valeurs interpolées au lieu des gradients
        interpolations = np.zeros((n_p, n_components))
        
        # Rayon dynamique (ajustez selon votre logique globale, ex: 2.5 * h)
        search_radius = 2.5 * h
        all_indices = tree.query_ball_point(pos, search_radius)
        
        for i in range(n_p):
            # Remplacer FLUID par votre constante / variable globale
            if types[i] != 1: 
                interpolations[i] = f[i] # Fallback
                continue
                
            idx = all_indices[i]
            
            if len(idx) < n_basis:
                interpolations[i] = f[i] # Fallback
                continue
                
            # Périodicité (adapter selon votre code exact)
            rij = (pos[idx] - pos[i])
                
            r = np.linalg.norm(rij, axis=1)
            valid = r > 2.2e-16
            
            if np.sum(valid) < n_basis:
                interpolations[i] = f[i] # Fallback
                continue
                
            idx = np.array(idx)[valid]
            rij = rij[valid]
            r = r[valid]
            
            rij_scaled = rij / h
            x_s = rij_scaled[:, 0]
            y_s = rij_scaled[:, 1]
            
            P = np.column_stack([
                np.ones_like(x_s),
                x_s, y_s,
                x_s**2, x_s*y_s, y_s**2
            ])  
            
            # Supposant kernel_cubic défini globalement dans votre script
            # W = kernel_cubic(r, h)
            # Ici, un exemple générique pour que ça compile si absent :
            q = r / h
            W = np.where(q < 1, 1 - 1.5*q**2 + 0.75*q**3, 0.25*(2-q)**3)
            
            weight = W * vol[idx]  
            
            M = P.T @ (weight[:, None] * P)  
            M += 2.2e-16 * np.eye(n_basis)
            
            df = f[idx] - f[i]  
            rhs = df.T @ (weight[:, None] * P)
             
            cond_MLS[i] = np.linalg.cond(M)                               
            if cond_MLS[i] < 10000.0:
                inv_M = np.linalg.pinv(M)
                coeffs = rhs @ inv_M
                
                # --- LA MODIFICATION EST ICI ---
                # coeffs[:, 0] est la correction MLS à l'origine locale.
                # La valeur absolue reconstruite est f[i] + cette correction.
                interpolations[i] = f[i] + coeffs[:, 0]
                p_ref[i] = 2             
            else:  
                interpolations[i] = f[i]
                p_ref[i] = 0
                        
        return interpolations[:, 0] if f_is_scalar else interpolations

    # =====================================
    # Test d'ordre MLS
    # =====================================
    def test_mls_order():

        resolutions=[40,80,160]
        global types
        errors_const=[]
        errors_lin=[]
        errors_quad=[]
        errors_sinus=[]
        errors_const2=[]
        errors_lin2=[]
        errors_quad2=[]
        errors_sinus2=[]
        error_interp_sinus=[]
        dx=[]
        p_ref=[]
        cond_MLS=[]
        for N in resolutions:

            pos,vol,h,tree=generate_particles(N)
            neighbor_lists = tree.query_ball_point(pos, r=2.0*h)
            
            dx.append(1.0/N)
            p_ref = np.zeros(len(pos))
            cond_MLS = np.zeros(len(pos))
            types = np.ones(len(pos)) * FLUID
            # ---- constant ----
            f=field_constant(pos)
            grad=compute_mls_gradients(pos,vol,f,h,neighbor_lists,p_ref,cond_MLS,types, testCase, 1, False)
            errors_const.append(np.linalg.norm(grad))

            # ---- linear ----
            f=field_linear(pos)
            grad=compute_mls_gradients(pos,vol,f,h,neighbor_lists,p_ref,cond_MLS,types, testCase, 1, False)
            exact=grad_linear_exact(pos)
            errors_lin.append(L2_error_interior(grad,exact, pos, vol))

            # ---- quadratic ----
            f=field_quadratic(pos)
            grad2=compute_mls_gradients(pos,vol,f,h,neighbor_lists,p_ref,cond_MLS,types, testCase, 1, False)
            exact2=grad_quadratic_exact(pos)
            errors_quad.append(L2_error_interior(grad2,exact2, pos, vol))

            #Sinus
            f=field_sinus(pos)
            grad2=compute_mls_gradients(pos,vol,f,h,neighbor_lists,p_ref,cond_MLS,types, testCase, 1, False)
            exact2=grad_sinus_exact(pos)
            errors_sinus.append(L2_error_interior(grad2,exact2, pos, vol))
            
            # --- CALCULS MLS2 UNIFIÉS ---
            
            # 1. Constant
            f = field_constant(pos)
            res_const = compute_mls_gradients_order2(pos, vol, f, h, tree, p_ref, cond_MLS, types)
            # Le gradient est dans les colonnes 0,1. On calcule la norme de ce gradient.
            errors_const2.append(np.linalg.norm(res_const[:, 0:2]))
            
            # 2. Linear
            f = field_linear(pos)
            res_lin = compute_mls_gradients_order2(pos, vol, f, h, tree, p_ref, cond_MLS, types)
            # On compare seulement le gradient (colonnes 0:2)
            errors_lin2.append(L2_error_interior(res_lin[:, 0:2], grad_linear_exact(pos), pos, vol))
            
            # 3. Sinus
            f = field_sinus(pos)
            res_sin = compute_mls_gradients_order2(pos, vol, f, h, tree, p_ref, cond_MLS, types)
            errors_sinus2.append(L2_error_interior(res_sin[:, 0:2], grad_sinus_exact(pos), pos, vol))
            
            # 4. Quadratic (Gradient + Hessienne)
            f_quad = field_quadratic(pos)
            res_quad = compute_mls_gradients_order2(pos, vol, f_quad, h, tree, p_ref, cond_MLS, types)
            
            # Erreur Gradient (colonnes 0:2)
            errors_quad2.append(L2_error_interior(res_quad[:, 0:2], grad_quadratic_exact(pos), pos, vol))
            
            # Erreur Hessienne (colonnes 2:5)
            # Rappel : ton tableau res_quad[:, 2:5] contient [hxx, hxy, hyy]
            error_hess2 = L2_error_interior(res_quad[:, 2:5], hess_quadratic_exact(pos), pos, vol)
            print(f"Erreur L2 Hessienne pour N={N}: {error_hess2}")
                       
           
           
            
        dx=np.array(dx)
        print("MLS1 :")
        print("Valeurs brutes errors_lin :", errors_lin)
        print("Valeurs brutes errors_quad :", errors_quad)
        print("Valeurs brutes errors_sinus :", errors_sinus)
        order_lin=np.polyfit(np.log(dx),np.log(errors_lin),1)[0]        
        order_quad=np.polyfit(np.log(dx),np.log(errors_quad),1)[0]
        order_sinus=np.polyfit(np.log(dx),np.log(errors_sinus),1)[0]
        
        print("MLS1 constant error:",  errors_const)
        print("MLS1 linear order:",    order_lin)
        print("MLS1 quadratic order:",order_quad)
        print("MLS1 sinus order:",order_sinus)
        
        print("MLS2 :")        
        print("Valeurs brutes errors_lin :", errors_lin2)
        print("Valeurs brutes errors_quad :", errors_quad2)
        print("Valeurs brutes errors_sinus :", errors_sinus2)
        order_lin2=np.polyfit(np.log(dx),np.log(errors_lin2),1)[0]        
        order_quad2=np.polyfit(np.log(dx),np.log(errors_quad2),1)[0]
        order_sinus2=np.polyfit(np.log(dx),np.log(errors_sinus2),1)[0]
        
        print("MLS2 constant error:",  errors_const2)
        print("MLS2 linear order:",    order_lin2)
        print("MLS2 quadratic order:",order_quad2)
        print("MLS2 sinus order:",order_sinus2)
        
        
        
        

    def test_renormalization_order():
        resolutions = [20, 40, 80]

        errors_const = []
        errors_lin = []
        errors_quad = []
        dx = []

        for N in resolutions:
            pos, vol, h, _ = generate_particles(N)
            dx.append(1/N)
            num_particles = len(pos)

            # 1. Requête de voisinage au format Batch (Matrice / Listes de listes

            # Pré-calcul des matrices de renormalisation L
            # (Assurez-vous que cette fonction accepte all_indices ou adaptez la si elle requiert des pairs)
            
            L = np.zeros((num_particles, 2, 2))

            # NOTE : Si compute_renormalization_matrices_robust requiert absolument des 'pairs', 
            # gardez la création de pairs UNIQUEMENT pour elle, mais pas pour le gradient.
            
            # Extraction des indices de voisinage à l'aide de l'arbre
            tree = cKDTree(pos)
            all_indices = tree.query_ball_point(pos, 2.0 * h)

            shepard = np.ones(num_particles)

            # Appel de la fonction modifiée
            L, cond, status = compute_renormalization_matrices_robust(
                pos, vol, h, all_indices, shepard, testCase, Periodic, L_domain=1.0  # Ajustez L_domain selon votre code
            )
            # =====================================================================
            # FONCTION VECTORISÉE : Plus de boucle 'for i, j in pairs' !
            # =====================================================================
            def compute_raw_sph_gradient_vectorized(field):
                grad_sph = np.zeros((num_particles, 2))

                for i in range(num_particles):
                    idx = all_indices[i]
                    if len(idx) <= 1: 
                        continue

                    # Extraction des vecteurs de distance
                    rij = pos[idx] - pos[i] # Forme (n_voisins, 2)
                    r = np.linalg.norm(rij, axis=1)

                    # Masque de sécurité
                    valid = (r > 2.2e-16) & (r <= 2.0 * h)
                    if not np.any(valid): 
                        continue

                    rij = rij[valid]
                    r = r[valid]
                    idx_valid = np.array(idx)[valid]

                    # Calcul de TOUS les gradients du noyau d'un coup pour la particule i
                    # Note : Adaptez si votre kernel_grad n'accepte pas les tableaux d'un coup
                    g_w, _ = kernel_grad(rij, r, h) # Forme (n_voisins_valides, 2)

                    # Formule SPH : Somme_j ( V_j * (f_j - f_i) * gradW_ij )
                    df = field[idx_valid] - field[i] # Forme (n_voisins_valides,)

                    # Produit terme à terme puis somme sur l'axe des voisins
                    # On multiplie par le volume des voisins j
                    term = vol[idx_valid, None] * df[:, None] * g_w
                    grad_sph[i] = np.sum(term, axis=0)

                return grad_sph

            # =====================================================================
            # Évaluations (Maintenant instantanées)
            # =====================================================================

            # ---- Constant ----
            f = field_constant(pos)
            grad_raw = compute_raw_sph_gradient_vectorized(f)
            grad_corrected = np.einsum("nij,nj->ni", L, grad_raw)
            errors_const.append(np.linalg.norm(grad_corrected))

            # ---- Linear ----
            f = field_linear(pos)
            grad_exact = grad_linear_exact(pos)
            grad_raw = compute_raw_sph_gradient_vectorized(f)
            grad_corrected = np.einsum("nij,nj->ni", L, grad_raw)
            errors_lin.append(L2_error_interior(grad_corrected, grad_exact, pos, vol))

            # ---- Quadratic ----
            f = field_quadratic(pos)
            grad_exact = grad_quadratic_exact(pos)
            grad_raw = compute_raw_sph_gradient_vectorized(f)
            grad_corrected = np.einsum("nij,nj->ni", L, grad_raw)
            errors_quad.append(L2_error_interior(grad_corrected, grad_exact, pos, vol))

        dx = np.array(dx)
        order_lin = np.polyfit(np.log(dx), np.log(errors_lin), 1)[0]
        order_quad = np.polyfit(np.log(dx), np.log(errors_quad), 1)[0]

        print("--- Résultats Renormalisation Corrigés ---")
        print("Valeurs brutes errors_lin :", errors_lin)
        print("Valeurs brutes errors_quad :", errors_quad)
        print("Renorm linear order:", order_lin)
        print("Renorm quadratic order:", order_quad)
    
    # ACTIVER LES TESTS:

    run_unit_tests()
    test_kernel_grad_uniform_flow()
    test_shifting_with_metrics()
    test_mls_order()
    test_renormalization_order()
    print("end mode debug:")

if ModeGEOM == True :
    n_current = len(pos)
    
    # REDIMENSIONNER LES ARRAYS SI NÉCESSAIRE
    if len(v_mesh) != n_current:
        v_mesh = np.zeros((n_current, 2))
        print(f"⚠️ v_mesh redimensionné : {len(v_mesh)} → {n_current}")
    

    
    # MAINTENANT appeler write_vtk
    write_vtk(os.path.join(vtk_dir, f"testcaseGEO{step:05d}.vtk"), 
              pos, vel, v_mesh-vel, pres, types, rho,
              vol, m, np.zeros(n_current), np.zeros(n_current),
              wall_normals, np.zeros(n_current), np.zeros(n_current), 
              np.zeros((n_current,2)), np.zeros((n_current,2)),
              np.zeros(n_current), np.zeros(n_current), 
              np.zeros((n_current,2)), np.zeros((n_current, 2, 2)),
              np.zeros((n_current,2)), np.zeros(n_current), np.zeros(n_current))  
    

# --- INITIALISATION DYNAMIQUE DU CAS MILL ---
if testCase == 'Mill':
    # Identification des indices des particules composant la roue
    idx_wheel_start = np.where(types == WHEEL)[0][0]
    n_wheel = np.sum(types == WHEEL)
    
    # Variables d'état cinématiques
    wheel_omega = 0.0   # Vitesse angulaire initiale (au repos)
    wheel_theta = 0.0   # Angle d'orientation initial
    
    # Propriétés de masse et d'inertie de la roue
    # À ajuster selon la densité solide réelle visée
    wheel_mass = np.sum(m[types == WHEEL])
    
    # Calcul du moment d'inertie de masse standard J = ∑ m_i * r_i²
    rel_pos_init = pos[types == WHEEL] - np.array([wheel_centerX, wheel_centerY])
    wheel_I = np.sum(m[types == WHEEL] * (rel_pos_init[:, 0]**2 + rel_pos_init[:, 1]**2))
# ------------------------------------------------------------------
# ---------------------TIMER START ------------------------------------
# ------------------------------------------------------------------
t_relax = 0
t_ramping = 0.2 * t_relaxmax  
t_start_ramp = t_relaxmax - t_ramping

if Relax : print(f"⚠️ Mesh RELAXATION  ⚠️: ")
    
while Relax and t_relax < t_relaxmax and ModeDEBUG == False and ModeGEOM == False :   
    
    n_current = len(pos) # La taille peut changer à chaque itération 
    tree = cKDTree(pos)    
    # ★ OPTIMISATION : On extrait la liste des voisins UNE SEULE FOIS pour tout le pas de temps ★
    neighbor_lists = tree.query_ball_point(pos, r=search_radius)
    pairs = list(tree.query_pairs(search_radius)) # Gardé si vos autres fonctions SPH utilisent 'pairs'    
    
                 
    mask_f = (types == FLUID)  
    
    solid_types = [WALL]
    if testCase == 'Mill':
        solid_types.append(WHEEL)
    if testCase == 'Car' :
        solid_types.append(CAR)
    if testCase =='BodyF' :
        solid_types.append(BODY)
        solid_types.append(FLAP)
    mask_wall = np.isin(types, solid_types)
    # Ré-initialisation à la taille courante   
    v_mesh = np.zeros((n_current,2))
    accel = np.zeros((n_current,2))
    drho = np.zeros(n_current)
    num_neighbors.fill(0)
    shepard_coeff.fill(0)   
    
    
    # --- PHASE  : CALCUL DU CHAMP DE SUPPORT (SHEPARD) --- 
    # On fait un premier passage uniquement pour le voisinage et le shepard   
    # 2. On applique le calcul uniquement sur ces indices
    shepard_coeff[mask_f] = kernel_cubic(0, h) * vol[mask_f]                 
    compute_shepard_coefficients(pairs, pos, vol, h,
                                    shepard_coeff,
                                    num_neighbors,
                                    kernel_cubic)
    # --------------------Filtrage passe bas ---------------------------------------    
    #application du filtre qu'au démarrage
    if Filtre and (step % stepfiltre == 0 ):  #application du filtre sous condition d'activitation et tous les stepfiltre      
        rho=apply_density_filter(pos, m, rho, vol, types, h,tree,shepard_coeff )
        pres = get_pressure(rho)
            
    if np.any(mask_wall):
        # ----- Compute pressure wall or bodyF or car ----     
        compute_pressure_wall(pairs, pos, vol, h, kernel_cubic, pres, types, g, rho0,testCase)    
  
    #---------------------------------------------------------------------------------------------------------------------  
    # Paramètres de contrôle du Shifting Formulation type PST (Particle Shifting Technique)
    v_mesh, S_G_local, S_dist, grad_S_G, grad_S_dist = compute_v_mesh_full(pos, vel, vol, h, dt, 
                                                                                  mesh_mode,shepard_coeff,tree,pairs )
    #---------------------------------------------------------------------------------------------------------------------  
    drho, accel =compute_derivatives(        
                                        pos, vel, rho, pres,
                                        vol, m,
                                        accel, drho,
                                        types,
                                        h, dt,
                                        shepard_coeff,
                                        wall_normals,
                                        v_mesh, tree,pairs,np.zeros((n_current, 2, 2)), 
        np.zeros((n_current,2)), np.zeros((n_current,2)), np.zeros((n_current,2)) )           
    #---------------------------------------------------------------------------------------------------------------------     
    # Intégration 
    if np.any(mask_f):           
             
        rho[mask_f]  += drho[mask_f] * dt
        vol[mask_f]  =  m[mask_f] / rho[mask_f]
            
        vel[mask_f]  += (accel[mask_f] + g) * dt
            
            # --- CALCUL DU COEFFICIENT D'AMORTISSEMENT DYNAMIQUE ---
        if t_relax < t_start_ramp:
                # Amortissement nominal au début
            current_relax = relax_coefficient
        else:
                # Décroissance fluide de relax_coefficient vers 0 (Ramping linéaire)
            tau = (t_relax - t_start_ramp) / t_ramping
            current_relax = relax_coefficient * (1.0 - tau)

            # Application de l'amortissement variable
        vel[mask_f] -= current_relax * vel[mask_f] * dt
            
            
        pres[mask_f] =  get_pressure(rho[mask_f])
        v_mesh[mask_f] = vel[mask_f]
        pos[mask_f]  += v_mesh[mask_f]* dt
        
    # Sortie & Update Time acoustic constrained
    fluid_vel = vel[mask_f]
    fluid_pres = pres[mask_f]

    max_v = np.max(np.linalg.norm(fluid_vel, axis=1)) if fluid_vel.size else 0
    max_p = np.max(np.abs(fluid_pres)) if fluid_pres.size else 0
    #if viscosity on 
    max_set = max(c0, max_v, np.sqrt(max_p / rho0))
    dt = CFL * h / max_set

    t_relax += dt; step += 1  
    if step % save == 0: 
        write_vtk(os.path.join(vtk_dir, f"testcaseRELAX{step:05d}.vtk"), pos, vel,v_mesh-vel , pres, types, rho,
                  vol, m, num_neighbors, shepard_coeff,wall_normals, S_G_local, S_dist, grad_S_G, grad_S_dist,
                  cond_array, renorm_status, np.zeros((n_current,2)), np.zeros((n_current,2,2)),np.zeros((n_current,2)),np.zeros(n_current),
                  np.zeros(n_current))
        print(f"Step RELAX {step} | t relax = {t_relax:.4f}s | dt = {dt:.6e} | nombre de particules = {n_fluid:.6e}")
       
 
#-------------------------------- END RELAXATION ----------------------------------

#--------------------- real time integration -------------------------------------

t=0.0
step=0
Relax = False
print(f"⚠️ START SIMULATION   ⚠️: ")
while t < finaltime and ModeDEBUG == False and ModeGEOM == False :   
        
    if testCase == 'bodyF':
        # ------------------------------------------------------------------
        # 8a. Mise à jour des frontières mobiles
        # ------------------------------------------------------------------

        # Clapet wavemaker : positions et vitesses prescrites
        update_flap_particles(pos, vel, wall_normals, t,
                              idx_flap_start, n_flap, n_layers, y0_arr, dx)
     
    
    if testCase == 'jet' or testCase == 'Mill':
        pos, vel, types, rho, pres, vol, m, num_neighbors, shepard_coeff = update_jet_buffers(
        testCase, pos, vel, types, rho, pres, vol, m,
        num_neighbors, shepard_coeff,bx, dx,
        y_inlet_top, y_inlet_bottom,y_inlet_middle,
        v_jet, rho0,
        BUFFER_TOP, BUFFER_BOTTOM,BUFFER_MIDDLE, FLUID,
        DOUBLE_JET, TRIPLE_JET,
        randompos,
        add_position_noise, - theta_deg1,decalbx,    decalbx1
        )                     
   #---------------------------------------------------------------------------------------------------------------------  
    n_current = len(pos) # La taille peut changer à chaque itération 
    # Pour Poiseuille : replier X AVANT construction de l'arbre
    if Periodic and testCase == 'Poiseuille':
        pos[mask_f, 0] = pos[mask_f, 0] % L   # repliement fluides seulement

    if Periodic:
        if testCase in ['TGV', 'TGVmesh','AirFoil']:
            tree = cKDTree(pos, boxsize=L)
        elif testCase == 'Poiseuille':
        # Arbre standard : coordonnées WALL en Y négatif => boxsize impossible
        # periodic_rij() corrige les vecteurs rij en X lors des calculs SPH
            tree = cKDTree(pos, boxsize=(L, None))
    else:
        tree = cKDTree(pos)
    
    # ★ OPTIMISATION : On extrait la liste des voisins UNE SEULE FOIS pour tout le pas de temps ★
    neighbor_lists = tree.query_ball_point(pos, r=search_radius)
    pairs = list(tree.query_pairs(search_radius)) # Gardé si vos autres fonctions SPH utilisent 'pairs'
    
    
      
     
                
    mask_f = (types == FLUID)  
    solid_types = [WALL]
    if testCase == 'Mill':
        solid_types.append(WHEEL)
    if testCase == 'Car' :
        solid_types.append(CAR)
    if testCase =='BodyF' :
        solid_types.append(BODY)
        solid_types.append(FLAP)
    mask_wall = np.isin(types, solid_types)
    # Ré-initialisation à la taille courante   
    v_mesh = np.zeros((n_current,2))
    accel = np.zeros((n_current,2))
    drho = np.zeros(n_current)
    num_neighbors.fill(0)
    shepard_coeff.fill(0)
    L_matrices =np.zeros((n_current, 2, 2)) 
    cond_array= np.zeros(n_current)
    renorm_status= np.zeros(n_current)
    p_ref= np.zeros(n_current)
    cond_MLS= np.zeros(n_current)
    
    
    
    # --- PHASE  : CALCUL DU CHAMP DE SUPPORT (SHEPARD) --- 
    # On fait un premier passage uniquement pour le voisinage et le shepard   
    # 2. On applique le calcul uniquement sur ces indices
    shepard_coeff[mask_f] = kernel_cubic(0, h) * vol[mask_f]                 
    compute_shepard_coefficients(pairs, pos, vol, h,
                                    shepard_coeff,
                                    num_neighbors,
                                    kernel_cubic)
    # --------------------Filtrage passe bas ---------------------------------------
    if Filtre and (step % stepfiltre == 0 ):  #application du filtre sous condition d'activitation et tous les stepfiltre
        rho=apply_density_filter(pos, m, rho, vol, types, h,tree,shepard_coeff )
        pres = get_pressure(rho)
            
    if RenormON:            
            L_matrices, cond_array, renorm_status = compute_renormalization_matrices_robust(pos, vol, h, neighbor_lists,
                                                                                            shepard_coeff,testCase,
                                                                                            Periodic, L)          
    if np.any(mask_wall):
        # ----- Compute pressure wall or bodyF or car ----     
        compute_pressure_wall(pairs, pos, vol, h, kernel_cubic, pres, types, g, rho0,testCase)
    
   #---- Functional computation- ALE Mesh test validation (only for TGV to begin)
#     if testCase=='TGV' or testCase=='TGVmesh'  :
#         # Calcul de la fonctionnelle AVANT déplacement
#         F1_before, FS_before, _, _, _ = compute_energy_functionals(
#             pos[mask_f], vol[mask_f], h, shepard_coeff[mask_f], pairs
#         )
#         F2_before = F1_before + alpha * FS_before  
    
#------------------------------------Euler RK1---------------------------------------------------------------------      
    if mode_RK == 1 :  
        if mode_p_refinement == 0:
             grad_p = np.zeros((n_current,2))
             grad_vel = np.zeros((n_current, 2, 2))
             grad_rho = np.zeros((n_current,2))
             p_ref=np.zeros(n_current)
             cond_MLS=np.zeros(n_current)
        #---------------------------------------------------------------------------------------------------------------------  
        # Paramètres de contrôle du Shifting Formulation type PST (Particle Shifting Technique)
        v_mesh, S_G_local, S_dist, grad_S_G, grad_S_dist = compute_v_mesh_full(pos, vel, vol, h, dt, 
                                                                                  mesh_mode,shepard_coeff,tree,pairs )
        #---------------------------------------------------------------------------------------------------------------------  
        drho, accel =compute_derivatives(        
                                        pos, vel, rho, pres,
                                        vol, m,
                                        accel, drho,
                                        types,
                                        h, dt,
                                        shepard_coeff,
                                        wall_normals,
                                        v_mesh, tree,pairs,L_matrices, grad_p, grad_vel, grad_rho )           
        if testCase == 'bodyF':
            # Cet appel ne lit que les variables liées au corps flottant
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel, m, types, g, dt,t,
                body_vel=body_vel, body_omega=body_omega, body_theta=body_theta, body_cm=body_cm,
                rel_pos0=rel_pos0, nrm_body0_global=nrm_body0_global, mass_body_2D=mass_body_2D, I_2D=I_2D,
                wall_normals=wall_normals, idx_body_start=idx_body_start, n_body=n_body
            )
            
            # Récupération des données mises à jour
            body_vel = rigid_updates['body_vel']
            body_omega = rigid_updates['body_omega']
            body_theta = rigid_updates['body_theta']
            body_cm = rigid_updates['body_cm']
        
        elif testCase == 'car':
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel, m, types, g, dt, t,
                CAR=CAR, car_mass=car_mass, car_I=car_I, car_vel=car_vel, car_omega=car_omega, car_theta=car_theta,
                wall_normals=wall_normals 
            )
            
            car_vel = rigid_updates['car_vel']
            car_omega = rigid_updates['car_omega']
            car_theta = rigid_updates['car_theta']
            
        elif testCase == 'Mill':
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel, m, types, g, dt, t,
                CAR=WHEEL, car_I=wheel_I, car_omega=wheel_omega, car_theta=wheel_theta,
                wall_normals=wall_normals 
            )
            
            wheel_alpha = rigid_updates['wheel_alpha']
            wheel_omega = rigid_updates['wheel_omega']
            wheel_theta = rigid_updates['wheel_theta']
        #---------------------------------------------------------------------------------------------------------------------     
        # Intégration 
        if np.any(mask_f):
            rho[mask_f]  += drho[mask_f] * dt
            vol[mask_f]  =  m[mask_f] / rho[mask_f]
            vel[mask_f]  += (accel[mask_f] + g) * dt
            pres[mask_f] =  get_pressure(rho[mask_f])
            pos[mask_f]  += v_mesh[mask_f]* dt
        # -----------------------------------------------------------------
        # GESTION DYNAMIQUE DE LA PÉRIODICITÉ
        # -----------------------------------------------------------------
        if Periodic:
            if testCase in ['TGV', 'TGVmesh','AirFoil']:
                pos[:, 0] = pos[:, 0] % L
                pos[:, 1] = pos[:, 1] % L
            elif testCase == 'Poiseuille':
                pos[mask_f, 0] = pos[mask_f, 0] % L  # fluides seulement, pas les WALL
                    # Pas de clip Y : les parois SPH assurent le confinement
                    
                    
#         if ShiftPart and np.any(mask_f):
#             if ShiftMethod == 'delta_sph':
#                 grad_C, C = compute_concentration_gradient(
#                     pos, vol, h, tree, pairs, 
#                     Periodic=Periodic, testCase=testCase, L=L
#                 )
#                 shift, shift_status = compute_shifting_displacement_delta_sph(
#                     pos, vol, grad_C, C, h,
#                     CFL=ShiftCFL, R=ShiftR,
#                     types=types, FLUID=FLUID
#                 )
#                 if ModeDEBUG:
#                     diagnose_shifting_statistics(shift, C, grad_C, h, step=step)
            
#             elif ShiftMethod == 'lloyd':
#                 shift, shift_status = compute_shifting_displacement_lloyd(
#                     pos, vol, h, tree, pairs,
#                     beta_lloyd=ShiftBetaLloyd,
#                     types=types, FLUID=FLUID
#                 )
#                 if ModeDEBUG:
#                     shift_mag = np.linalg.norm(shift, axis=1)
#                     print(f"Lloyd shift: mean={np.mean(shift_mag)/h:.3f}h")
            
#             pos = apply_shifting_correction(
#                 pos, shift,
#                 boundary_layer_width=2.0*h, h=h,
#                 types=types, WALL=WALL
#             )
        
     
                
                            
#------------------------------------RK2---------------------------------------------------------------------      
    if mode_RK == 2 :
        accel1 = np.zeros((n_current,2))
        drho1 = np.zeros(n_current)
        accel2 = np.zeros((n_current,2))
        drho2 = np.zeros(n_current)
        v_mesh1 = np.zeros((n_current,2))
        v_mesh2 = np.zeros((n_current,2))
        
        pres_star = np.zeros_like(pres)        
        pos_star = pos.copy()        
        vel_star = vel.copy()        # idem
        rho_star = rho.copy()        # idem
        vol_star = vol.copy()        # idem
        shepard_coeff_star = np.zeros_like( shepard_coeff)
        L_matrices_star =np.zeros((n_current, 2, 2))
              
        grad_p = np.zeros((n_current,2))
        grad_vel = np.zeros((n_current, 2, 2))
        grad_rho = np.zeros((n_current,2))
        
        S_G_local = np.zeros(n_current)
        S_dist    = 0.0
        grad_S_G  = np.zeros((n_current, 2))
        grad_S_dist = np.zeros((n_current, 2))
         
        # -------------------------------- STEP 1: PREDICTOR ---------------------------------------------------------------------------------
        #---------------------------------------------------------------------------------------------------------------------  
        # Paramètres de contrôle du Shifting Formulation type PST (Particle Shifting Technique)
        v_mesh1, S_G_local, S_dist, grad_S_G, grad_S_dist = compute_v_mesh_full(pos, vel, vol, h, dt, 
                                                                                  mesh_mode,shepard_coeff,tree,pairs )
        #---------------------------------------------------------------------------------------------------------------------       
        # Après tree_star, pairs_star :
        if mode_p_refinement == 1:
            if mode_p_refinement == 1:
                grad_rho = compute_mls_gradients(pos, vol, rho,  h, neighbor_lists, p_ref, cond_MLS, types, testCase, L, Periodic)
                grad_vel = compute_mls_gradients(pos, vol, vel,  h, neighbor_lists, p_ref, cond_MLS, types, testCase, L, Periodic)
                grad_p   = compute_mls_gradients(pos, vol, pres, h, neighbor_lists, p_ref, cond_MLS, types, testCase, L, Periodic)

        drho1, accel1 = compute_derivatives(pos, vel, rho, pres,vol, m,accel, drho,types, h, dt,shepard_coeff,
                                                wall_normals,
                                                v_mesh1, tree,pairs, L_matrices, grad_p, grad_vel, grad_rho)   
        # --- FSI ---
        if testCase == 'bodyF':
            # Cet appel ne lit que les variables liées au corps flottant
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel, m, types, g, dt,t,
                body_vel=body_vel, body_omega=body_omega, body_theta=body_theta, body_cm=body_cm,
                rel_pos0=rel_pos0, nrm_body0_global=nrm_body0_global, mass_body_2D=mass_body_2D, I_2D=I_2D,
                wall_normals=wall_normals, idx_body_start=idx_body_start, n_body=n_body
            )
            
            # Récupération des données mises à jour
            body_vel = rigid_updates['body_vel']
            body_omega = rigid_updates['body_omega']
            body_theta = rigid_updates['body_theta']
            body_cm = rigid_updates['body_cm']
        
        elif testCase == 'car':
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel, m, types, g, dt, t,
                CAR=CAR, car_mass=car_mass, car_I=car_I, car_vel=car_vel, car_omega=car_omega, car_theta=car_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            car_vel = rigid_updates['car_vel']
            car_omega = rigid_updates['car_omega']
            car_theta = rigid_updates['car_theta']
            
        elif testCase == 'Mill':
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel, m, types, g, dt, t,
                CAR=WHEEL, car_I=wheel_I, car_omega=wheel_omega, car_theta=wheel_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            wheel_alpha = rigid_updates['wheel_alpha']
            wheel_omega = rigid_updates['wheel_omega']
            wheel_theta = rigid_updates['wheel_theta']
        #---------------------------------------------------------------------------------------------------------------------         
        if np.any(mask_f):
            rho_star[mask_f] = rho[mask_f] + drho1[mask_f] * dt
            vel_star[mask_f] = vel[mask_f] + (accel1[mask_f] + g )* dt
            vol_star[mask_f] = m[mask_f] / rho_star[mask_f]
            pos_star[mask_f] += (v_mesh1[mask_f])* dt
        pres_star=get_pressure(rho_star)
        # -----------------------------------------------------------------
            # GESTION DYNAMIQUE DE LA PÉRIODICITÉ
            # -----------------------------------------------------------------
        if Periodic:
            if testCase in ['TGV', 'TGVmesh','AirFoil']:
                pos_star[:, 0] = pos_star[:, 0] % L
                pos_star[:, 1] = pos_star[:, 1] % L
            elif testCase == 'Poiseuille':
                pos_star[mask_f, 0] = pos_star[mask_f, 0] % L  # fluides seulement, pas les WALL
                    # Pas de clip Y : les parois SPH assurent le confinement
        
        
        #       
        #recalcul des voisins 
        if Periodic:
            if testCase in ['TGV', 'TGVmesh','AirFoil']:
                tree_star = cKDTree(pos_star, boxsize=L)
            elif testCase == 'Poiseuille':
            # Arbre standard : coordonnées WALL en Y négatif => boxsize impossible
            # periodic_rij() corrige les vecteurs rij en X lors des calculs SPH
                
                tree_star = cKDTree(pos_star, boxsize=(L, None))
        else:
            tree_star = cKDTree(pos_star)
        
        # ★ OPTIMISATION : On extrait la liste des voisins pour l'état star ★
        neighbor_lists_star = tree_star.query_ball_point(pos_star, r=search_radius)
        # Extraction des paires uniques (i, j) avec i < j
        pairs_star = []
        for i, neighbors in enumerate(neighbor_lists_star):
            for j in neighbors:
                if i < j:
                    pairs_star.append((i, j))
        pairs_star = np.array(pairs_star, dtype=np.int32)
        
        if mode_p_refinement == 1:
            grad_rho = compute_mls_gradients(pos_star, vol_star, rho_star, h, neighbor_lists_star, p_ref, cond_MLS, types, testCase, L, Periodic)
            grad_vel = compute_mls_gradients(pos_star, vol_star, vel_star, h, neighbor_lists_star, p_ref, cond_MLS, types, testCase, L, Periodic)
            grad_p   = compute_mls_gradients(pos_star, vol_star, pres_star, h, neighbor_lists_star, p_ref, cond_MLS, types, testCase, L, Periodic)
        
        #recalcul de shepard
        # --- PHASE  : CALCUL DU CHAMP DE SUPPORT (SHEPARD) --- 
        # On fait un premier passage uniquement pour le voisinage et le shepard  
        num_neighbors.fill(0)
        shepard_coeff_star[mask_f] = kernel_cubic(0, h) * vol_star[mask_f]                    
        compute_shepard_coefficients(pairs_star, pos_star, vol_star, h,
                                        shepard_coeff_star,
                                        num_neighbors,  #attention à mettre en star
                                        kernel_cubic)
        if RenormON: 
            L_matrices_star, cond_array_star, renorm_status_star = compute_renormalization_matrices_robust( pos_star,
                                                                                          vol_star, h, neighbor_lists_star,
                                                                                          shepard_coeff_star,
                                                                                         testCase, Periodic, L
                                                                                          )  
        
       # ----- Compute pressure wall or bodyF or car ----         
        compute_pressure_wall(pairs_star, pos_star, vol_star, h, kernel_cubic, pres_star, types, g, rho0,testCase)
        # ------------------------------- ------------------ ---------------------------------------------------------------
        # ------------------------------- STEP 2: CORRECTOR ---------------------------------------------------------------------------------
        # ------------------------------- ------------------ ---------------------------------------------------------------
       
        # Paramètres de contrôle du Shifting Formulation type PST (Particle Shifting Technique)
        v_mesh2, S_G_local, S_dist, grad_S_G, grad_S_dist = compute_v_mesh_full(pos_star, vel_star, vol_star, h, dt, 
                                                                                  mesh_mode,shepard_coeff_star,tree_star,pairs_star)
        #---------------------------------------------------------------------------------------------------------------------  
        # On passe np.zeros_like au lieu des anciens accel/drho
        drho2, accel2 = compute_derivatives(pos_star, vel_star, rho_star, pres_star,
                                                vol_star, m,
                                                np.zeros_like(accel), np.zeros_like(drho),
                                                types, h, dt, shepard_coeff_star, wall_normals,
                                                v_mesh2, tree_star,pairs_star,L_matrices_star,
                                                grad_p, grad_vel, grad_rho)

        # --- FSI  ---
        if testCase == 'bodyF':
            # Cet appel ne lit que les variables liées au corps flottant
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel2, m, types, g, dt,t,
                body_vel=body_vel, body_omega=body_omega, body_theta=body_theta, body_cm=body_cm,
                rel_pos0=rel_pos0, nrm_body0_global=nrm_body0_global, mass_body_2D=mass_body_2D, I_2D=I_2D,
                wall_normals=wall_normals, idx_body_start=idx_body_start, n_body=n_body
            )
            
            # Récupération des données mises à jour
            body_vel = rigid_updates['body_vel']
            body_omega = rigid_updates['body_omega']
            body_theta = rigid_updates['body_theta']
            body_cm = rigid_updates['body_cm']
        
        elif testCase == 'car':
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel2, m, types, g, dt, t,
                CAR=CAR, car_mass=car_mass, car_I=car_I, car_vel=car_vel, car_omega=car_omega, car_theta=car_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            car_vel = rigid_updates['car_vel']
            car_omega = rigid_updates['car_omega']
            car_theta = rigid_updates['car_theta']
            
        elif testCase == 'Mill':
            rigid_updates, wall_normals = update_rigid_body_physics(
                testCase, pos, vel, accel2, m, types, g, dt, t,
                CAR=WHEEL, car_I=wheel_I, car_omega=wheel_omega, car_theta=wheel_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            wheel_alpha = rigid_updates['wheel_alpha']
            wheel_omega = rigid_updates['wheel_omega']
            wheel_theta = rigid_updates['wheel_theta']
        #---------------------------------------------------------------------------------------------------------------------         
       # Moyenne des deux pentes              
        if np.any(mask_f):
            rho[mask_f] +=0.5* (drho1[mask_f]+drho2[mask_f])* dt
            vol[mask_f] = m[mask_f] / rho[mask_f]
            vel[mask_f] += 0.5*(accel1[mask_f] + accel2[mask_f])*dt +g*dt
            pres[mask_f] = get_pressure(rho[mask_f])
            pos[mask_f] +=0.5*(v_mesh1[mask_f]+v_mesh2[mask_f])* dt
#         # ★ SHIFTING (après stage 2) ★
#         if ShiftPart and np.any(mask_f):
#             if ShiftMethod == 'delta_sph':
#                 grad_C, C = compute_concentration_gradient(
#                         pos, vol, h, tree, pairs, 
#                         Periodic=Periodic, testCase=testCase, L=L
#                 )
#                 shift, shift_status = compute_shifting_displacement_delta_sph(
#                         pos, vol, grad_C, C, h,
#                         CFL=ShiftCFL, R=ShiftR,
#                         types=types, FLUID=FLUID
#                 )
                
#             elif ShiftMethod == 'lloyd':
#                 shift, shift_status = compute_shifting_displacement_lloyd(
#                         pos, vol, h, tree, pairs,
#                         beta_lloyd=ShiftBetaLloyd,
#                         types=types, FLUID=FLUID
#                 )
                
#             pos = apply_shifting_correction(
#                     pos, shift, boundary_layer_width=2.0*h, h=h,
#                     types=types, WALL=WALL
#             )        
        # -----------------------------------------------------------------
        # GESTION DYNAMIQUE DE LA PÉRIODICITÉ
        # -----------------------------------------------------------------
        if Periodic:
            if testCase in ['TGV', 'TGVmesh','AirFoil']:
                pos[:, 0] = pos[:, 0] % L
                pos[:, 1] = pos[:, 1] % L
            elif testCase == 'Poiseuille':
                pos[mask_f, 0] = pos[mask_f, 0] % L  # fluides seulement, pas les WALL
                # Pas de clip Y : les parois SPH assurent le confinement          
        v_mesh = v_mesh2.copy() 
        
#---------------------------------------------------------------------------------------------------------------------             
    if mode_RK == 3:  # RK3-TVD (Shu & Osher 1988, SSP)
        accel1 = np.zeros((n_current,2))
        drho1 = np.zeros(n_current)
        v_mesh1 = np.zeros((n_current,2))
        
        accel2 = np.zeros((n_current,2))
        drho2 = np.zeros(n_current)      
        v_mesh2 = np.zeros((n_current,2))
        
        accel3 = np.zeros((n_current,2))
        drho3 = np.zeros(n_current)      
        v_mesh3 = np.zeros((n_current,2))
        
               
        shepard1 = np.zeros_like( shepard_coeff)
        shepard2 = np.zeros_like( shepard_coeff)
        L_matrices =np.zeros((n_current, 2, 2))
        p_ref= np.zeros(n_current)
        cond_MLS= np.zeros(n_current)        
        grad_p = np.zeros((n_current,5))
        grad_vel = np.zeros((n_current, 5, 5))
        grad_rho = np.zeros((n_current,5))

        hess_p = np.zeros((n_current,2))
        hess_velx = np.zeros((n_current, 2, 2))
        hess_vely = np.zeros((n_current, 2, 2))
        hess_rho = np.zeros((n_current,2))
        
        S_G_local = np.zeros(n_current)
        S_dist    = 0.0
        grad_S_G  = np.zeros((n_current, 2))
        grad_S_dist = np.zeros((n_current, 2))
        
        # --- Étage 1 : u^(1) = u^n + dt * L(u^n) ---
        v_mesh1, *_ = compute_v_mesh_full(pos, vel, vol, h, dt, mesh_mode, shepard_coeff, tree, pairs)
        if mode_p_refinement == 2:
            grad_rho  = compute_mls_gradients_order2(pos, vol, rho,  h, tree,p_ref,cond_MLS,types)
            grad_p = compute_mls_gradients_order2(pos, vol, pres, h, tree,p_ref,cond_MLS,types)
            grad_vel[:,0]  = compute_mls_gradients_order2(pos, vol, vel[:,0] ,  h, tree,p_ref,cond_MLS,types)
            grad_vel[:,1]  = compute_mls_gradients_order2(pos, vol, vel[:,1] ,  h, tree,p_ref,cond_MLS,types)
            
            
            # idem vel, pres
        drho1, accel1 = compute_derivatives(pos, vel, rho, pres, vol, m,
                                             accel, drho, types, h, dt,
                                             shepard_coeff, wall_normals,
                                             v_mesh1, neighbor_lists, L_matrices,grad_p, grad_vel, grad_rho)
        # --- FSI ---
        if testCase == 'bodyF':
            # Cet appel ne lit que les variables liées au corps flottant
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel1, m, types, g, dt,t,
                body_vel=body_vel, body_omega=body_omega, body_theta=body_theta, body_cm=body_cm,
                rel_pos0=rel_pos0, nrm_body0_global=nrm_body0_global, mass_body_2D=mass_body_2D, I_2D=I_2D,
                wall_normals=wall_normals, idx_body_start=idx_body_start, n_body=n_body
            )
            
            # Récupération des données mises à jour
            body_vel = rigid_updates['body_vel']
            body_omega = rigid_updates['body_omega']
            body_theta = rigid_updates['body_theta']
            body_cm = rigid_updates['body_cm']
        
        elif testCase == 'car':
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel1, m, types, g, dt, t,
                CAR=CAR, car_mass=car_mass, car_I=car_I, car_vel=car_vel, car_omega=car_omega, car_theta=car_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            car_vel = rigid_updates['car_vel']
            car_omega = rigid_updates['car_omega']
            car_theta = rigid_updates['car_theta']
            
        elif testCase == 'Mill':
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel1, m, types, g, dt, t,
                CAR=WALL, car_I=wheel_I, car_omega=wheel_omega, car_theta=wheel_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            wheel_alpha = rigid_updates['wheel_alpha']
            wheel_omega = rigid_updates['wheel_omega']
            wheel_theta = rigid_updates['wheel_theta']
        pos1  = pos.copy()
        vel1  = vel.copy()
        rho1  = rho.copy()
        if np.any(mask_f):
            rho1[mask_f]  = rho[mask_f]  + dt * drho1[mask_f]
            vel1[mask_f]  = vel[mask_f]  + dt * (accel1[mask_f] + g)
            pos1[mask_f]  += dt * v_mesh1[mask_f]
            vol1 = m / rho1
        pres1 = get_pressure(rho1)
        
        
        # --- Étage 2 : u^(2) = 3/4 u^n + 1/4 (u^(1) + dt * L(u^(1))) ---
        # Avant update_neighborhood dans le predictor RK2 :
        if Periodic and testCase == 'Poiseuille':
            pos1[mask_f, 0] = pos1[mask_f, 0] % L        
        #recalcul des voisins 
        tree1, pairs1 = update_neighborhood(pos1, h)
        #recalcul des voisins  
        # ★ OPTIMISATION : On extrait la liste des voisins pour l'état star ★
        neighbor_lists_1 = tree_star.query_ball_point(pos1, r=search_radius)
        shepard1 = np.zeros_like(shepard_coeff)
        shepard1[mask_f] = kernel_cubic(0, h) * vol1[mask_f] 
        # ----- Compute pressure wall or bodyF or car ---- 
        compute_pressure_wall(pairs1, pos1, vol1, h, kernel_cubic, pres1, types, g, rho0,testCase)
        num_neighbors.fill(0)
        compute_shepard_coefficients(pairs1, pos1, vol1, h, shepard1, num_neighbors, kernel_cubic)
        v_mesh2, *_ = compute_v_mesh_full(pos1, vel1, vol1, h, dt, mesh_mode, shepard1, tree1, pairs1)      
        
        
        if mode_p_refinement == 2:
            grad_rho = compute_mls_gradients_order2(pos1, vol1, rho1,  h, tree1,p_ref,cond_MLS,types)
            grad_p   = compute_mls_gradients_order2(pos1, vol1, pres1, h, tree1,p_ref,cond_MLS,types)
            grad_vel[:,0] = compute_mls_gradients_order2(pos1, vol1, vel1[:,0] ,  h, tree1,p_ref,cond_MLS,types)
            grad_vel[:,1] = compute_mls_gradients_order2(pos1, vol1, vel1[:,1] ,  h, tree1,p_ref,cond_MLS,types)
            
            
        drho2, accel2 = compute_derivatives(pos1, vel1, rho1, pres1, vol1, m,
                                             np.zeros_like(accel), np.zeros_like(drho),
                                             types, h, dt, shepard1, wall_normals,
                                             v_mesh2, neighbor_lists_1, L_matrices, grad_p, grad_vel, grad_rho)
       
        # --- FSI ---
        if testCase == 'bodyF':
            # Cet appel ne lit que les variables liées au corps flottant
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel2, m, types, g, dt,t,
                body_vel=body_vel, body_omega=body_omega, body_theta=body_theta, body_cm=body_cm,
                rel_pos0=rel_pos0, nrm_body0_global=nrm_body0_global, mass_body_2D=mass_body_2D, I_2D=I_2D,
                wall_normals=wall_normals, idx_body_start=idx_body_start, n_body=n_body
            )
            
            # Récupération des données mises à jour
            body_vel = rigid_updates['body_vel']
            body_omega = rigid_updates['body_omega']
            body_theta = rigid_updates['body_theta']
            body_cm = rigid_updates['body_cm']
        
        elif testCase == 'car':
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel2, m, types, g, dt, t,
                CAR=CAR, car_mass=car_mass, car_I=car_I, car_vel=car_vel, car_omega=car_omega, car_theta=car_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            car_vel = rigid_updates['car_vel']
            car_omega = rigid_updates['car_omega']
            car_theta = rigid_updates['car_theta']
            
        elif testCase == 'Mill':
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel2, m, types, g, dt, t,
                CAR=WALL, car_I=wheel_I, car_omega=wheel_omega, car_theta=wheel_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            wheel_alpha = rigid_updates['wheel_alpha']
            wheel_omega = rigid_updates['wheel_omega']
            wheel_theta = rigid_updates['wheel_theta']
        pos2  = pos.copy()
        vel2  = vel.copy()
        rho2  = rho.copy()
        if np.any(mask_f):
            rho2[mask_f]  = 0.75*rho[mask_f]  + 0.25*(rho1[mask_f]  + dt*drho2[mask_f])
            vel2[mask_f]  = 0.75*vel[mask_f]  + 0.25*(vel1[mask_f]  + dt*(accel2[mask_f]+g))
            pos2[mask_f]  = 0.75*pos[mask_f]  + 0.25*(pos1[mask_f]  + dt*v_mesh2[mask_f])
            vol2 = m / rho2
        pres2 = get_pressure(rho2)
        
        # --- Étage 3 : u^(n+1) = 1/3 u^n + 2/3 (u^(2) + dt * L(u^(2))) ---
        # Avant update_neighborhood dans le predictor RK2 :
        if Periodic and testCase == 'Poiseuille':
            pos2[mask_f, 0] = pos2[mask_f, 0] % L        
        
        #recalcul des voisins         
        tree2, pairs2 = update_neighborhood(pos2, h)
        #recalcul des voisins  
        # ★ OPTIMISATION : On extrait la liste des voisins pour l'état star ★
        neighbor_lists_2 = tree_star.query_ball_point(popos2, r=search_radius)
        shepard2 = np.zeros_like(shepard_coeff)
        shepard2[mask_f] = kernel_cubic(0, h) * vol2[mask_f] 
        # ----- Compute pressure wall or bodyF or car ---- 
        compute_pressure_wall(pairs2, pos2, vol2, h, kernel_cubic, pres2, types, g, rho0, testCase)
        num_neighbors.fill(0)
        compute_shepard_coefficients(pairs2, pos2, vol2, h, shepard2, num_neighbors, kernel_cubic)
        v_mesh3, S_G_local, S_dist, grad_S_G, grad_S_dist = compute_v_mesh_full(pos2, vel2, vol2, h, dt, mesh_mode, shepard2, tree2, pairs2)
        if mode_p_refinement == 2:
            grad_rho  = compute_mls_gradients_order2(pos2, vol2, rho2,  h, tree2,p_ref,cond_MLS,types)
            grad_p = compute_mls_gradients_order2(pos2, vol2, pres2, h, tree2,p_ref,cond_MLS,types)
            grad_vel[:,0]  = compute_mls_gradients_order2(pos2, vol2, vel2[:,0] ,  h, tree2,p_ref,cond_MLS,types)
            grad_vel[:,1] = compute_mls_gradients_order2(pos2, vol2, vel2[:,1] ,  h, tree2,p_ref,cond_MLS,types)
            
        drho3, accel3 = compute_derivatives(pos2, vel2, rho2, pres2, vol2, m,
                                             np.zeros_like(accel), np.zeros_like(drho),
                                             types, h, dt, shepard2, wall_normals,
                                             v_mesh3, neighbor_lists_2, L_matrices,grad_p, grad_vel, grad_rho)
        # --- FSI ---
        if testCase == 'bodyF':
            # Cet appel ne lit que les variables liées au corps flottant
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel3, m, types, g, dt,t,
                body_vel=body_vel, body_omega=body_omega, body_theta=body_theta, body_cm=body_cm,
                rel_pos0=rel_pos0, nrm_body0_global=nrm_body0_global, mass_body_2D=mass_body_2D, I_2D=I_2D,
                wall_normals=wall_normals, idx_body_start=idx_body_start, n_body=n_body
            )
            
            # Récupération des données mises à jour
            body_vel = rigid_updates['body_vel']
            body_omega = rigid_updates['body_omega']
            body_theta = rigid_updates['body_theta']
            body_cm = rigid_updates['body_cm']
        
        elif testCase == 'car':
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel3, m, types, g, dt, t,
                CAR=CAR, car_mass=car_mass, car_I=car_I, car_vel=car_vel, car_omega=car_omega, car_theta=car_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            car_vel = rigid_updates['car_vel']
            car_omega = rigid_updates['car_omega']
            car_theta = rigid_updates['car_theta']
            
        elif testCase == 'Mill':
            rigid_updates = update_rigid_body_physics(
                testCase, pos, vel, accel3, m, types, g, dt, t,
                CAR=WALL, car_I=wheel_I, car_omega=wheel_omega, car_theta=wheel_theta,
                wall_normals=wall_normals # AJOUT INDISPENSABLE
            )
            
            wheel_alpha = rigid_updates['wheel_alpha']
            wheel_omega = rigid_updates['wheel_omega']
            wheel_theta = rigid_updates['wheel_theta']
        if np.any(mask_f):
            rho[mask_f]  = (1/3)*rho[mask_f]  + (2/3)*(rho2[mask_f]  + dt*drho3[mask_f])
            vel[mask_f]  = (1/3)*vel[mask_f]  + (2/3)*(vel2[mask_f]  + dt*(accel3[mask_f]+g))
            pos[mask_f]  = (1/3)*pos[mask_f]  + (2/3)*(pos2[mask_f]  + dt*v_mesh3[mask_f])
            vol[mask_f]  = m[mask_f] / rho[mask_f]
            pres[mask_f] = get_pressure(rho[mask_f])
                    # ★ SHIFTING (après stage 2) ★
        if ShiftPart and np.any(mask_f):
            if ShiftMethod == 'delta_sph':
                grad_C, C = compute_concentration_gradient(
                        pos, vol, h, tree, pairs, 
                        Periodic=Periodic, testCase=testCase, L=L
                )
                shift, shift_status = compute_shifting_displacement_delta_sph(
                        pos, vol, grad_C, C, h,
                        CFL=ShiftCFL, R=ShiftR,
                        types=types, FLUID=FLUID
                )
                
            elif ShiftMethod == 'lloyd':
                shift, shift_status = compute_shifting_displacement_lloyd(
                        pos, vol, h, tree, pairs,
                        beta_lloyd=ShiftBetaLloyd,
                        types=types, FLUID=FLUID
                )
                
            pos = apply_shifting_correction(
                    pos, shift, boundary_layer_width=2.0*h, h=h,
                    types=types, WALL=WALL
            ) 
            # -----------------------------------------------------------------
            # GESTION DYNAMIQUE DE LA PÉRIODICITÉ
            # -----------------------------------------------------------------
            if Periodic:
                if testCase in ['TGV', 'TGVmesh','AirFoil']:
                    pos[:, 0] = pos[:, 0] % L
                    pos[:, 1] = pos[:, 1] % L
                elif testCase == 'Poiseuille':
                    pos[mask_f, 0] = pos[mask_f, 0] % L  # fluides seulement, pas les WALL
                    # Pas de clip Y : les parois SPH assurent le confinement
       
        v_mesh = v_mesh3.copy() 
                   
#---------------------------------------------------------------------------------------------------------------------             
#     if testCase=='TGV' or testCase=='TGVmesh':
#         # Calcul de la fonctionnelle APRÈS déplacement
#         F1_after, FS_after, _, _, _ = compute_energy_functionals(
#             pos[mask_f], vol[mask_f], h, shepard_coeff[mask_f], pairs
#         )
#         F2_after = F1_after + alpha * FS_after

#         # Vérification de la décroissance
#         if F2_after > F2_before * (1.0 + 1e-6):  # tolérance numérique
#             # Backtracking : réduire beta_geom de moitié
#             beta_geom *= 0.5
#             print(f"[WARNING] F2 non décroissante : {F2_before:.4e} → {F2_after:.4e}. β réduit à {beta_geom:.4e}")
#         else:
#             delta_F2 = F2_before - F2_after
#             if step % save == 0:
#                 print(f"F1={F1_after:.4e} | FS={FS_after:.4e} | F2={F2_after:.4e} | ΔF2={delta_F2:.4e}")
    
    if testCase == 'funnel': 
        H = funnel_height(pos, types, FLUID, vertical_axis=1 )            
        time_history.append(t)   
        H_history.append(H)            
        Q, Cd = funnel_discharge_coefficient(
                                 pos, vel, vol, types, FLUID,
                                 np.array([0.0, 0.0]), # Centre de sortie approché (x, y)
                                 R_bottom,
                                 np.array([0.0, -1.0]),
                                 g, H)            
        Cd_history.append(Cd)
        Q_history.append(Q)      
        
    if testCase == 'DamBreak': 
        # Calcul de l'énergie cinétique
        kinetic_energy = 0.5 * np.sum(m[mask_f] * np.linalg.norm(vel[mask_f], axis=1)**2)
        # Calcul de l'énergie potentielle
        potential_energy = np.sum(m[mask_f] * g_acc * pos[mask_f, 1])
        # Énergie mécanique totale
        mechanical_energy = kinetic_energy + potential_energy
            
        time_history.append(t)   
        E_history.append(mechanical_energy)
    
    
    
    # Sortie & Update Time acoustic constrained
    fluid_vel = vel[mask_f]
    fluid_pres = pres[mask_f]

    max_v = np.max(np.linalg.norm(fluid_vel, axis=1)) if fluid_vel.size else 0
    max_p = np.max(np.abs(fluid_pres)) if fluid_pres.size else 0
    #if viscosity on 
    max_set = max(c0, max_v, np.sqrt(max_p / rho0))
    dt = CFL * h / max_set
    
    if testCase == 'bodyF':
        dt = CFL * h / max(c0, max_v, np.sqrt(max_p / rho0))
    if testCase == 'car':
        dt = CFL * h / max(c0, max_v, np.sqrt(max_p / rho0), carvitesse)
        dt_spring = 0.2 * np.sqrt(car_mass / k_local)
        dt = min(dt, dt_spring)
        
    if mesh_mode == 'ale' :
        max_v_mesh = np.max(np.linalg.norm(v_mesh[mask_f], axis=1)) if np.any(mask_f) else 0
        dt = CFL * h / (c0 + max_v_mesh)
        
    if ViscoONOFF:
        # 2. dt visqueux (Morris condition)
        dt_viscous = 0.125 * (rho0 * h**2) / (mu + 2.2e-16)
        dt = min(dt,dt_viscous)
    #--------------------------------------------------------------------------------------------------------------------- 
    #----------------------------------  Visualisations          ---------------------------------------------------------- 
    #---------------------------------------------------------------------------------------------------------------------       
    # 1. Advection exclusive des particules de fluide et de buffer (types > 2)
    if testCase in ['jet', 'Mill']:
        mask_fluid_buffer = (types > 2)
        pos[mask_fluid_buffer] += vel[mask_fluid_buffer] * dt
        
        # 2. CORRECTION : On ne réinitialise wall_normals QUE pour le cas 'jet' pur
        # Surtout pas pour 'Mill' où les normales doivent être préservées et tourner !
        if testCase == 'jet' and DOUBLE_JET:
            wall_normals = np.zeros((len(pos), 2))
   
        
    shepard_visual = np.where(shepard_coeff > 2.2e-16, shepard_coeff, 0.0)
    # Calcul du nombre de particules fluides
    n_fluid = np.sum(mask_f)
    total_mass = np.sum(m[mask_f])
    # Calcul de l'énergie cinétique
    kinetic_energy = 0.5 * np.sum(m[mask_f] * np.linalg.norm(vel[mask_f], axis=1)**2)
    # Calcul de l'énergie potentielle
    potential_energy = np.sum(m[mask_f] * g_acc * pos[mask_f, 1])
    # Énergie mécanique totale
    mechanical_energy = kinetic_energy + potential_energy   
    #-----------------------------------SAVE STEP----------------------------------------------------------------------     
    if step % save == 0:
        
            
        write_vtk(os.path.join(vtk_dir, f"testcase{step:05d}.vtk"), pos, vel,v_mesh-vel , pres, types, rho,
                  vol, m, num_neighbors, shepard_coeff,wall_normals, S_G_local, S_dist, grad_S_G, grad_S_dist,
                  cond_array, renorm_status, grad_p, grad_vel,grad_rho,p_ref,cond_MLS)
        print(f"Step {step} | t = {t:.4f}s | dt = {dt:.6e} | nombre de particules = {n_fluid:.6e}")
        print(f"Total Mass: {np.sum(m[mask_f]):.6e}, Total Volume: {np.sum(vol[mask_f]):.6e}, Mechanical Energy: {mechanical_energy:.6e}")
        #print(f"Reynolds (Exit): {reynolds_exit:.1f} | Reynolds (Max): {reynolds_max:.1f}")
    #--------------------------------------------------------------------------------------------------------------------- 
    if testCase=='TGV' or testCase=='TGVmesh': 
        E = np.sum(0.5 * rho * np.sum(vel**2,axis=1) * vol)  
        if step % save == 0:
            v_ana_f, _ = get_tgv_solution(pos[:n], t, L, U0, nu)
            err = np.sqrt(np.sum((vel[:n] - v_ana_f)**2) / np.sum(v_ana_f**2))
            print("E=",E)
            #print(f"Step {step:04d} | Mode: {mesh_mode} | L2 Error: {err:.4e} | S_G_mean: {np.mean(S_G_local):.6e} | S_dist: {S_dist:.6e}")
            print(f"Step {step:04d} | Mode: {mesh_mode} | L2 Error debug: {err:.4e} ")
        if abs(t - t_star) < 0.5*dt:
            v_ana_f, _ = get_tgv_solution(pos[:n], t, L, U0, nu)
            err = np.sqrt(np.sum(vol[:n] * np.sum((vel[:n]-v_ana_f)**2, axis=1)) /
                          np.sum(vol[:n] * np.sum(v_ana_f**2, axis=1)))
            print("E=",E)
            #print(f"Step {step:04d} | Mode: {mesh_mode} | L2 Error: {err:.4e} | S_G_mean: {np.mean(S_G_local):.6e} | S_dist: {S_dist:.6e}")
            print(f"Step {step:04d} | Mode: {mesh_mode} | L2 Error convergence: {err:.4e} ")
    
    #-----------------------------------CLEANING----------------------------------------------------------------------
    # Suppression des particules sorties (ex: y < -0.2m)
    if testCase == 'funnel':
        keep = pos[:, 1] > -0.005
        if not np.all(keep):
            num_neighbors=num_neighbors[keep]
            shepard_coeff=shepard_coeff[keep]
            cond_array=cond_array[keep]
            renorm_status=renorm_status[keep]
            pos = pos[keep]
            vel = vel[keep]
            v_mesh = v_mesh[keep]
            types = types[keep]
            rho = rho[keep]
            pres = pres[keep]
            m = m[keep]
            vol = vol[keep]
            # Important : Mettre à jour n_p pour le calcul des masques au prochain step
            n_p = len(pos)     
       
    if testCase == 'car':    
        keep = (pos[:,1]>-1.5) & (pos[:,1]<3.2) & (pos[:,0]>-2.1) & (pos[:,0]<24.1)
        if not np.all(keep):
            pos=pos[keep]; vel=vel[keep]; types=types[keep]
            rho=rho[keep]; pres=pres[keep]; vol=vol[keep]; m=m[keep]
            num_neighbors=num_neighbors[keep]; shepard_coeff=shepard_coeff[keep]
            
    if testCase == 'jet' :    
        keep = (pos[:,1]>-0.2) & (pos[:,1]<2.1) & (pos[:,0]>-0.1) & (pos[:,0]<1.1)
        if not np.all(keep):
            pos=pos[keep]; vel=vel[keep]; types=types[keep]
            rho=rho[keep]; pres=pres[keep]; vol=vol[keep]; m=m[keep]
            num_neighbors=num_neighbors[keep]; shepard_coeff=shepard_coeff[keep]
            
    if testCase == 'Mill' :    
        keep = (pos[:,1]>-0.2) & (pos[:,1]<2.0) & (pos[:,0]>0.2) & (pos[:,0]<1.4)
        # 2. Exclusion circulaire dynamique centrée sur la roue
        # Remplacez wheel_centerX et wheel_centerY par vos variables effectives
        # Rayon du trou central à adapter selon votre géométrie
        distance_center = np.sqrt((pos[:,0] - wheel_centerX)**2 + (pos[:,1] - wheel_centerY)**2)
        keep2 = distance_center < r_hub
        #keep2 = (pos[:,1]>0.38) & (pos[:,1]<0.42) & (pos[:,0]>0.77) & (pos[:,0]<0.82)
        
          
            
        keep= keep & ~keep2
        if not np.all(keep):
            pos=pos[keep]; vel=vel[keep]; types=types[keep]
            rho=rho[keep]; pres=pres[keep]; vol=vol[keep]; m=m[keep]
            num_neighbors=num_neighbors[keep]; shepard_coeff=shepard_coeff[keep]
    #---------------------------------------------------------------------------------------------------------------------         
    if testCase=='Poiseuille':    
        plot_poiseuille_comparison(pos, vel, types, t, L, Fe_x, mu, nu, save_dir= vtk_dir)
   
    if testCase == 'bodyF':
        # ------------------------------------------------------------------
        # 8j. Enregistrement des données (sondes et corps)
        # ------------------------------------------------------------------
        eta1 = measure_surface_elevation(pos, types, rho, rho0, x_probe1, h)
        eta2 = measure_surface_elevation(pos, types, rho, rho0, x_probe2, h)

        probe_t.append(t)
        probe_eta1.append(eta1)
        probe_eta2.append(eta2)
        body_heave.append(body_cm[1] - y_body_init)
        body_sway.append(body_cm[0]  - x_body_init)
        body_roll.append(np.degrees(body_theta))
        body_cm_hist.append(body_cm.copy())

        # ------------------------------------------------------------------
        # 8k. Sorties VTK et console
        # ------------------------------------------------------------------
        if step % save == 0:
            fname = os.path.join(vtk_dir, f"WaveFloat{step:05d}.vtk")
            write_vtk_wf(fname, pos, vel, v_mesh - vel, pres, types,
                         rho, vol, m, num_neighbors, shepard_coeff, wall_normals,
                         body_cm, body_theta, body_vel, body_omega)

            theta_deg = np.degrees(flap_angle(t))
            print(f"Step {step:5d} | t={t:6.3f}s | dt={dt:.2e} | "
                  f"θ_clap={theta_deg:+.2f}° | "
                  f"Heave={body_heave[-1]*100:+.2f}cm | "
                  f"Sway={body_sway[-1]*100:+.2f}cm | "
                  f"Roll={body_roll[-1]:+.2f}°")
    
    
    
    t += dt; step += 1
    
if testCase=='Poiseuille':    
   plot_poiseuille_comparison(pos, vel, types, t, L, Fe_x, mu, nu, save_dir= vtk_dir) 
    
    
if testCase=='bodyF':
    # ==============================================================================
    # 9.  POST-TRAITEMENT ET COMPARAISON EXPÉRIMENTALE
    # ==============================================================================

    print("\n" + "="*60)
    print("  POST-TRAITEMENT")
    print("="*60)

    t_arr    = np.array(probe_t)
    eta1_arr = np.array(probe_eta1)
    eta2_arr = np.array(probe_eta2)
    heave_arr = np.array(body_heave)
    sway_arr  = np.array(body_sway)
    roll_arr  = np.array(body_roll)
    cm_arr    = np.array(body_cm_hist)

    # Écriture des fichiers ASCII
    write_probe_ascii(vtk_dir, t_arr, eta1_arr, eta2_arr,
                      heave_arr, sway_arr, roll_arr, cm_arr)

    # Figures de comparaison
    plot_comparison(vtk_dir, t_arr, eta1_arr, eta2_arr,
                    heave_arr, sway_arr, roll_arr, exp_data)

    # Statistiques de synthèse
    print(f"\n--- Résumé statistiques ---")
    print(f"Heave max        : {np.max(np.abs(heave_arr))*100:.2f} cm")
    print(f"Sway max         : {np.max(np.abs(sway_arr))*100:.2f} cm")
    print(f"Roll max         : {np.max(np.abs(roll_arr)):.2f}°")
    print(f"η max (sonde 1)  : {np.max(np.abs(eta1_arr))*100:.2f} cm")
    print(f"η max (sonde 2)  : {np.max(np.abs(eta2_arr))*100:.2f} cm")
    print(f"\nSimulation terminée. Résultats dans : {vtk_dir}/")

if testCase == 'DamBreak':        
    print("\n" + "="*60)
    print("  POST-TRAITEMENT DamBreak - GÉNÉRATION DES COURBES")
    print("="*60)

    times = np.array(time_history)
    E = np.array(E_history)
    
    
    # Sécurité pour éviter la division par zéro
    if len(E) > 0 and E[0] > 1e-6:
        E = E / E[0]
    else:
        E = np.zeros_like(E)

    # --- Définition des chemins explicites vers le dossier de simulation ---
    path_E = os.path.join(vtk_dir, 'courbe_E_SPHIRIT.png')
    

    # --- Graphique 1 : Htilde ---
    plt.figure(figsize=(10, 6))
    plt.plot(times, E, label='E/E0 SPHIRIT', lw=2)
    plt.xlabel('Temps (s)')
    plt.ylabel('E / E0')
    plt.title('Évolution de l\'energie adimensionnée')
    plt.legend()
    plt.grid(True)
    plt.savefig(path_E, dpi=300)
    plt.close() 

    print(f"✅ Fichiers PNG générés avec succès dans le dossier : {vtk_dir}")    
    
if testCase == 'funnel':        
    print("\n" + "="*60)
    print("  POST-TRAITEMENT FUNNEL - GÉNÉRATION DES COURBES")
    print("="*60)

    times = np.array(time_history)
    H = np.array(H_history)
    Q = np.array(Q_history)
    Cd = np.array(Cd_history)
    
    # Sécurité pour éviter la division par zéro
    if len(H) > 0 and H[0] > 1e-6:
        Htilde = H / H[0]
    else:
        Htilde = np.zeros_like(H)

    # --- Définition des chemins explicites vers le dossier de simulation ---
    path_H = os.path.join(vtk_dir, 'courbe_Htilde_SPHIRIT.png')
    path_Cd = os.path.join(vtk_dir, 'courbe_Cd_SPHIRIT.png')
    path_Q = os.path.join(vtk_dir, 'courbe_Q_SPHIRIT.png')

    # --- Graphique 1 : Htilde ---
    plt.figure(figsize=(10, 6))
    plt.plot(times, Htilde, label='H/H0 SPHIRIT', lw=2)
    plt.xlabel('Temps (s)')
    plt.ylabel('H / H0')
    plt.title('Évolution de la hauteur d\'eau adimensionnée')
    plt.legend()
    plt.grid(True)
    plt.savefig(path_H, dpi=300)
    plt.close() 

    # --- Graphique 2 : Coefficient de décharge Cd ---
    plt.figure(figsize=(10, 6))
    plt.plot(times, Cd, label='Cd (Coefficient de décharge)', color='green', lw=2)
    plt.xlabel('Temps (s)')
    plt.ylabel('Cd')
    plt.title('Coefficient de décharge')
    plt.legend()
    plt.grid(True)
    plt.savefig(path_Cd, dpi=300)
    plt.close()

    # --- Graphique 3 : Débit Q ---
    plt.figure(figsize=(10, 6))
    plt.plot(times, Q, label='Débit SPHIRIT', color='red', lw=2)
    plt.xlabel('Temps (s)')
    plt.ylabel('Débit (m³/s)')
    plt.title('Débit en sortie de l\'entonnoir')
    plt.legend()
    plt.grid(True)
    plt.savefig(path_Q, dpi=300)
    plt.close()

    print(f"✅ Fichiers PNG générés avec succès dans le dossier : {vtk_dir}")
    
    
    
    
    
    
    
    
# MAJ BIBLIO
# TGV Eulerian : ajout du terme de transport ALE dans l'équation de continuité
# En mode Eulérien (v_mesh=0), le flux de masse de l'équation de continuité doit inclure le terme d'advection : drho/dt + ρ div(v) = 0. Le code utilise la formulation ALE de Vila (1999) qui est correcte pour v_mesh=v, mais quand v_mesh=0, le terme ρ·(u_star - v_interface) n'est pas le bon opérateur. Bauer & Springel (2012 MNRAS 423:2558) détaillent la formulation correcte pour le mode Eulérien pur.    
    
    
    
    
    
    
    
    # MAJ BIBLIO
#     Solveur acoustique : correction Low-Mach de Thornber et al. (2008)
# Le solveur Riemann actuel (acoustique centré β=0.5) introduit une dissipation artificielle O(M) sur la vitesse. Pour TGV à M=0.1, l'énergie cinétique décroît trop vite. Thornber et al. (2008 JCP 227:4873) proposent de remplacer ΔU = uR−uL par ΔU_LM = f(M)·(uR−uL) avec f(M) = min(1, max(|M_L|,|M_R|)). Réduit la dissipation à O(M²).
# # Fix Thornber 2008 dans riemann_solver :
# M_L = abs(uL) / c0;  M_R = abs(uR) / c0
# f_M = min(1.0, max(M_L, M_R, 1e-6))
# u_star = 0.5*(uL+uR) - beta * f_M * (pR_dyn-pL_dyn)/(rho_avg*c0)

# MAJ BIBLIO
# Particle Shifting : remplacer PST ad hoc par Sun et al. (2017) δ-SPH shifting
# Le shifting actuel via ALE/Wasserstein est original mais non validé. Sun et al. (2017 CMAME 315:25) proposent un shifting δ-SPH rigoureux basé sur le gradient de concentration ∇C avec terme correctif Fickien : δx_i = CFL·h·R·∑_j(−∂C/∂x_j)·∇W_ij·V_j. Préserve la masse, a des garanties de stabilité, et est largement validé sur TGV et DamBreak.

# MAJ BIBLIO
# TGV convergence : ajouter terme de correction δ-SPH pour réduire l'erreur O(h)
# Antuono et al. (2010 CMAME 199:1526) montrent que le terme diffusif δρ·∇²ρ ajouté à l'équation de continuité réduit les oscillations de pression et améliore la convergence. Le code utilise Tait WCSPH standard ; l'ajout de ce terme δ avec δ≈0.1 améliore typiquement la convergence de 0.5 ordre sur TGV.
    
    